# Zeitliche Ausbreitung und funktionale Kontrollen

**A100 → Alle ausführen.** Im kleinen Starter das bereitgestellte
**temporal_functional_paket.zip** auswählen. Kein Drive und kein weiterer Upload.

16 festgelegte Eingaben werden in acht Kalibrierungs- und acht Testfälle getrennt.
Eine gemeinsame Richtung wird auf der Kalibrierung festgelegt und vor den
Testmessungen eingefroren. Identische Fortsetzungen und native Cache-Kontrollen
trennen zeitliche Zustandsänderungen von abweichenden Eingabetokens.
Funktionale Endpunkte werden getrennt von der reinen Geometrie gespeichert.

Originales Qwen3-1.7B, FP32 mit ausgeschaltetem TF32, dieselben 286 Koordinaten
und dieselbe REF-W-Normierung bleiben erhalten. Modellgewichte werden nicht
trainiert. Der Versuch ist explorativ und belegt für sich kein Gedächtnis und
keinen allgemeinen Selbstüberwachungsmechanismus.

GPU-Torch bleibt installiert; vorhandene Modellgewichte im Cache werden
wiederverwendet. Das Rechenbudget beträgt 20 Minuten mit Checkpoints;
Installation und Modellladen kommen hinzu. Das ist keine Laufzeitgarantie.
Ein Status `paused` bedeutet einen fortsetzbaren Teilstand.

Am Ende **temporal_functional_results.zip** für die Auswertung herunterladen.
**temporal_functional_results_RAW.zip** ebenfalls sichern: nur die Rohdatei ist
zur Wiederaufnahme geeignet. Bei Fehlern wird ein Diagnose-ZIP angeboten.
Das private Paket nicht öffentlich hochladen. Die GPU-Sitzung danach beenden,
wenn sie nicht mehr gebraucht wird.


In [ ]:
# Private Nutzdatei: unverändert lassen und Alle ausführen.
# Öffentliches Template: optional vorhandene Originaldateien in diesem Verzeichnis.
REFERENCE_DIRECTORY = ""
RESUME_ZIP = ""  # Nur für einen vorhandenen Checkpoint von einer anderen Sitzung.
RUN_DIRECTORY = ""  # Leer: automatisch, wiederholbar in derselben Sitzung.
MIN_FREE_VRAM_GIB = 12.0


In [ ]:
# Eingebettete Quellen und optionale private Originalbytes; nicht manuell ändern.
TRANSFORMERS_VERSION = '5.22.2'
ACCELERATE_VERSION = '1.16.0'
REFERENCE_SHA256 = {'hn_set.json': '11390233b62c63b9d34805482ecbd3675eadd80570b9c01ef9e268bf565c98fd', 'htq_policy.json': '160c420603b719afba4b909d12d31316535290e13b00aea2a6dea88110684f7c', 'refw_h286.json': '68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc', 'temporal_items.json': '997f59b8ab3fae7190a609958def98227bcad2546666e1cc66bcdf89778e225e', 'temporal_plan.json': 'af42159b61e639da19cdb99c140e048cd5e88ef2c82bd467b5217f88dfb40b62'}
REFERENCE_FILES_B64 = {}
MODULE_SOURCES = {'attention_probe/__init__.py': '"""Isolated, optional attention-pulse pilot; existing experiment code is unchanged."""\n', 'attention_probe/reference.py': '"""Bind a new attention pilot to the private, exact DSSE H_tq selection.\n\nThis module validates identities and provenance only. It imports no model runtime,\ncontains no private neuron coordinates, and does not map them to another model.\n\n    python -m attention_probe.reference --hn-set /private/hn_set.json \\\n        --policy /private/htq_policy.json --out /private/reference.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom typing import Any\n\n\nHN_SET_SHA256 = "11390233b62c63b9d34805482ecbd3675eadd80570b9c01ef9e268bf565c98fd"\nPOLICY_SHA256 = "160c420603b719afba4b909d12d31316535290e13b00aea2a6dea88110684f7c"\nMODEL_ID = "qwen3-1.7b"\nMODEL_REPO = "Qwen/Qwen3-1.7B"\nMODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"\nN_COORDINATES, N_LAYERS, MLP_WIDTH = 286, 28, 6144\nH_POLICY = "htq_v2"\n\n\nclass ReferenceError(ValueError):\n    """The supplied files do not establish the required DSSE reference."""\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelPins:\n    repo: str\n    revision: str\n    sha256: tuple[tuple[str, str], ...]\n    model_id: str = MODEL_ID\n    n_layers: int = N_LAYERS\n    mlp_width: int = MLP_WIDTH\n\n    def to_dict(self) -> dict[str, Any]:\n        return dict(model_id=self.model_id, repo=self.repo, revision=self.revision,\n                    sha256=dict(self.sha256), n_layers=self.n_layers, mlp_width=self.mlp_width)\n\n\n@dataclass(frozen=True, slots=True)\nclass Reference:\n    coordinates: tuple[tuple[int, int], ...]\n    hn_sha256: str\n    policy_sha256: str\n    model: ModelPins\n    h_policy: str = H_POLICY\n    post_hoc: bool = True\n    amendment: int = 2\n\n    def to_dict(self) -> dict[str, Any]:\n        """Return a fresh JSON binding; its coordinates remain private run data."""\n        return dict(schema="dsse_h286_reference_v1", coordinates=[list(x) for x in self.coordinates],\n                    coordinate_order="hn_set.H_op (identical to H_tq)",\n                    n_coordinates=len(self.coordinates),\n                    source_sha256={"hn_set.json": self.hn_sha256, "htq_policy.json": self.policy_sha256},\n                    model=self.model.to_dict(), h_policy=self.h_policy, post_hoc=self.post_hoc,\n                    amendment=self.amendment, confirmatory_eligible=False,\n                    scope="Neuron identities only; no CETT equivalence or cross-model coordinate mapping.")\n\n\ndef _digest(raw: bytes) -> str:\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef _sha(value: Any, label: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:\n        raise ReferenceError(f"{label} must be a lowercase SHA-256 digest.")\n    return value\n\n\ndef _object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:\n    out = {}\n    for key, value in pairs:\n        if key in out:\n            raise ReferenceError("Duplicate JSON object key.")\n        out[key] = value\n    return out\n\n\ndef _reject_constant(value: str) -> Any:\n    raise ReferenceError("Non-finite JSON constants are not allowed.")\n\n\ndef _document(raw: bytes, label: str) -> dict[str, Any]:\n    try:\n        value = json.loads(raw, object_pairs_hook=_object, parse_constant=_reject_constant)\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise ReferenceError(f"{label} is not valid JSON.") from exc\n    if not isinstance(value, dict):\n        raise ReferenceError(f"{label} must be a JSON object.")\n    return value\n\n\ndef _coordinates(value: Any, label: str) -> tuple[tuple[int, int], ...]:\n    if not isinstance(value, list) or len(value) != N_COORDINATES:\n        raise ReferenceError(f"{label} must contain exactly {N_COORDINATES} coordinates.")\n    result = []\n    for i, coordinate in enumerate(value):\n        if not isinstance(coordinate, list) or len(coordinate) != 2 or any(type(x) is not int for x in coordinate):\n            raise ReferenceError(f"{label}[{i}] must be a pair of integer coordinates.")\n        layer, index = coordinate\n        if not (0 <= layer < N_LAYERS and 0 <= index < MLP_WIDTH):\n            raise ReferenceError(f"{label}[{i}] is outside the 28 by 6144 MLP architecture.")\n        result.append((layer, index))\n    if len(set(result)) != N_COORDINATES:\n        raise ReferenceError(f"{label} contains duplicate coordinates.")\n    return tuple(result)\n\n\ndef _policy(document: dict[str, Any], label: str) -> None:\n    if document.get("h_policy") != H_POLICY or document.get("post_hoc") is not True:\n        raise ReferenceError(f"{label} must identify the post-hoc htq_v2 policy.")\n    if type(document.get("amendment")) is not int or document["amendment"] != 2:\n        raise ReferenceError(f"{label} must identify amendment 2.")\n    if document.get("H_op_name") != "H_tq":\n        raise ReferenceError(f"{label} must use H_tq as the operative set.")\n\n\ndef _model(document: dict[str, Any], label: str) -> ModelPins:\n    provenance = document.get("provenance")\n    if not isinstance(provenance, dict) or not isinstance(provenance.get("models"), dict):\n        raise ReferenceError(f"{label} has no model provenance.")\n    for field, expected in (("h_policy", H_POLICY), ("post_hoc", True), ("amendment", 2), ("mock", False)):\n        if field in provenance and (type(provenance[field]) is not type(expected) or provenance[field] != expected):\n            raise ReferenceError(f"{label} has conflicting {field} provenance.")\n    model = provenance["models"].get(MODEL_ID)\n    if not isinstance(model, dict) or model.get("repo") != MODEL_REPO or model.get("revision") != MODEL_REVISION:\n        raise ReferenceError(f"{label} does not identify the pinned Qwen3-1.7B model.")\n    hashes = model.get("sha256")\n    if not isinstance(hashes, dict) or not hashes or any(not isinstance(k, str) or not k for k in hashes):\n        raise ReferenceError(f"{label} has no model-file SHA-256 pins.")\n    pins = tuple(sorted((name, _sha(digest, f"{label} model file")) for name, digest in hashes.items()))\n    return ModelPins(repo=model["repo"], revision=model["revision"], sha256=pins)\n\n\ndef verify_documents(hn_bytes: bytes, policy_bytes: bytes, *, expected_hn_sha256: str,\n                     expected_policy_sha256: str) -> Reference:\n    """Low-level verification with explicit trust anchors, including synthetic tests.\n\n    Normal pilot code must call load_reference(), whose anchors are the production\n    hashes. Checksums cover original bytes, so JSON reformatting is not equivalent.\n    """\n    expected_hn_sha256 = _sha(expected_hn_sha256, "Expected hn_set hash")\n    expected_policy_sha256 = _sha(expected_policy_sha256, "Expected policy hash")\n    if not isinstance(hn_bytes, bytes) or not isinstance(policy_bytes, bytes):\n        raise ReferenceError("Reference documents must be immutable bytes.")\n    hn_digest, policy_digest = _digest(hn_bytes), _digest(policy_bytes)\n    if hn_digest != expected_hn_sha256:\n        raise ReferenceError("hn_set.json SHA-256 mismatch.")\n    if policy_digest != expected_policy_sha256:\n        raise ReferenceError("htq_policy.json SHA-256 mismatch.")\n    hn, policy = _document(hn_bytes, "hn_set.json"), _document(policy_bytes, "htq_policy.json")\n    _policy(hn, "hn_set.json"); _policy(policy, "htq_policy.json")\n    coordinates = _coordinates(hn.get("H_op"), "H_op")\n    if coordinates != _coordinates(hn.get("H_tq"), "H_tq"):\n        raise ReferenceError("H_op and H_tq must be identical, including coordinate order.")\n    if type(hn.get("n_H")) is not int or hn["n_H"] != N_COORDINATES:\n        raise ReferenceError("hn_set n_H must equal 286.")\n    if type(policy.get("n_H_op")) is not int or policy["n_H_op"] != N_COORDINATES:\n        raise ReferenceError("Policy n_H_op must equal 286.")\n    hn_model, policy_model = _model(hn, "hn_set.json"), _model(policy, "htq_policy.json")\n    if hn_model != policy_model:\n        raise ReferenceError("The two files have different model-file pins.")\n    return Reference(coordinates=coordinates, hn_sha256=hn_digest,\n                     policy_sha256=policy_digest, model=hn_model)\n\n\ndef load_reference(hn_set_path: str | Path, policy_path: str | Path) -> Reference:\n    """Load only the exact DSSE production files; no caller-supplied hash bypass."""\n    return verify_documents(Path(hn_set_path).read_bytes(), Path(policy_path).read_bytes(),\n                            expected_hn_sha256=HN_SET_SHA256, expected_policy_sha256=POLICY_SHA256)\n\n\ndef write_reference(reference: Reference, out: str | Path) -> None:\n    """Atomically write a binding to the caller\'s explicit private output path."""\n    path = Path(out)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = json.dumps(reference.to_dict(), indent=2, ensure_ascii=False, allow_nan=False) + "\\n"\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,\n                                         prefix="." + path.name + ".", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            handle.write(payload); handle.flush(); os.fsync(handle.fileno())\n        os.replace(temporary, path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--hn-set", required=True, type=Path)\n    parser.add_argument("--policy", required=True, type=Path)\n    parser.add_argument("--out", required=True, type=Path)\n    args = parser.parse_args(argv)\n    if args.out.resolve() in (args.hn_set.resolve(), args.policy.resolve()):\n        parser.error("--out must not overwrite a source document.")\n    try:\n        reference = load_reference(args.hn_set, args.policy)\n        write_reference(reference, args.out)\n    except (OSError, ReferenceError) as exc:\n        parser.error(str(exc))\n    print(f"Validated {len(reference.coordinates)} H_tq coordinates; private binding written to {args.out}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'attention_probe/fixed_refw.py': '"""Validate the private, fixed H286 REF-W extract without model imports.\n\nThe compact file contains existing population means/SDs, already cast to FP32\nas in DSSE\'s SetReader. This module neither fits nor recomputes normalization.\nCETT uses max(||down_proj(z)||, CETT_EPS); zCETT uses the saved c_mu and\nmax(c_sd, STANDARDIZATION_EPS). Zero saved SD is valid and stays zero here:\nthe consuming measurement applies the documented denominator floor.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nimport struct\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .reference import (\n    HN_SET_SHA256,\n    MLP_WIDTH,\n    N_COORDINATES,\n    N_LAYERS,\n    POLICY_SHA256,\n    Reference,\n    ReferenceError,\n)\n\nFIXED_REFW_SHA256 = "68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc"\nSOURCE_HN_META_SHA256 = "9c61bc34e804719e5bf5e72b644e8a38a94ad0f6c1708efaef75386acdae49ff"\nCETT_EPS = 1e-9\nSTANDARDIZATION_EPS = 1e-6\nSCHEMA = "dsse_h286_refw_v1"\n_VECTOR_NAMES = {"z_mu", "z_sd", "c_mu", "c_sd"}\n_FIELDS = {"schema", "source_hn_meta_sha256", "hn_sha256", "coordinates", "refw"}\n\n\nclass FixedRefWError(ReferenceError):\n    """The supplied extract does not establish the fixed normalization."""\n\n\ndef _sha(value: Any, label: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:\n        raise FixedRefWError(f"{label} must be a lowercase SHA-256 digest.")\n    return value\n\n\ndef _object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:\n    result = {}\n    for key, value in pairs:\n        if key in result:\n            raise FixedRefWError("Duplicate JSON object key.")\n        result[key] = value\n    return result\n\n\ndef _reject_constant(value: str) -> Any:\n    raise FixedRefWError(f"Non-finite JSON constant: {value}")\n\n\ndef verify_fixed_refw(\n    raw: bytes,\n    reference: Reference,\n    *,\n    expected_sha256: str,\n    expected_source_hn_meta_sha256: str,\n    expected_hn_sha256: str,\n) -> dict[str, Any]:\n    """Low-level validation with explicit trust anchors for synthetic tests.\n\n    Production callers must use load_fixed_refw(), which fixes all three\n    anchors. The source hash binds the previously verified full hn_meta file;\n    this compact loader does not claim to inspect that large file again.\n    """\n    expected_sha256 = _sha(expected_sha256, "Expected compact hash")\n    expected_source_hn_meta_sha256 = _sha(expected_source_hn_meta_sha256, "Expected source hash")\n    expected_hn_sha256 = _sha(expected_hn_sha256, "Expected H selection hash")\n    if not isinstance(raw, bytes):\n        raise FixedRefWError("The compact document must be immutable bytes.")\n    if hashlib.sha256(raw).hexdigest() != expected_sha256:\n        raise FixedRefWError("Fixed REF-W SHA-256 mismatch.")\n    if not isinstance(reference, Reference) or reference.hn_sha256 != expected_hn_sha256:\n        raise FixedRefWError("Reference has a different H selection hash.")\n    try:\n        document = json.loads(raw, object_pairs_hook=_object, parse_constant=_reject_constant)\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise FixedRefWError("Fixed REF-W is not valid JSON.") from exc\n    if not isinstance(document, dict) or set(document) != _FIELDS:\n        raise FixedRefWError("Unexpected fixed REF-W document fields.")\n    if document["schema"] != SCHEMA:\n        raise FixedRefWError("Unsupported fixed REF-W schema.")\n    if document["source_hn_meta_sha256"] != expected_source_hn_meta_sha256:\n        raise FixedRefWError("Full hn_meta source hash mismatch.")\n    if document["hn_sha256"] != expected_hn_sha256:\n        raise FixedRefWError("Compact H selection hash mismatch.")\n\n    coordinates = document["coordinates"]\n    if not isinstance(coordinates, list) or len(coordinates) != N_COORDINATES:\n        raise FixedRefWError("Expected exactly 286 ordered coordinates.")\n    checked = []\n    for coordinate in coordinates:\n        if (not isinstance(coordinate, list) or len(coordinate) != 2\n                or any(type(value) is not int for value in coordinate)):\n            raise FixedRefWError("Coordinates must be integer pairs.")\n        layer, neuron = coordinate\n        if not (0 <= layer < N_LAYERS and 0 <= neuron < MLP_WIDTH):\n            raise FixedRefWError("Coordinate outside the Qwen3-1.7B MLP architecture.")\n        checked.append((layer, neuron))\n    if len(set(checked)) != N_COORDINATES or tuple(checked) != reference.coordinates:\n        raise FixedRefWError("Coordinates differ from the ordered H286 reference.")\n\n    vectors = document["refw"]\n    if not isinstance(vectors, dict) or set(vectors) != _VECTOR_NAMES:\n        raise FixedRefWError("Expected exactly z_mu, z_sd, c_mu and c_sd.")\n    for name, vector in vectors.items():\n        if not isinstance(vector, list) or len(vector) != N_COORDINATES:\n            raise FixedRefWError(f"{name} must have exactly 286 values.")\n        for value in vector:\n            if type(value) not in (int, float):\n                raise FixedRefWError(f"{name} must contain finite FP32 numbers.")\n            try:\n                fp32 = struct.unpack("<f", struct.pack("<f", value))[0]\n            except (OverflowError, struct.error) as exc:\n                raise FixedRefWError(f"{name} contains a value outside FP32.") from exc\n            if not math.isfinite(fp32) or fp32 != value:\n                raise FixedRefWError(f"{name} contains a non-finite or non-FP32 value.")\n            if name.endswith("_sd") and value < 0:\n                raise FixedRefWError(f"{name} cannot contain negative standard deviations.")\n    return document\n\n\ndef load_fixed_refw(path: str | Path, reference: Reference) -> dict[str, Any]:\n    """Load only the exact production extract bound to the original reference."""\n    if not isinstance(reference, Reference) or reference.policy_sha256 != POLICY_SHA256:\n        raise FixedRefWError("Expected the original production reference policy.")\n    return verify_fixed_refw(\n        Path(path).read_bytes(), reference,\n        expected_sha256=FIXED_REFW_SHA256,\n        expected_source_hn_meta_sha256=SOURCE_HN_META_SHA256,\n        expected_hn_sha256=HN_SET_SHA256,\n    )\n', 'attention_probe/interventions.py': '"""One-pass attention pulses and read-only activation capture for Qwen3.\n\nPositions are zero-based columns of the complete input sequence, not relative\nindices of a cached decode chunk. Selected positions must be real tokens in\nevery batch row. Qwen3 queries are scaled *after* q_norm and before RoPE, so\nnormalization cannot cancel the pulse. Scaling queries scales their attention\nlogits; it makes no claim about FLOPs, energy, or a computational bottleneck.\n\nCoordinateCapture stores raw gated-MLP z at down_proj input, not CETT.\n"""\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nimport math\nimport inspect\nfrom numbers import Integral\nfrom typing import Iterable, Sequence\n\nimport torch\n\n\ndef _integers(values: Iterable[int], name: str, *, nonempty: bool = True) -> tuple[int, ...]:\n    result = tuple(values)\n    if nonempty and not result:\n        raise ValueError(f"{name} must not be empty")\n    if any(isinstance(x, bool) or not isinstance(x, Integral) or x < 0 for x in result):\n        raise ValueError(f"{name} must contain nonnegative integers")\n    if len(set(result)) != len(result):\n        raise ValueError(f"{name} must not contain duplicates")\n    return tuple(int(x) for x in result)\n\n\ndef _layers(model):\n    base = getattr(model, "model", model)\n    layers = getattr(base, "layers", None)\n    if layers is None or not len(layers):\n        raise ValueError("Expected Hugging Face decoder model.model.layers (or model.layers)")\n    return layers\n\n\ndef _cache_length(cache) -> int | None:\n    if cache is None:\n        return 0\n    getter = getattr(cache, "get_seq_length", None)\n    if getter is not None:\n        try:\n            return int(getter())\n        except (TypeError, ValueError, RuntimeError):\n            return None\n    if isinstance(cache, (tuple, list)):\n        if not cache:\n            return 0\n        try:\n            return int(cache[0][0].shape[-2])\n        except (IndexError, AttributeError, TypeError):\n            return None\n    return None\n\n\nclass _OneFullPass:\n    """Bind hooks to one complete, uncached outer model forward and clean them up."""\n\n    def __init__(self, model, positions: Sequence[int]):\n        self.model = model\n        self.positions = _integers(positions, "positions")\n        self._handles = []\n        self._entered = False\n        self._used = False\n        self._claimed = False\n        self._window = None\n        self.full_passes = 0\n        self.skipped_cached_passes = 0\n\n    def _model_pre(self, module, args, kwargs):\n        self._window = None\n        if self._claimed:\n            return\n        # HF forwards allow both positional and keyword cache arguments.\n        try:\n            named = inspect.signature(module.forward).bind_partial(*args, **kwargs).arguments\n        except (TypeError, ValueError):\n            named = kwargs\n        fields = dict(kwargs)\n        fields.update({name: value for name, value in named.items() if name not in ("args", "kwargs")})\n        cache = fields.get("past_key_values", fields.get("past_key_value"))\n        length = _cache_length(cache)\n        cache_position = fields.get("cache_position")\n        nonzero_offset = (cache_position is not None and torch.as_tensor(cache_position).numel()\n                          and int(torch.as_tensor(cache_position).reshape(-1)[0]) != 0)\n        if length is None or length > 0 or nonzero_offset:\n            self.skipped_cached_passes += 1\n            return\n        ids = fields.get("input_ids")\n        embeds = fields.get("inputs_embeds")\n        value = ids if ids is not None else embeds\n        if value is None and args:\n            value = args[0]\n        if not isinstance(value, torch.Tensor) or value.ndim not in (2, 3):\n            raise ValueError("Cannot identify full input sequence for absolute token positions")\n        batch, tokens = value.shape[:2]\n        if max(self.positions) >= tokens:\n            raise ValueError(f"position {max(self.positions)} is outside the full sequence of {tokens} tokens")\n        mask = fields.get("attention_mask")\n        if mask is not None:\n            if tuple(mask.shape) != (batch, tokens):\n                raise ValueError("Expected a two-dimensional outer attention mask")\n            if not bool(mask[:, list(self.positions)].bool().all()):\n                raise ValueError("Selected absolute position is padding in at least one batch row")\n        self._claimed = True\n        self.full_passes += 1\n        self._window = (int(batch), int(tokens))\n\n    def _model_post(self, module, args, kwargs, output):\n        self._window = None\n\n    def _install(self):\n        raise NotImplementedError\n\n    def validate_fired(self):\n        raise NotImplementedError\n\n    def __enter__(self):\n        if self._used:\n            raise RuntimeError("Use a fresh hook context for each attempt")\n        if self.model.training:\n            raise ValueError("Attention pilot requires model.eval(); model mode is never changed by hooks")\n        self._used = self._entered = True\n        try:\n            self._handles.append(self.model.register_forward_pre_hook(self._model_pre, with_kwargs=True))\n            self._handles.append(self.model.register_forward_hook(self._model_post, with_kwargs=True, always_call=True))\n            self._install()\n        except BaseException:\n            self.close()\n            raise\n        return self\n\n    def close(self):\n        for handle in reversed(self._handles):\n            handle.remove()\n        self._handles.clear()\n        self._window = None\n        self._entered = False\n\n    def __exit__(self, kind, error, tb):\n        self.close()\n        if kind is None:\n            self.validate_fired()\n        return False\n\n\nclass AttentionPulse(_OneFullPass):\n    """Scale post-q_norm queries, or add a per-head matched random delta, once.\n\n    Primary architecture is config.model_type=\'qwen3\'. Qwen2\'s unnormalized\n    q_proj output is supported only with explicit architecture=\'qwen2\'. Random\n    deltas match ||(beta-1) q|| independently for each selected token/query head\n    before output-dtype rounding. actual_delta_norms reports rounding effects.\n    A local CPU generator never reads or changes PyTorch\'s global RNG state.\n    """\n\n    def __init__(self, model, block: int, positions: Sequence[int], beta: float = 1.0,\n                 mode: str = "scale", seed: int = 0, *, architecture: str = "qwen3"):\n        super().__init__(model, positions)\n        if not isinstance(beta, (int, float)) or not math.isfinite(beta) or not 0 <= beta <= 1:\n            raise ValueError("beta must be finite and between zero and one")\n        if mode not in ("scale", "random"):\n            raise ValueError("mode must be \'scale\' or \'random\'")\n        if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:\n            raise ValueError("seed must be a nonnegative integer")\n        self.layers = _layers(model)\n        self.block = _integers((block,), "block")[0]\n        if self.block >= len(self.layers):\n            raise ValueError("block is outside the decoder")\n        if architecture not in ("qwen3", "qwen2") or getattr(model.config, "model_type", None) != architecture:\n            raise ValueError("Unsupported architecture: Qwen3 is primary; Qwen2 requires explicit opt-in")\n        self.architecture, self.mode, self.beta = architecture, mode, float(beta)\n        self.seed = int(seed)\n        attention = self.layers[self.block].self_attn\n        self.heads = int(model.config.num_attention_heads)\n        self.head_dim = int(getattr(attention, "head_dim", getattr(model.config, "head_dim", 0)))\n        if self.heads <= 0 or self.head_dim <= 0:\n            raise ValueError("Query-head shape is not declared")\n        if architecture == "qwen3":\n            self.hook_module = getattr(attention, "q_norm", None)\n            self.hook_location = "self_attn.q_norm.output (B,T,H,D), before RoPE"\n            if self.hook_module is None:\n                raise ValueError("Qwen3 requires post-q_norm intervention; q_proj fallback is forbidden")\n        else:\n            if getattr(attention, "q_norm", None) is not None:\n                raise ValueError("Qwen2 fallback expects no query normalization")\n            self.hook_module = getattr(attention, "q_proj", None)\n            self.hook_location = "self_attn.q_proj.output, explicit Qwen2 fallback"\n            if self.hook_module is None:\n                raise ValueError("Qwen2 query projection is absent")\n        self.fired_count = 0\n        self.requested_delta_norms = None\n        self.actual_delta_norms = None\n        self._generator = torch.Generator(device="cpu").manual_seed(self.seed)\n\n    def _install(self):\n        self._handles.append(self.hook_module.register_forward_hook(self._pulse))\n\n    def _pulse(self, module, args, output):\n        if self._window is None or self.fired_count:\n            return None\n        batch, tokens = self._window\n        if not isinstance(output, torch.Tensor):\n            raise ValueError("Query hook did not return a tensor")\n        shape = (batch, tokens, self.heads, self.head_dim)\n        if self.architecture == "qwen3":\n            if tuple(output.shape) != shape:\n                raise ValueError(f"Unsupported Qwen3 post-q_norm layout: expected {shape}, got {tuple(output.shape)}")\n            query = output\n        else:\n            if tuple(output.shape) != (batch, tokens, self.heads * self.head_dim):\n                raise ValueError("Unsupported Qwen2 q_proj output shape")\n            query = output.reshape(shape)\n        selected = query[:, list(self.positions), :, :]\n        if not bool(torch.isfinite(selected).all()):\n            raise ValueError("Nonfinite query in selected pulse positions")\n        self.fired_count += 1\n        norms = (1-self.beta) * selected.detach().double().cpu().norm(dim=-1)\n        self.requested_delta_norms = norms.clone()\n        if self.beta == 1:\n            self.actual_delta_norms = torch.zeros_like(norms)\n            return None  # Exact sham: preserve the original tensor, no multiplication or RNG draw.\n        if self.mode == "scale":\n            changed = selected * self.beta\n        else:\n            random = torch.randn(selected.shape, generator=self._generator, dtype=torch.float64, device="cpu")\n            random = random / random.norm(dim=-1, keepdim=True).clamp_min(torch.finfo(torch.float64).tiny)\n            delta = (random * norms[..., None]).to(device=selected.device, dtype=selected.dtype)\n            changed = selected + delta\n        self.actual_delta_norms = (changed.detach().double().cpu()-selected.detach().double().cpu()).norm(dim=-1)\n        result = query.clone()\n        result[:, list(self.positions), :, :] = changed\n        return result if self.architecture == "qwen3" else result.reshape(output.shape)\n\n    def validate_fired(self):\n        if self.fired_count != 1:\n            raise RuntimeError(f"Attention pulse must fire once on a full/prefill pass; fired {self.fired_count}")\n        return self.fired_count\n\n\nclass CoordinateCapture(_OneFullPass):\n    """Read exact ordered down_proj-input coordinates and decoder residuals.\n\n    values is CPU float32 [batch, positions, coordinates]; coordinate order is\n    exactly the supplied order. residuals maps each requested decoder layer to\n    CPU float32 [batch, positions, hidden_size]. No input/output tensor changes.\n    This is raw z capture, not CETT or a causal-neuron importance estimate.\n    """\n\n    feature_kind = "raw gated-MLP activation z at down_proj input; not CETT"\n\n    def __init__(self, model, coordinates: Sequence[tuple[int, int]], positions: Sequence[int],\n                 residual_layers: Sequence[int] | None = None, *, expected_count: int = 286):\n        super().__init__(model, positions)\n        self.layers = _layers(model)\n        if getattr(model.config, "model_type", None) != "qwen3":\n            raise ValueError("Original H coordinates require the Qwen3 decoder architecture")\n        if isinstance(expected_count, bool) or not isinstance(expected_count, Integral) or expected_count < 1:\n            raise ValueError("expected_count must be a positive integer")\n        self.coordinates = tuple(tuple(pair) for pair in coordinates)\n        if len(self.coordinates) != expected_count or not self.coordinates:\n            raise ValueError(f"Expected exactly {expected_count} ordered H coordinates")\n        if len(set(self.coordinates)) != len(self.coordinates):\n            raise ValueError("Duplicate H coordinates")\n        self._by_layer = defaultdict(list)\n        for output_index, pair in enumerate(self.coordinates):\n            if len(pair) != 2:\n                raise ValueError("Each coordinate must be (block, down_proj_input_index)")\n            # A coordinate may have equal block/channel numbers; duplicates are\n            # forbidden across coordinate pairs, not within a pair.\n            if any(isinstance(x, bool) or not isinstance(x, Integral) or x < 0 for x in pair):\n                raise ValueError("Coordinate indices must be nonnegative integers")\n            block, neuron = (int(x) for x in pair)\n            if block >= len(self.layers):\n                raise ValueError("Coordinate block is outside the decoder")\n            down = getattr(getattr(self.layers[block], "mlp", None), "down_proj", None)\n            if down is None or not hasattr(down, "in_features") or neuron >= down.in_features:\n                raise ValueError("Coordinate channel is outside mlp.down_proj input")\n            self._by_layer[block].append((output_index, neuron))\n        self.residual_layers = _integers(residual_layers if residual_layers is not None else sorted(self._by_layer),\n                                        "residual_layers", nonempty=False)\n        if any(block >= len(self.layers) for block in self.residual_layers):\n            raise ValueError("Residual layer is outside the decoder")\n        self._values = None\n        self.residuals = {}\n        self._seen = set()\n        self.fired_count = 0\n\n    def _install(self):\n        for block in self._by_layer:\n            self._handles.append(self.layers[block].mlp.down_proj.register_forward_pre_hook(self._coordinate_hook(block)))\n        for block in self.residual_layers:\n            self._handles.append(self.layers[block].register_forward_hook(self._residual_hook(block)))\n\n    def _coordinate_hook(self, block):\n        def hook(module, args):\n            if self._window is None:\n                return None\n            if block in self._seen:\n                raise RuntimeError("down_proj was called twice in the selected full pass")\n            z = args[0]\n            batch, tokens = self._window\n            if not isinstance(z, torch.Tensor) or z.ndim != 3 or tuple(z.shape[:2]) != (batch, tokens):\n                raise ValueError("Unsupported down_proj input shape")\n            pairs = self._by_layer[block]\n            observed = z[:, list(self.positions), :][:, :, [channel for _, channel in pairs]].detach().float().cpu().clone()\n            if not bool(torch.isfinite(observed).all()):\n                raise ValueError("Nonfinite selected raw activation")\n            if self._values is None:\n                self._values = torch.empty((batch, len(self.positions), len(self.coordinates)), dtype=torch.float32)\n            self._values[:, :, [index for index, _ in pairs]] = observed\n            self._seen.add(block)\n            self.fired_count = int(self._seen == set(self._by_layer))\n            return None\n        return hook\n\n    def _residual_hook(self, block):\n        def hook(module, args, output):\n            if self._window is None:\n                return None\n            if block in self.residuals:\n                raise RuntimeError("Decoder layer was called twice in the selected full pass")\n            value = output[0] if isinstance(output, (tuple, list)) else output\n            if not isinstance(value, torch.Tensor) or value.ndim != 3 or tuple(value.shape[:2]) != self._window:\n                raise ValueError("Unsupported decoder residual output shape")\n            observed = value[:, list(self.positions)].detach().float().cpu().clone()\n            if not bool(torch.isfinite(observed).all()):\n                raise ValueError("Nonfinite selected decoder residual")\n            self.residuals[block] = observed\n            return None\n        return hook\n\n    @property\n    def values(self):\n        self.validate_fired()\n        return self._values\n\n    def validate_fired(self):\n        if self.fired_count != 1 or self._seen != set(self._by_layer) or set(self.residuals) != set(self.residual_layers):\n            raise RuntimeError("Raw-coordinate capture did not complete exactly one full/prefill pass")\n        return self.fired_count\n', 'qwen3_h286_runtime.py': '"""Short exploratory ORIGINAL Qwen3/H286 pilot; embedded notebook entry point.\n\nNo re-selection, fitting, generation, training, or cross-model coordinate mapping.\nOriginal FP32, TF32-off, fixed REF-W and efficient SDPA are retained. The two\ntechnical checks precede a fixed 16-request/five-arm exploratory run. Results do\nnot constitute a held-out confirmatory test or a proven attention bottleneck.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport datetime as dt\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport shutil\nimport tempfile\nimport time\nimport traceback\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nMODEL = \'Qwen/Qwen3-1.7B\'\nREVISION = \'70d244cc86ccca08cf5af4e1e306ecf908b1ad5e\'\nTRANSFORMERS = \'5.22.2\'\nACCELERATE = \'1.16.0\'\nREFW_SHA256 = \'68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc\'\nMETA_SHA256 = \'9c61bc34e804719e5bf5e72b644e8a38a94ad0f6c1708efaef75386acdae49ff\'\nSEED = 20261117\nARMS = (\n    {\'name\': \'sham\', \'block\': 14, \'beta\': 1., \'mode\': \'scale\'},\n    {\'name\': \'soft_103\', \'block\': 14, \'beta\': 1 / 1.03, \'mode\': \'scale\'},\n    {\'name\': \'soft_108\', \'block\': 14, \'beta\': 1 / 1.08, \'mode\': \'scale\'},\n    {\'name\': \'random_matched_108\', \'block\': 14, \'beta\': 1 / 1.08, \'mode\': \'random\'},\n    {\'name\': \'other_block_108\', \'block\': 7, \'beta\': 1 / 1.08, \'mode\': \'scale\'},\n)\n# Declared before any new model result. These are technical/exploratory items,\n# not the frozen DSSE evaluation sample or a benchmark accuracy test.\nITEMS_SHA256 = \'12a3cba862185b9f25ca39ea3ea49bd3bb6fc5f248c12dbbe02bcc8836663fb3\'\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=False, allow_nan=False).encode()\n\n\ndef sha_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 << 20), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent, prefix=\'.\' + path.name, delete=False) as handle:\n        temporary = Path(handle.name)\n        handle.write(canonical(value) + b\'\\n\')\n        handle.flush()\n        os.fsync(handle.fileno())\n    try:\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef archive(output):\n    target = output / \'qwen3_h286_results.zip\'\n    temporary = output / \'.qwen3_h286_results.zip.tmp\'\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_DEFLATED) as bundle:\n        for item in sorted(output.rglob(\'*\')):\n            if item.is_file() and item not in (target, temporary) and not any(part.startswith(\'.\') for part in item.relative_to(output).parts):\n                bundle.write(item, item.relative_to(output))\n    os.replace(temporary, target)\n    return target\n\n\ndef load_inputs(hn_path, policy_path, refw_path, items_path):\n    from attention_probe.fixed_refw import load_fixed_refw\n    from attention_probe.reference import load_reference\n    reference = load_reference(hn_path, policy_path)\n    stats = load_fixed_refw(refw_path, reference)\n    raw = Path(items_path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != ITEMS_SHA256:\n        raise ValueError(\'The exact prespecified 16-item file is required; no replacement or post-result selection\')\n    items = json.loads(raw)\n    if len(items) != 16 or len({item[\'id\'] for item in items}) != 16:\n        raise ValueError(\'Exactly 16 unique fixed requests are required\')\n    return reference, stats, items\n\n\ndef hardware_preflight(torch, cache_dir):\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA GPU erforderlich; keine Warteschleife. A100 empfohlen, L4 ausreichend.\')\n    free, total = torch.cuda.mem_get_info(0)\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    disk = shutil.disk_usage(cache_dir).free\n    if free < 10 * (1 << 30):\n        raise RuntimeError(\'Mindestens 10 GiB freie GPU-VRAM für unverändertes FP32 erforderlich.\')\n    if disk < 8 * (1 << 30):\n        raise RuntimeError(\'Mindestens 8 GiB freier Cache-Speicher erforderlich; keine Gewichte heruntergeladen.\')\n    available = None\n    meminfo = Path(\'/proc/meminfo\')\n    if meminfo.exists():\n        fields = dict(line.split(\':\', 1) for line in meminfo.read_text().splitlines())\n        available = int(fields[\'MemAvailable\'].split()[0]) * 1024\n        if available < 10 * (1 << 30):\n            raise RuntimeError(\'Mindestens 10 GiB verfügbarer Host-RAM erforderlich.\')\n    return {\'gpu\': torch.cuda.get_device_name(0), \'free_vram\': free, \'total_vram\': total,\n            \'free_disk\': disk, \'available_host_ram\': available, \'device\': \'cuda:0\'}\n\n\ndef model_load_kwargs(torch):\n    return {\'revision\': REVISION, \'dtype\': torch.float32, \'device_map\': {\'\': 0},\n            \'attn_implementation\': \'sdpa\', \'trust_remote_code\': False, \'use_safetensors\': True,\n            \'local_files_only\': True}\n\n\ndef load_model(reference, cache_dir):\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer\n    snapshot = Path(snapshot_download(MODEL, revision=REVISION, cache_dir=str(cache_dir),\n                                     allow_patterns=[\'*.json\', \'*.safetensors\', \'*.jinja\']))\n    if snapshot.name != REVISION:\n        raise ValueError(\'HF snapshot path does not identify the exact pinned revision\')\n    for filename, expected in reference.model.sha256:\n        if sha_file(snapshot / filename) != expected:\n            raise ValueError(\'Original model/tokenizer SHA mismatch: \' + filename)\n    config = AutoConfig.from_pretrained(str(snapshot), revision=REVISION, trust_remote_code=False,\n                                       local_files_only=True)\n    if config.model_type != \'qwen3\' or config.num_hidden_layers != 28 or config.intermediate_size != 6144:\n        raise ValueError(\'Original Qwen3-1.7B architecture mismatch\')\n    tokenizer = AutoTokenizer.from_pretrained(str(snapshot), revision=REVISION, trust_remote_code=False,\n                                             local_files_only=True)\n    tokenizer.padding_side = \'left\'\n    model = AutoModelForCausalLM.from_pretrained(str(snapshot), config=config, **model_load_kwargs(torch))\n    model.eval().requires_grad_(False)\n    if any(parameter.device.type != \'cuda\' or parameter.device.index != 0 or parameter.dtype != torch.float32\n           for parameter in model.parameters()):\n        raise ValueError(\'All model parameters must remain full FP32 on cuda:0; no offload/quantization\')\n    return tokenizer, model\n\n\ndef prepare_request(tokenizer, index, item):\n    import torch\n    category, question, answer = item[\'category\'], item[\'prompt\'], item[\'target\']\n    formatted = tokenizer.apply_chat_template([{\'role\': \'user\', \'content\': question}], tokenize=False,\n                                              add_generation_prompt=True, enable_thinking=False)\n    prompt = tokenizer(formatted, add_special_tokens=False)[\'input_ids\']\n    answer_ids = tokenizer(answer, add_special_tokens=False)[\'input_ids\']\n    if not prompt or not answer_ids or len(prompt) + len(answer_ids) > 128:\n        raise ValueError(\'Unexpected empty or long fixed teacher-forcing request; no silent truncation\')\n    # Token at p predicts token p+1. The last answer activation is recorded but\n    # cannot cause its own prediction. Answer likelihood uses preceding positions.\n    ids = prompt + answer_ids\n    positions = list(range(max(0, len(ids) - 4), len(ids)))\n    return {\'request_id\': f\'r{index:03d}\', \'item_id\': item[\'id\'], \'pair_id\': item[\'pair_id\'], \'category\': category, \'question\': question, \'answer\': answer,\n            \'formatted_prompt\': formatted, \'formatted_prompt_sha256\': hashlib.sha256(formatted.encode()).hexdigest(), \'prompt_ids\': prompt, \'answer_ids\': answer_ids,\n            \'input_ids\': ids, \'capture_positions\': positions, \'pulse_positions\': [len(prompt) - 1],\n            \'answer_logit_positions\': list(range(len(prompt) - 1, len(ids) - 1)),\n            \'thinking_enabled\': False, \'ids\': torch.tensor([ids], dtype=torch.long)}\n\n\n@contextlib.contextmanager\ndef cett_tap(model, coordinates, positions, weight_norms=None):\n    import torch\n    groups = defaultdict(list)\n    for order, (layer, neuron) in enumerate(coordinates):\n        groups[layer].append((order, neuron))\n    result = {\'cett\': torch.empty((1, len(positions), len(coordinates)), dtype=torch.float32),\n              \'denominator\': torch.empty((1, len(positions), len(coordinates)), dtype=torch.float32)}\n    handles, seen = [], set()\n    try:\n        for layer, pairs in groups.items():\n            down = model.model.layers[layer].mlp.down_proj\n            columns = [neuron for _, neuron in pairs]\n            weight_norm = (weight_norms[layer] if weight_norms is not None\n                           else down.weight.float().norm(dim=0)[columns])\n\n            def hook(_module, args, output, layer=layer, pairs=pairs, columns=columns, weight_norm=weight_norm):\n                if layer in seen:\n                    raise ValueError(\'CETT tap fired repeatedly\')\n                z = args[0][:, positions][:, :, columns].abs().float()\n                den = output.norm(dim=-1)[:, positions].float()\n                contribution = z * weight_norm / den.clamp_min(1e-9)[..., None]\n                order = [index for index, _ in pairs]\n                result[\'cett\'][:, :, order] = contribution.detach().cpu()\n                result[\'denominator\'][:, :, order] = den[..., None].expand_as(contribution).detach().cpu()\n                seen.add(layer)\n            handles.append(down.register_forward_hook(hook))\n        yield result\n        if seen != set(groups):\n            raise ValueError(\'CETT capture incomplete\')\n    finally:\n        for handle in reversed(handles):\n            handle.remove()\n\n\ndef parameter_evidence(model):\n    return tuple((name, id(value), value.data_ptr(), value._version, str(value.dtype), str(value.device))\n                 for name, value in model.named_parameters())\n\n\ndef run_arm(model, request, coordinates, stats, arm, *, efficient_gpu=True, include_logits=False, weight_norms=None):\n    import torch\n\n    from attention_probe.interventions import AttentionPulse, CoordinateCapture\n    device = next(model.parameters()).device\n    ids = request[\'ids\'].to(device)\n    attention = (torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION)\n                 if efficient_gpu else contextlib.nullcontext())\n    started = time.monotonic()\n    with (torch.inference_mode(), attention,\n          CoordinateCapture(model, coordinates, request[\'capture_positions\'],\n                            residual_layers=tuple(layer for layer in (3, 7, 14, 27) if layer < len(model.model.layers)),\n                            expected_count=len(coordinates)) as capture,\n          cett_tap(model, coordinates, request[\'capture_positions\'], weight_norms) as contributions,\n          AttentionPulse(model, arm[\'block\'], request[\'pulse_positions\'], arm[\'beta\'], arm[\'mode\'],\n                         seed=SEED + int(request[\'request_id\'][1:])) as pulse):\n        output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)\n        if not bool(torch.isfinite(output.logits).all()):\n            raise ValueError(\'Nonfinite model logits\')\n        selected_logits = output.logits[:, request[\'answer_logit_positions\']].float()\n        targets = torch.tensor(request[\'answer_ids\'], device=device)[None, :, None]\n        logprob = selected_logits.log_softmax(-1).gather(-1, targets).squeeze(-1).cpu()\n        raw = capture.values\n        cett = contributions[\'cett\']\n        mu = torch.tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n        sd = torch.tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32).clamp_min(1e-6)\n        zcett = (cett - mu) / sd\n        result = {\'arm\': arm, \'answer_token_logprobs\': logprob.flatten().tolist(),\n                  \'answer_logprob_sum\': float(logprob.sum()), \'answer_logprob_mean\': float(logprob.mean()),\n                  \'pulse_fired\': pulse.fired_count, \'capture_fired\': capture.fired_count,\n                  \'requested_delta_norms\': pulse.requested_delta_norms.tolist(),\n                  \'actual_delta_norms\': pulse.actual_delta_norms.tolist(),\n                  \'elapsed_seconds\': time.monotonic() - started}\n        arrays = {\'raw_z\': raw.numpy(), \'cett\': cett.numpy(), \'zcett\': zcett.numpy(),\n                  \'down_output_norm_raw\': contributions[\'denominator\'].numpy(),\n                  \'gold_token_logprob\': logprob.numpy()}\n        for layer, value in capture.residuals.items():\n            arrays[f\'residual_layer_{layer}\'] = value.numpy()\n        if include_logits:\n            arrays[\'technical_logits\'] = selected_logits.detach().cpu().numpy()\n    return result, arrays\n\n\ndef plan_document(reference, stats, items):\n    import attention_probe.reference as reference_module\n    module_root = Path(reference_module.__file__).parent\n    module_hashes = {name: sha_file(module_root / name) for name in (\'reference.py\', \'interventions.py\', \'fixed_refw.py\')}\n    return {\'schema\': \'qwen3_h286_short_pilot_v1\', \'model\': reference.model.to_dict(),\n            \'hn_sha256\': reference.hn_sha256, \'policy_sha256\': reference.policy_sha256,\n            \'refw_sha256\': REFW_SHA256, \'refw_source_sha256\': stats[\'source_hn_meta_sha256\'],\n            \'coordinates\': [list(pair) for pair in reference.coordinates], \'seed\': SEED,\n            \'arms\': list(ARMS), \'items\': items, \'items_sha256\': ITEMS_SHA256,\n            \'helper_sha256\': sha_file(__file__), \'module_sha256\': module_hashes,\n            \'dtype\': \'float32\', \'tf32\': False, \'attention\': \'sdpa/EFFICIENT_ATTENTION\',\n            \'transformers\': TRANSFORMERS, \'thinking_enabled\': False,\n            \'claim\': \'externally supplied draft-checking proxy; not own prior output or proof of self-monitoring; no held-out confirmatory test\',\n            \'candidate_block\': \'14 zero-based, prespecified candidate; not a proven bottleneck\',\n            \'cett\': \'abs(z)*norm(Wdown_col)/max(norm(down(z)),1e-9)\',\n            \'zcett\': \'(CETT-fixed_REF_W_c_mu)/max(fixed_REF_W_c_sd,1e-6)\'}\n\n\ndef verified_request(output, request_id, plan_hash):\n    directory = output / \'requests\' / request_id\n    if not directory.exists():\n        return None\n    record = json.loads((directory / \'result.json\').read_text())\n    if (record.get(\'state\') != \'complete\' or record.get(\'plan_sha256\') != plan_hash\n            or record.get(\'request_id\') != request_id or len(record.get(\'arms\', [])) != 5\n            or [row[\'arm\'] for row in record[\'arms\']] != list(ARMS)):\n        raise ValueError(\'Existing request does not match fixed all-five-arm plan: \' + request_id)\n    proof = record.get(\'technical_checks\')\n    if not isinstance(proof, dict) or proof.get(\'state\') != \'passed\' or proof.get(\'plan_sha256\') != plan_hash:\n        raise ValueError(\'Request lacks bound technical-check evidence: \' + request_id)\n    if sha_file(directory / \'arrays.npz\') != record[\'npz_sha256\']:\n        raise ValueError(\'Existing request array SHA mismatch: \' + request_id)\n    return record\n\n\ndef restore_resume(resume_zip, output, plan_hash):\n    if not resume_zip:\n        return\n    with zipfile.ZipFile(resume_zip) as bundle:\n        source_plan = json.loads(bundle.read(\'run_plan.json\'))\n        if hashlib.sha256(canonical(source_plan)).hexdigest() != plan_hash:\n            raise ValueError(\'Resume ZIP belongs to a different immutable plan/reference\')\n        for index in range(16):\n            request_id = f\'r{index:03d}\'\n            prefix = f\'requests/{request_id}/\'\n            names = [prefix + \'result.json\', prefix + \'arrays.npz\']\n            present = [name in bundle.namelist() for name in names]\n            if not any(present):\n                continue\n            if not all(present):\n                raise ValueError(\'Resume ZIP contains an incomplete request chunk\')\n            destination = output / \'requests\' / request_id\n            if destination.exists():\n                verified_request(output, request_id, plan_hash)\n                if any((destination / Path(name).name).read_bytes() != bundle.read(name) for name in names):\n                    raise ValueError(\'Resume ZIP conflicts with existing complete request; no overwrite\')\n                continue\n            parent = destination.parent\n            parent.mkdir(parents=True, exist_ok=True)\n            temporary = Path(tempfile.mkdtemp(prefix=\'.\' + request_id + \'_\', dir=parent))\n            try:\n                for name in names:\n                    if bundle.getinfo(name).file_size > 128 * (1 << 20):\n                        raise ValueError(\'Oversized request chunk in resume ZIP\')\n                    (temporary / Path(name).name).write_bytes(bundle.read(name))\n                record = json.loads((temporary / \'result.json\').read_text())\n                if record.get(\'plan_sha256\') != plan_hash or sha_file(temporary / \'arrays.npz\') != record.get(\'npz_sha256\'):\n                    raise ValueError(\'Resume request failed hash/plan verification\')\n                os.rename(temporary, destination)\n                verified_request(output, request_id, plan_hash)\n            finally:\n                if temporary.exists():\n                    shutil.rmtree(temporary)\n\n\ndef write_request(output, request, results, arrays, plan_hash, hardware, technical_checks=None):\n    import numpy as np\n    destination = output / \'requests\' / request[\'request_id\']\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists():\n        raise ValueError(\'Refusing to overwrite existing request outputs\')\n    temporary = Path(tempfile.mkdtemp(dir=destination.parent, prefix=\'.\' + destination.name + \'_\'))\n    try:\n        np.savez_compressed(temporary / \'arrays.npz\',\n                            **{key: np.stack([row[key] for row in arrays]) for key in arrays[0]})\n        with (temporary / \'arrays.npz\').open(\'rb\') as stream:\n            os.fsync(stream.fileno())\n        public_request = {key: value for key, value in request.items() if key != \'ids\'}\n        record = {\'state\': \'complete\', \'request_id\': request[\'request_id\'], \'request\': public_request,\n                  \'plan_sha256\': plan_hash, \'npz_sha256\': sha_file(temporary / \'arrays.npz\'), \'arms\': results,\n                  \'hardware\': hardware, \'technical_checks\': technical_checks, \'completed_utc\': dt.datetime.now(dt.UTC).isoformat()}\n        atomic_json(temporary / \'result.json\', record)\n        os.rename(temporary, destination)  # Both files become visible as one paired-request checkpoint.\n        return record\n    finally:\n        if temporary.exists():\n            shutil.rmtree(temporary)\n\n\ndef run(hn_set, policy, refw, items_path, out_dir, cache_dir=None, resume_zip=None):\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    status = {\'state\': \'running\', \'stage\': \'reference\', \'completed_requests\': 0,\n              \'confirmatory_eligible\': False, \'scope\': \'original_fixed_H286_exploratory_pilot\'}\n    metrics = {\'pretrained_model_run\': False, \'checks\': {}, \'no_training\': True}\n    started = time.monotonic()\n    def save():\n        atomic_json(output / \'status.json\', status)\n        atomic_json(output / \'metrics.json\', metrics)\n    save()\n    try:\n        reference, stats, items = load_inputs(hn_set, policy, refw, items_path)\n        plan = plan_document(reference, stats, items)\n        plan_hash = hashlib.sha256(canonical(plan)).hexdigest()\n        existing = output / \'run_plan.json\'\n        if existing.exists() and existing.read_bytes() != canonical(plan) + b\'\\n\':\n            raise ValueError(\'Existing output uses a different plan; choose a fresh directory\')\n        if not existing.exists():\n            atomic_json(existing, plan)\n        restore_resume(resume_zip, output, plan_hash)\n        done = [verified_request(output, f\'r{index:03d}\', plan_hash) for index in range(len(items))]\n        status[\'completed_requests\'] = sum(item is not None for item in done)\n        if all(item is not None for item in done):\n            metrics[\'technical_checks_from_completed_requests\'] = [row.get(\'technical_checks\') for row in done]\n            metrics[\'checks_executed_this_attempt\'] = False\n            if any(not isinstance(row.get(\'technical_checks\'), dict) or row[\'technical_checks\'].get(\'state\') != \'passed\'\n                   or row[\'technical_checks\'].get(\'plan_sha256\') != plan_hash for row in done):\n                raise ValueError(\'Completed resume lacks the bound technical-check evidence\')\n            status.update(state=\'complete\', stage=\'complete\', resumed_without_gpu=True)\n            return status\n        from importlib.metadata import version\n\n        import torch\n        import transformers\n        if transformers.__version__ != TRANSFORMERS or version(\'accelerate\') != ACCELERATE:\n            raise ValueError(\'Exact transformers5.22.2 and accelerate1.16.0 are required\')\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.set_float32_matmul_precision(\'highest\')\n        torch.manual_seed(SEED)\n        disable_bar = getattr(transformers.utils.logging, \'disable_progress_bar\', None)\n        if disable_bar is not None:\n            disable_bar()\n        metrics.update(torch=torch.__version__, transformers=transformers.__version__, python=platform.python_version())\n        status[\'stage\'] = \'hardware_preflight\'\n        cache = Path(cache_dir or Path.home() / \'.cache\' / \'h286_pilot\')\n        hardware = hardware_preflight(torch, cache)\n        metrics[\'hardware\'] = hardware\n        status[\'stage\'] = \'pinned_download_and_load\'\n        save()\n        os.environ[\'HF_DEACTIVATE_ASYNC_LOAD\'] = \'1\'\n        print(\'Status: Original Qwen3-1.7B laden, exakte Dateien prüfen; FP32/SDPA.\', flush=True)\n        tokenizer, model = load_model(reference, cache)\n        metrics[\'weights_loaded\'] = True\n        hardware.update(torch=torch.__version__, cuda=torch.version.cuda, python=platform.python_version(), compute_capability=list(torch.cuda.get_device_capability(0)), transformers=transformers.__version__, attention=\'sdpa/EFFICIENT_ATTENTION\', dtype=\'float32\', tf32=False)\n        groups = defaultdict(list)\n        for layer, neuron in reference.coordinates:\n            groups[layer].append(neuron)\n        with torch.inference_mode():\n            weight_norms = {layer: model.model.layers[layer].mlp.down_proj.weight.float().norm(dim=0)[columns]\n                            for layer, columns in groups.items()}\n        evidence = parameter_evidence(model)\n        first_index = next(index for index, result in enumerate(done) if result is None)\n        warm = prepare_request(tokenizer, first_index, items[first_index])\n        ids = warm[\'ids\'].to(\'cuda:0\')\n        status[\'stage\'] = \'warmup\'\n        save()\n        with torch.inference_mode(), torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION):\n            model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)  # untimed warmup\n            baseline = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits.detach().cpu()\n        metrics[\'pretrained_model_run\'] = True\n        status[\'stage\'] = \'technical_checks\'\n        print(\'Status: Zwei technische Prüfungen vor der festen 16-Request-Auswertung.\', flush=True)\n        _sham, sham_arrays = run_arm(model, warm, reference.coordinates, stats, ARMS[0], include_logits=True, weight_norms=weight_norms)\n        selected_baseline = baseline[:, warm[\'answer_logit_positions\']].float()\n        observed = torch.from_numpy(sham_arrays[\'technical_logits\'])\n        delta = (selected_baseline - observed).abs()\n        metrics[\'checks\'][\'sham\'] = {\'bitwise_equal\': torch.equal(selected_baseline, observed),\n                                     \'max_abs_logit_difference\': float(delta.max()), \'rtol\': 1e-5, \'atol\': 1e-6}\n        if not torch.allclose(selected_baseline, observed, rtol=1e-5, atol=1e-6):\n            raise ValueError(\'Technical sham logit check failed; exploratory queue not started\')\n        pulse_result, pulse_arrays = run_arm(model, warm, reference.coordinates, stats, ARMS[1], weight_norms=weight_norms)\n        import numpy as np\n        upstream = [index for index, (layer, _) in enumerate(reference.coordinates) if layer < 14]\n        earlier = [index for index, position in enumerate(warm[\'capture_positions\']) if position < warm[\'pulse_positions\'][0]]\n        upstream_equal = np.array_equal(sham_arrays[\'raw_z\'][..., upstream], pulse_arrays[\'raw_z\'][..., upstream])\n        earlier_equal = np.array_equal(sham_arrays[\'raw_z\'][:, earlier], pulse_arrays[\'raw_z\'][:, earlier])\n        metrics[\'checks\'][\'causality\'] = {\'upstream_coordinates\': len(upstream), \'upstream_raw_equal\': upstream_equal,\n                                           \'earlier_positions\': len(earlier), \'earlier_raw_equal\': earlier_equal}\n        if not upstream_equal or not earlier_equal:\n            raise ValueError(\'Technical causal guard failed; exploratory queue not started\')\n        metrics[\'checks\'][\'pulse_and_capture\'] = {\'pulse_once\': pulse_result[\'pulse_fired\'] == 1,\n                                                  \'capture_once\': pulse_result[\'capture_fired\'] == 1,\n                                                  \'frozen_parameter_evidence\': parameter_evidence(model) == evidence}\n        if parameter_evidence(model) != evidence:\n            raise ValueError(\'Model parameter identity/storage/version counters changed\')\n        metrics[\'checks_executed_this_attempt\'] = True\n        technical_checks = {\'state\': \'passed\', \'plan_sha256\': plan_hash, \'checks\': metrics[\'checks\'],\n                            \'hardware\': hardware, \'executed_utc\': dt.datetime.now(dt.UTC).isoformat()}\n        measured = []\n        for index, item in enumerate(items):\n            if done[index] is not None:\n                continue\n            request = prepare_request(tokenizer, index, item)\n            status.update(stage=\'exploratory_requests\', current_request=request[\'request_id\'])\n            save()\n            print(f"Request {index + 1}/16: {request[\'request_id\']} — fünf feste Arme", flush=True)\n            t0 = time.monotonic()\n            order = list(range(len(ARMS)))\n            random.Random(SEED + index).shuffle(order)\n            request[\'execution_order\'] = [ARMS[arm_index][\'name\'] for arm_index in order]\n            results, array_rows = [None] * len(ARMS), [None] * len(ARMS)\n            cpu_rng, cuda_rng = torch.random.get_rng_state().clone(), torch.cuda.get_rng_state().clone()\n            for arm_index in order:\n                result, arrays = run_arm(model, request, reference.coordinates, stats, ARMS[arm_index], weight_norms=weight_norms)\n                results[arm_index], array_rows[arm_index] = result, arrays\n            if not torch.equal(cpu_rng, torch.random.get_rng_state()) or not torch.equal(cuda_rng, torch.cuda.get_rng_state()):\n                raise ValueError(\'Global RNG changed during fixed five-arm request\')\n            if parameter_evidence(model) != evidence:\n                raise ValueError(\'Frozen parameter evidence changed; request is not published complete\')\n            record = write_request(output, request, results, array_rows, plan_hash, hardware, technical_checks)\n            done[index] = record\n            measured.append(time.monotonic() - t0)\n            status[\'completed_requests\'] = sum(row is not None for row in done)\n            remaining = len(items) - status[\'completed_requests\']\n            eta_seconds = sum(measured) / len(measured) * remaining\n            finish = dt.datetime.now(ZoneInfo(\'Europe/Berlin\')) + dt.timedelta(seconds=eta_seconds)\n            status.update(estimated_remaining_seconds=eta_seconds, estimated_finish_berlin=finish.isoformat())\n            save()\n            print(f"Gespeichert {status[\'completed_requests\']}/16; Restschätzung {eta_seconds / 60:.1f} min; Berlin {finish:%H:%M}", flush=True)\n            if remaining and (eta_seconds > 3600 or finish.date() > dt.datetime.now(ZoneInfo(\'Europe/Berlin\')).date()):\n                status.update(state=\'paused\', stage=\'safe_request_checkpoint\',\n                              reason=\'Runtime estimate exceeds one hour or today; resume ZIP keeps paired requests together\')\n                break\n        else:\n            status.update(state=\'complete\', stage=\'complete\')\n        metrics[\'weights_unchanged_evidence\'] = \'same parameter identities, storage pointers, version counters; no training; no full tensor rehash\'\n        metrics[\'hardware_comparison_warning\'] = \'Each complete paired request stays on one GPU; resumed chunks can differ by GPU/software.\'\n    except BaseException as exc:  # noqa: BLE001 — preserve diagnostics also on interruption\n        status.update(state=\'failed\', error_type=type(exc).__name__, error=str(exc))\n        (output / \'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \' + str(exc), flush=True)\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic() - started\n        save()\n        archive(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--hn-set\', required=True)\n    parser.add_argument(\'--policy\', required=True)\n    parser.add_argument(\'--refw\', required=True)\n    parser.add_argument(\'--items\', required=True)\n    parser.add_argument(\'--out-dir\', required=True)\n    parser.add_argument(\'--cache-dir\')\n    parser.add_argument(\'--resume-zip\')\n    args = parser.parse_args()\n    result = run(args.hn_set, args.policy, args.refw, args.items, args.out_dir, args.cache_dir, args.resume_zip)\n    print(json.dumps(result), flush=True)\n    raise SystemExit(0 if result[\'state\'] in (\'complete\', \'paused\') else 1)\n', 'attention_probe/result_bundle.py': '"""Lossless state summary of a completed/diagnostic ZIP, never a resume archive.\n\nOriginal JSON/log bytes stay unchanged. Their NPZ/logit hashes still name the\nfull archive; transformed state NPZs receive different filenames and their own\nhashes. Full-vocabulary KL remains a recorded metric once logits are omitted.\nOnly the existing local ZIP is read; this module uses no model or network.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport stat\nimport tempfile\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\n\nARCHIVE_KIND = "analysis_summary_not_resume"\nPACKAGE_FILE = "COMPACT_PACKAGE.json"\nMANIFEST_FILE = "COMPACT_MANIFEST.json"\nFULL_MANIFEST_FILE = "full_archive_manifest.json"\n\n\ndef _sha(raw: bytes) -> str:\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef _file_sha(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _json(value) -> bytes:\n    return (\n        json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)\n        + "\\n"\n    ).encode()\n\n\ndef _vocab_sizes(document):\n    sizes = set()\n    if isinstance(document, dict):\n        for key, value in document.items():\n            if (\n                key == "vocab_size"\n                and isinstance(value, int)\n                and not isinstance(value, bool)\n                and value > 0\n            ):\n                sizes.add(value)\n            sizes.update(_vocab_sizes(value))\n    elif isinstance(document, list):\n        for value in document:\n            sizes.update(_vocab_sizes(value))\n    return sizes\n\n\ndef _array_metadata(array: np.ndarray) -> dict:\n    return {\n        "dtype": str(array.dtype),\n        "shape": list(array.shape),\n        "nbytes": int(array.nbytes),\n        "sha256": _sha(array.tobytes(order="C")),\n        "hash_order": "C contiguous raw bytes",\n    }\n\n\ndef _filtered_npz(raw: bytes, vocab_sizes: set[int]):\n    included, omitted, states = {}, {}, {}\n    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:\n        if len(archive.files) != len(set(archive.files)):\n            raise ValueError("Duplicate NPZ array keys")\n        for key in archive.files:\n            array = archive[key]  # Object arrays are rejected by allow_pickle=False.\n            metadata = _array_metadata(array)\n            name = key.lower()\n            named_logits = re.search(r"(?:^|[/._])(?:full_)?logits$", name) is not None\n            vocabulary_vector = (\n                "logits" in name and array.ndim == 1 and array.shape[0] in vocab_sizes\n            )\n            if named_logits or vocabulary_vector:\n                omitted[key] = metadata\n            else:\n                included[key] = metadata\n                states[key] = array\n    output = io.BytesIO()\n    np.savez_compressed(output, **states)\n    return output.getvalue(), included, omitted\n\n\ndef build_compact_result(full_zip: Path, compact_zip: Path) -> dict:\n    """Atomically create a separately hashed analysis ZIP, preserving the full ZIP.\n\n    A malformed input raises and leaves an existing compact destination intact.\n    The caller can then offer the full archive instead. All arrays retained in\n    state NPZs keep their exact dtype/shape/values; there is no FP16 conversion.\n    Excluded logits have raw-C-byte SHA256, dtype, shape and nbytes receipts.\n    """\n    full, compact = Path(full_zip), Path(compact_zip)\n    if full.resolve() == compact.resolve() or (\n        compact.exists() and os.path.samefile(full, compact)\n    ):\n        raise ValueError("Compact output must not overwrite the full result archive")\n    full_sha, full_bytes = _file_sha(full), full.stat().st_size\n    payload, transforms, vocab_sizes = {}, [], set()\n    plan_sha = plan_file_sha = None\n    with zipfile.ZipFile(full) as archive:\n        members = archive.infolist()\n        names = [member.filename for member in members]\n        if len(names) != len(set(names)):\n            raise ValueError("Duplicate ZIP members")\n        if any(\n            name in names for name in (PACKAGE_FILE, MANIFEST_FILE, FULL_MANIFEST_FILE)\n        ):\n            raise ValueError(\n                "Input is already a compact analysis summary or uses reserved names"\n            )\n        for member in members:\n            path = PurePosixPath(member.filename)\n            if (\n                path.is_absolute()\n                or member.filename.rstrip("/") != str(path)\n                or ".." in path.parts\n                or "\\\\" in member.filename\n                or member.flag_bits & 1\n                or stat.S_ISLNK(member.external_attr >> 16)\n            ):\n                raise ValueError("Unsafe, encrypted or noncanonical ZIP member")\n        if archive.testzip() is not None:\n            raise ValueError("Full ZIP CRC verification failed")\n        # Inspect metadata, preserving the exact original bytes when copied later.\n        for member in members:\n            if member.is_dir() or not member.filename.endswith(".json"):\n                continue\n            raw = archive.read(member)\n            document = json.loads(raw)\n            vocab_sizes.update(_vocab_sizes(document))\n            if member.filename in ("run_plan.json", "plan.json") and plan_sha is None:\n                canonical = json.dumps(\n                    document,\n                    sort_keys=True,\n                    separators=(",", ":"),\n                    ensure_ascii=False,\n                    allow_nan=False,\n                ).encode()\n                plan_sha, plan_file_sha = _sha(canonical), _sha(raw)\n        for member in members:\n            if member.is_dir():\n                continue\n            name, raw = member.filename, archive.read(member)\n            transformed = name\n            if name.endswith(".npz"):\n                original_sha = _sha(raw)\n                original_bytes = len(raw)\n                raw, included, excluded = _filtered_npz(raw, vocab_sizes)\n                path = PurePosixPath(name)\n                basename = (\n                    "state_arrays.npz"\n                    if path.name == "arrays.npz"\n                    else path.stem + "_state_arrays.npz"\n                )\n                transformed = str(path.with_name(basename))\n                transforms.append(\n                    {\n                        "original_path": name,\n                        "original_sha256": original_sha,\n                        "original_bytes": original_bytes,\n                        "transformed_path": transformed,\n                        "transformed_sha256": _sha(raw),\n                        "transformed_bytes": len(raw),\n                        "included_keys": list(included),\n                        "excluded_keys": list(excluded),\n                        "included_arrays": included,\n                        "excluded_arrays": excluded,\n                    }\n                )\n            elif name == "WRAPPER_MANIFEST.json":\n                transformed = FULL_MANIFEST_FILE\n            if transformed in payload:\n                raise ValueError("Transformed ZIP member collision")\n            payload[transformed] = raw\n    included_keys = [key for row in transforms for key in row["included_keys"]]\n    available = {\n        "state_geometry": bool(included_keys),\n        "jvp_error": any(\n            any(\n                word in key.lower()\n                for word in ("jvp", "fd", "transport", "jacobian", "jac_")\n            )\n            for key in included_keys\n        ),\n        "full_vocab_KL": False,\n    }\n    metadata = {\n        "schema": "compact_result_v1",\n        "archive_kind": ARCHIVE_KIND,\n        "resume_eligible": False,\n        "full_archive": {\n            "filename": full.name,\n            "sha256": full_sha,\n            "bytes": full_bytes,\n        },\n        "full_archive_payload_included": False,\n        "npz_transforms": transforms,\n        "plan_sha256": plan_sha,\n        "plan_file_sha256": plan_file_sha,\n        "plan_canonicalization": "sorted compact UTF8 JSON, ensure_ascii=False",\n        "metrics_source": "runtime_recorded_fp64",\n        "independent_recompute_available": available,\n        "availability_scope": "Only corresponding preserved measurements are available; no new measurement is implied.",\n        "original_record_hash_scope": "Unchanged JSON NPZ/logit hashes and full_archive_manifest.json refer to the full archive, not transformed state files.",\n        "not_resume_reason": "Full arrays.npz members are replaced by separately named state-only NPZs; omitted full-vocabulary values are not embedded.",\n    }\n    payload[PACKAGE_FILE] = _json(metadata)\n    manifest = {\n        name: {"sha256": _sha(raw), "bytes": len(raw)}\n        for name, raw in sorted(payload.items())\n    }\n    payload[MANIFEST_FILE] = _json(manifest)\n    compact.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(\n        dir=compact.parent, prefix="." + compact.name + ".", suffix=".tmp", delete=False\n    ) as stream:\n        temporary = Path(stream.name)\n    try:\n        with zipfile.ZipFile(\n            temporary, "w", zipfile.ZIP_DEFLATED, compresslevel=6\n        ) as archive:\n            for name, raw in sorted(payload.items()):\n                info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))\n                info.external_attr = 0o600 << 16\n                archive.writestr(\n                    info, raw, compress_type=zipfile.ZIP_DEFLATED, compresslevel=6\n                )\n        with zipfile.ZipFile(temporary) as archive:\n            if archive.testzip() is not None:\n                raise ValueError("Compact ZIP CRC verification failed")\n        os.replace(temporary, compact)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return {\n        "archive_kind": ARCHIVE_KIND,\n        "resume_eligible": False,\n        "compact_path": str(compact),\n        "compact_sha256": _file_sha(compact),\n        "compact_bytes": compact.stat().st_size,\n        "full_archive_filename": full.name,\n        "full_archive_sha256": full_sha,\n        "full_archive_bytes": full_bytes,\n        "npz_count": len(transforms),\n        "included_array_count": len(included_keys),\n        "excluded_array_count": sum(len(row["excluded_keys"]) for row in transforms),\n        "plan_sha256": plan_sha,\n        "metrics_source": "runtime_recorded_fp64",\n        "independent_recompute_available": available,\n    }\n', 'attention_probe/answer_validation.py': '"""Prompt-only greedy answer validation; the attention pulse ends after prefill.\n\nThe old teacher-forcing runner is unchanged. This module retains its fixed REF-W\nCETT definition but never supplies request[\'ids\'], input_ids or answer_ids to\ngeneration. All saved activation arrays contain prompt positions only.\n"""\nfrom __future__ import annotations\n\nimport contextlib\nimport hashlib\nimport re\nimport time\nfrom numbers import Integral\n\nimport torch\n\nfrom .interventions import AttentionPulse, CoordinateCapture, _cache_length\n\nSEED = 20261117\nMAX_NEW_TOKENS = 48\nTERMINAL_PUNCTUATION = ".!?,;:"\n\n\ndef normalize_answer(text: str) -> str:\n    """Only case, outer whitespace and one simple terminal punctuation mark."""\n    if not isinstance(text, str):\n        raise TypeError(\'Answer text must be a string\')\n    result = text.strip().casefold()\n    if result and result[-1] in TERMINAL_PUNCTUATION:\n        result = result[:-1].rstrip()\n    return result\n\n\ndef classify_answer(generated: str, expected: str, *, truncated: bool = False) -> dict:\n    """Strict surface comparison, never extract an answer from an explanation.\n\n    Numeric values are not parsed or rounded: \'15.0\', \'+15\' and \'015\' do not\n    match \'15\'. A capped output is always classified as truncated, even if its\n    current surface matches. Noncanonical is not a semantic-incorrect verdict.\n    """\n    normalized, gold = normalize_answer(generated), normalize_answer(expected)\n    if not gold:\n        raise ValueError(\'Expected answer must not be empty\')\n    surface = (\'exact\' if generated == expected else\n               \'format_equivalent\' if normalized == gold else \'noncanonical\')\n    return {"classification": \'truncated\' if truncated else surface,\n                "surface_classification": surface, "normalized_generated": normalized,\n                "normalized_expected": gold, "truncated": bool(truncated),\n                "normalization": \'casefold; outer whitespace; at most one final . ! ? , ; :\',\n                "interpretation": \'Surface comparison only; noncanonical is not a semantic-wrong label.\'}\n\n\ndef _decode(tokenizer, ids):\n    return tokenizer.decode(list(ids), skip_special_tokens=False, clean_up_tokenization_spaces=False)\n\n\ndef _eos_ids(model, vocab_size):\n    configured = getattr(getattr(model, \'generation_config\', None), \'eos_token_id\', None)\n    source = \'generation_config\'\n    if configured is None:\n        configured = getattr(model.config, \'eos_token_id\', None)\n        source = \'config\'\n    if configured is None:\n        return (), \'none\'\n    values = [configured] if isinstance(configured, Integral) else list(configured)\n    if any(isinstance(v, bool) or not isinstance(v, Integral) or not 0 <= v < vocab_size for v in values):\n        raise ValueError(\'EOS IDs must be integer vocabulary IDs\')\n    return tuple(sorted({int(v) for v in values})), source\n\n\ndef _cett_context(model, coordinates, positions, weight_norms):\n    # The source bundle installs the old runner at its root. A repository checkout\n    # can instead resolve it as the notebooks namespace; neither path changes it.\n    try:\n        from qwen3_h286_runtime import cett_tap\n    except ModuleNotFoundError as error:\n        if error.name != \'qwen3_h286_runtime\':\n            raise\n        from notebooks.qwen3_h286_runtime import cett_tap\n    return cett_tap(model, coordinates, positions, weight_norms)\n\n\ndef _sync(device):\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n\n\ndef run_generation_arm(model, tokenizer, request, coordinates, stats, arm, *,\n                       efficient_gpu=True, weight_norms=None, max_new_tokens=48,\n                       top_k=10):\n    """Return a JSON-compatible record and fixed-shape prefill numpy arrays.\n\n    arm=None disables the pulse; passive prefill capture still runs. Only that\n    prefill has hooks. Decode uses its returned cache and newly selected tokens.\n    Selected EOS is logged in steps/selected_token_ids but excluded from emitted\n    generated_token_ids and generated_text. The 48-step maximum includes EOS.\n    """\n    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, Integral) or not 1 <= max_new_tokens <= MAX_NEW_TOKENS:\n        raise ValueError(\'max_new_tokens must be an integer from 1 through 48\')\n    if isinstance(top_k, bool) or not isinstance(top_k, Integral) or not 1 <= top_k <= 10:\n        raise ValueError(\'top_k must be an integer from 1 through 10\')\n    prompt = list(request[\'prompt_ids\'])  # Sole source of generation input IDs.\n    vocab_size = int(model.config.vocab_size)\n    if not prompt or any(isinstance(v, bool) or not isinstance(v, Integral) or not 0 <= v < vocab_size for v in prompt):\n        raise ValueError(\'prompt_ids must be a nonempty list of integer vocabulary IDs\')\n    request_id = request[\'request_id\']\n    if not isinstance(request_id, str) or re.fullmatch(r\'r[0-9]+\', request_id) is None:\n        raise ValueError(\'request_id must have the original rNNN form\')\n    seed = SEED + int(request_id[1:])\n    coordinates = tuple(tuple(pair) for pair in coordinates)\n    device = next(model.parameters()).device\n    if any(parameter.is_floating_point() and (parameter.dtype != torch.float32 or parameter.device != device)\n           for parameter in model.parameters()):\n        raise ValueError(\'Generation validation requires all floating parameters in FP32 on one device\')\n    if efficient_gpu:\n        if device.type != \'cuda\':\n            raise ValueError(\'efficient_gpu=True requires a CUDA model\')\n        if getattr(model.config, \'_attn_implementation\', None) != \'sdpa\':\n            raise ValueError(\'The GPU validation requires the sdpa attention implementation\')\n        if torch.backends.cuda.matmul.allow_tf32:\n            raise ValueError(\'Disable TF32 before full-FP32 generation validation\')\n    model.eval()\n    prompt_length = len(prompt)\n    positions = list(range(max(0, prompt_length-4), prompt_length))\n    pulse_positions = [prompt_length-1]\n    mu = torch.as_tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n    sd = torch.as_tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32)\n    if mu.shape != (len(coordinates),) or sd.shape != mu.shape or not bool(torch.isfinite(mu).all() and torch.isfinite(sd).all()) or bool((sd < 0).any()):\n        raise ValueError(\'Fixed REF-W means/SDs must match the finite ordered coordinates\')\n    sd = sd.clamp_min(1e-6)\n    eos_ids, eos_source = _eos_ids(model, vocab_size)\n    ids = torch.tensor([prompt], dtype=torch.long, device=device)\n    attention = (torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION)\n                 if efficient_gpu else contextlib.nullcontext())\n    pulse = None\n    steps, emitted, selected_ids = [], [], []\n    decode_forwards = 0\n    _sync(device)\n    started = time.monotonic()\n    with torch.no_grad(), torch.autocast(device_type=device.type, enabled=False), attention:\n        with contextlib.ExitStack() as hooks:\n            capture = hooks.enter_context(CoordinateCapture(model, coordinates, positions,\n                residual_layers=tuple(l for l in (3, 7, 14, 27) if l < len(model.model.layers)),\n                expected_count=len(coordinates)))\n            contributions = hooks.enter_context(_cett_context(model, coordinates, positions, weight_norms))\n            if arm is not None:\n                pulse = hooks.enter_context(AttentionPulse(model, arm[\'block\'], pulse_positions,\n                    arm[\'beta\'], arm[\'mode\'], seed=seed))\n            output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=True)\n            cache = output.past_key_values\n            if cache is None:\n                raise RuntimeError(\'Prefill did not return a cache\')\n            # Capture arrays are copied to CPU by the existing read-only hooks.\n            raw = capture.values\n            cett = contributions[\'cett\']\n            zcett = (cett-mu)/sd\n            if not all(bool(torch.isfinite(v).all()) for v in (raw, cett, zcett, contributions[\'denominator\'])):\n                raise ValueError(\'Nonfinite prefill raw/CETT/reference-normalized capture\')\n            arrays = {"raw_z": raw.numpy(), "cett": cett.numpy(), "zcett": zcett.numpy(),\n                          "down_output_norm_raw": contributions[\'denominator\'].numpy()}\n            arrays.update({f\'residual_layer_{l}\': value.numpy() for l, value in capture.residuals.items()})\n            logits = output.logits[0, -1].float()\n        # ExitStack has removed every pulse/capture hook before any decode call.\n        _sync(device)\n        prefill_seconds = time.monotonic()-started\n        if not bool(torch.isfinite(logits).all()):\n            raise ValueError(\'Nonfinite prefill logits\')\n        first_logits = logits.detach().cpu().clone()\n        first_logits_sha = hashlib.sha256(first_logits.numpy().tobytes()).hexdigest()\n        stop_reason = \'max_new_tokens\'\n        for step in range(int(max_new_tokens)):\n            if not bool(torch.isfinite(logits).all()):\n                raise ValueError(\'Nonfinite generation logits\')\n            logprobs = logits.log_softmax(-1)\n            chosen = int(torch.argmax(logits))\n            values, indices = torch.topk(logprobs, k=min(int(top_k), vocab_size))\n            top_tokens = [{"token_id": int(t), "text": _decode(tokenizer, [int(t)]), "logprob": float(v)}\n                          for t, v in zip(indices.detach().cpu().tolist(), values.detach().cpu().tolist())]\n            is_eos = chosen in eos_ids\n            steps.append({"step": step, "phase": \'prefill\' if step == 0 else \'decode\',\n                "logit_position": prompt_length-1+step, "selected_token_id": chosen,\n                "selected_token_text": _decode(tokenizer, [chosen]), "selected_token_logprob": float(logprobs[chosen]),\n                "is_eos": is_eos, "top_tokens": top_tokens})\n            selected_ids.append(chosen)\n            if is_eos:\n                stop_reason = \'eos\'\n                break\n            emitted.append(chosen)\n            if step+1 == max_new_tokens:\n                break\n            # Only the model\'s own last selected token is passed to cached decode.\n            current = torch.tensor([[chosen]], dtype=torch.long, device=device)\n            mask = torch.ones((1, prompt_length+len(emitted)), dtype=torch.long, device=device)\n            output = model(input_ids=current, attention_mask=mask, past_key_values=cache, use_cache=True)\n            cache = output.past_key_values\n            if cache is None:\n                raise RuntimeError(\'Decode lost its cache\')\n            logits = output.logits[0, -1].float()\n            decode_forwards += 1\n        _sync(device)\n        elapsed = time.monotonic()-started\n    generated_text = _decode(tokenizer, emitted)\n    truncated = stop_reason == \'max_new_tokens\'\n    # Gold is consulted only after generation has ended, solely for diagnostics.\n    answer = request[\'answer\']\n    classification = classify_answer(generated_text, answer, truncated=truncated)\n    gold_ids = tokenizer(answer, add_special_tokens=False)[\'input_ids\']\n    if not gold_ids or not isinstance(gold_ids[0], Integral) or not 0 <= gold_ids[0] < vocab_size:\n        raise ValueError(\'Expected answer must tokenize to a valid first vocabulary ID\')\n    gold_first = int(gold_ids[0])\n    first_logprobs = first_logits.log_softmax(-1)\n    first_gold = {"token_id": gold_first, "text": _decode(tokenizer, [gold_first]),\n        "rank": int((first_logits > first_logits[gold_first]).sum())+1,\n        "rank_rule": \'one plus number of strictly higher logits; ties can share rank\',\n        "logprob": float(first_logprobs[gold_first]), "is_top1": int(torch.argmax(first_logits)) == gold_first}\n    record = {"schema": \'h286_prompt_only_generation_v1\', "request_id": request_id,\n        "arm": dict(arm) if arm is not None else None, "seed": seed, "prompt_length": prompt_length,\n        "generation_input_source": \'request.prompt_ids only; no teacher-forced answer input\',\n        "prefill_capture_positions": positions, "pulse_positions": pulse_positions if pulse is not None else [],\n        "max_new_tokens": int(max_new_tokens), "top_k": int(top_k), "eos_token_ids": list(eos_ids), "eos_source": eos_source,\n        "generated_token_ids": emitted, "generated_text": generated_text, "selected_token_ids": selected_ids,\n        "eos_convention": \'selected EOS is recorded in steps/selected_token_ids but not emitted text/IDs\',\n        "steps": steps, "num_steps": len(steps), "num_generated_tokens": len(emitted), "stop_reason": stop_reason,\n        "truncated": truncated, "classification": classification[\'classification\'], "classification_details": classification,\n        "first_token_gold": first_gold, "first_step_logits_sha256": first_logits_sha,\n        "pulse_fired": 0 if pulse is None else pulse.fired_count, "capture_fired": capture.fired_count,\n        "prefill_forward_count": 1, "decode_forward_count": decode_forwards, "final_cache_length": _cache_length(cache),\n        "requested_delta_norms": None if pulse is None else pulse.requested_delta_norms.tolist(),\n        "actual_delta_norms": None if pulse is None else pulse.actual_delta_norms.tolist(),\n        "elapsed_seconds": elapsed, "prefill_seconds": prefill_seconds, "decode_and_logging_seconds": elapsed-prefill_seconds,\n        "timing_note": \'Synchronized wall time includes capture and top-k logging; not energy telemetry.\',\n        "dtype": \'float32\', "attention": \'sdpa/EFFICIENT_ATTENTION\' if efficient_gpu else str(getattr(model.config, \'_attn_implementation\', None)),\n        "capture_scope": \'last min(4,prompt_length) PROMPT positions; no generated-token activation capture\'}\n    record[\'baseline_kind\'] = \'no_pulse_with_passive_capture\' if arm is None else \'prefill_pulse_with_passive_capture\'\n    return record, arrays\n', 'attention_probe/temporal_cache.py': '"""Native Qwen3 temporal cache operations; no training or cache reconstruction.\n\nOffsets count *consumed* continuation tokens. With a prompt of L tokens, offset\n0 captures absolute L-1; offset k captures L+k-1 and the cache has length L+k.\nA cache swap made after offset8 affects future forwards, not its existing logits.\nOnly HF DynamicCache containing full-attention DynamicLayer is supported.\n"""\nfrom __future__ import annotations\n\nimport contextlib\nimport copy\nimport hashlib\nfrom dataclasses import dataclass\nfrom numbers import Integral\n\nimport torch\n\nfrom .interventions import AttentionPulse, CoordinateCapture, _integers\n\n\ndef _cache_layers(cache):\n    from transformers.cache_utils import DynamicCache, DynamicLayer\n    if type(cache) is not DynamicCache or getattr(cache, \'offloading\', False):\n        raise ValueError(\'Expected a non-offloaded native DynamicCache\')\n    if not cache.layers or any(type(layer) is not DynamicLayer for layer in cache.layers):\n        raise ValueError(\'Only initialized full-attention DynamicLayer caches are supported\')\n    lengths = []\n    for layer in cache.layers:\n        k, v = layer.keys, layer.values\n        if (not layer.is_initialized or not isinstance(k, torch.Tensor) or\n                not isinstance(v, torch.Tensor) or k.ndim != 4 or k.shape != v.shape or\n                k.dtype != torch.float32 or v.dtype != k.dtype or k.device != v.device or\n                k.shape[0] != 1 or k.shape[-2] < 1):\n            raise ValueError(\'Cache requires batch1 FP32 K/V [B,KV-heads,absolute-position,D]\')\n        lengths.append(int(k.shape[-2]))\n    if len(set(lengths)) != 1 or int(cache.get_seq_length()) != lengths[0]:\n        raise ValueError(\'Native cache layer lengths must be identical\')\n    return cache.layers\n\n\ndef cache_length(cache):\n    return 0 if cache is None else int(_cache_layers(cache)[0].keys.shape[-2])\n\n\ndef clone_cache(cache):\n    """Deep-copy native metadata and tensors; no legacy conversion or forward.\n\n    Qwen3 can mutate a supplied cache even with use_cache=False. Every branch\n    therefore receives a fresh clone, and the donor is never fed to a model.\n    """\n    if cache is None:\n        return None\n    layers = _cache_layers(cache)\n    if any(x.requires_grad for l in layers for x in (l.keys, l.values)):\n        raise ValueError(\'Temporal inference cache must be detached\')\n    result = copy.deepcopy(cache)\n    for before, after in zip(layers, _cache_layers(result)):\n        if before is after or before.keys.data_ptr() == after.keys.data_ptr() or before.values.data_ptr() == after.values.data_ptr():\n            raise RuntimeError(\'Cache clone retained shared tensor storage\')\n    return result\n\n\ndef _sha(tensor):\n    return hashlib.sha256(tensor.detach().contiguous().cpu().numpy().tobytes()).hexdigest()\n\n\ndef cache_manifest(cache, *, hash_tensors=True):\n    """Small JSON evidence; full hashes are optional to avoid per-step transfers."""\n    layers = _cache_layers(cache)\n    records = []\n    for index, layer in enumerate(layers):\n        row = {\'layer\': index, \'type\': type(layer).__name__, \'shape\': list(layer.keys.shape),\n               \'sequence_length\': int(layer.get_seq_length()), \'dtype\': str(layer.keys.dtype),\n               \'device\': str(layer.keys.device), \'is_initialized\': bool(layer.is_initialized)}\n        if hash_tensors:\n            row.update(keys_sha256=_sha(layer.keys), values_sha256=_sha(layer.values))\n        records.append(row)\n    return {\'cache_type\': type(cache).__name__, \'sequence_length\': cache_length(cache),\n            \'layers\': records, \'hashes_included\': bool(hash_tensors),\n            \'layout\': \'[batch, native KV heads, absolute token position, head dimension]\',\n            \'key_semantics\': \'Native normalized and RoPE-rotated keys; do not rotate again.\'}\n\n\ndef swap_cache_positions(receiver, donor, *, layer_indices, absolute_positions):\n    """Replace only specified native K/V slots in a fresh receiver clone.\n\n    Both histories must contain the identical token/position sequence. This\n    function validates lengths/layout; the caller must bind the token-ID hash.\n    It does not alter already computed states or logits at the swap boundary.\n    """\n    receiver_layers, donor_layers = _cache_layers(receiver), _cache_layers(donor)\n    blocks = _integers(layer_indices, \'layer_indices\')\n    positions = _integers(absolute_positions, \'absolute_positions\')\n    if len(receiver_layers) != len(donor_layers) or cache_length(receiver) != cache_length(donor):\n        raise ValueError(\'Receiver and donor must have identical layer counts and sequence lengths\')\n    if max(blocks) >= len(receiver_layers) or max(positions) >= cache_length(receiver):\n        raise ValueError(\'Requested cache layer or absolute position is out of range\')\n    for a, b in zip(receiver_layers, donor_layers):\n        if a.keys.shape != b.keys.shape or a.keys.device != b.keys.device:\n            raise ValueError(\'Donor and receiver cache layouts/devices must match\')\n    before = cache_manifest(receiver)\n    donor_before = cache_manifest(donor)\n    result = clone_cache(receiver)\n    changes = []\n    with torch.no_grad():\n        for block in blocks:\n            rr, dd = result.layers[block], donor_layers[block]\n            for kind in (\'keys\', \'values\'):\n                dest, src = getattr(rr, kind), getattr(dd, kind)\n                old = dest[..., list(positions), :].clone()\n                replacement = src[..., list(positions), :]\n                dest[..., list(positions), :] = replacement\n                diff = replacement.double()-old.double()\n                changes.append({\'layer\': block, \'kind\': kind,\n                                \'changed_scalars\': int(torch.count_nonzero(diff)),\n                                \'delta_norm\': float(diff.norm()),\n                                \'replacement_sha256\': _sha(replacement)})\n    # Full-slot checks are intentionally strict and only used at swap boundaries.\n    selected = set(positions)\n    others = [p for p in range(cache_length(result)) if p not in selected]\n    for block, (old, new) in enumerate(zip(receiver_layers, result.layers)):\n        for kind in (\'keys\', \'values\'):\n            oldt, newt = getattr(old, kind), getattr(new, kind)\n            if block not in blocks:\n                equal = torch.equal(oldt, newt)\n            else:\n                equal = torch.equal(oldt[..., others, :], newt[..., others, :])\n                equal = equal and torch.equal(newt[..., list(positions), :], getattr(donor_layers[block], kind)[..., list(positions), :])\n            if not equal:\n                raise RuntimeError(\'Cache swap modified an unintended slot\')\n    receiver_unchanged = before == cache_manifest(receiver)\n    donor_unchanged = donor_before == cache_manifest(donor)\n    if not receiver_unchanged or not donor_unchanged:\n        raise RuntimeError(\'A donor/receiver snapshot was mutated during cache swap\')\n    return result, {\'layer_indices\': list(blocks), \'absolute_positions\': list(positions),\n                    \'receiver_before\': before, \'donor\': donor_before,\n                    \'result\': cache_manifest(result), \'changes\': changes,\n                    \'donor_unchanged\': donor_unchanged, \'receiver_unchanged\': receiver_unchanged,\n                    \'untouched_slots_exact\': True,\n                    \'timing\': \'Swap changes future forwards only; existing boundary logits/states are unchanged.\'}\n\n\nclass AbsoluteCapture(CoordinateCapture):\n    """One forward capture supporting cached chunks with absolute token indices."""\n\n    def __init__(self, model, coordinates, positions, stats, *, residual_layers=(13,14,15,17,18,19), weight_norms=None):\n        super().__init__(model, coordinates, positions, residual_layers,\n                         expected_count=len(coordinates))\n        self.absolute_positions = tuple(self.positions)\n        self.stats = stats\n        self.weight_norms = weight_norms\n        self.cett = self.denominator = None\n        self._cett_seen = set()\n        self.input_offset = None\n        self.input_tokens = None\n        self.mu = torch.as_tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n        self.sd = torch.as_tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32)\n        if (self.mu.shape != (len(coordinates),) or self.sd.shape != self.mu.shape or\n                not bool(torch.isfinite(self.mu).all() and torch.isfinite(self.sd).all()) or bool((self.sd < 0).any())):\n            raise ValueError(\'Fixed REF-W must be finite and match the ordered coordinates\')\n        self.sd = self.sd.clamp_min(1e-6)\n\n    def _model_pre(self, module, args, kwargs):\n        if self._claimed:\n            raise RuntimeError(\'AbsoluteCapture supports exactly one model forward\')\n        ids = kwargs.get(\'input_ids\', args[0] if args else None)\n        if not isinstance(ids, torch.Tensor) or ids.ndim != 2 or ids.shape[0] != 1:\n            raise ValueError(\'Absolute capture expects batch1 input_ids\')\n        offset = cache_length(kwargs.get(\'past_key_values\'))\n        tokens = int(ids.shape[1])\n        positions = [p-offset for p in self.absolute_positions]\n        if min(positions) < 0 or max(positions) >= tokens:\n            raise ValueError(\'Absolute capture position is outside this cached input chunk\')\n        expected = torch.arange(offset,offset+tokens,device=ids.device).unsqueeze(0)\n        position_ids = kwargs.get(\'position_ids\')\n        if position_ids is not None and not torch.equal(position_ids, expected):\n            raise ValueError(\'Position IDs must match contiguous absolute cache positions\')\n        mask = kwargs.get(\'attention_mask\')\n        if mask is not None and (tuple(mask.shape)!=(1,offset+tokens) or not bool(mask.bool().all())):\n            raise ValueError(\'Only unpadded contiguous causal sequences are supported\')\n        self.positions = tuple(positions)\n        self.input_offset, self.input_tokens = offset,tokens\n        self._claimed = True\n        self.full_passes += 1\n        self._window = (1,tokens)\n\n    def _install(self):\n        super()._install()\n        for block,pairs in self._by_layer.items():\n            down = self.layers[block].mlp.down_proj\n            columns = [channel for _,channel in pairs]\n            weights = (down.weight.detach().float().norm(dim=0)[columns] if self.weight_norms is None\n                       else torch.as_tensor(self.weight_norms[block],device=down.weight.device,dtype=torch.float32))\n            if tuple(weights.shape)!=(len(columns),) or not bool(torch.isfinite(weights).all()):\n                raise ValueError(\'Precomputed weight norms must follow selected coordinate order per layer\')\n\n            def tap(module,args,output,block=block,pairs=pairs,columns=columns,weights=weights):\n                if self._window is None:\n                    return\n                if block in self._cett_seen:\n                    raise RuntimeError(\'CETT layer captured twice\')\n                z = args[0][:,list(self.positions),:][:,:,columns].abs().float()\n                den = output.norm(dim=-1)[:,list(self.positions)].float()\n                contribution = z*weights/den.clamp_min(1e-9)[...,None]\n                if not bool(torch.isfinite(contribution).all() and torch.isfinite(den).all()):\n                    raise ValueError(\'Nonfinite CETT/denominator\')\n                if self.cett is None:\n                    self.cett = torch.empty((1,len(self.positions),len(self.coordinates)),dtype=torch.float32)\n                    self.denominator = torch.empty_like(self.cett)\n                order = [index for index,_ in pairs]\n                self.cett[:,:,order] = contribution.detach().cpu()\n                self.denominator[:,:,order] = den[...,None].expand_as(contribution).detach().cpu()\n                self._cett_seen.add(block)\n                return\n            self._handles.append(down.register_forward_hook(tap))\n\n    def validate_fired(self):\n        super().validate_fired()\n        if self._cett_seen != set(self._by_layer):\n            raise RuntimeError(\'CETT capture incomplete\')\n        return self.fired_count\n\n    def arrays(self):\n        self.validate_fired()\n        result={\'raw_z\':self.values,\'cett\':self.cett,\'zcett\':(self.cett-self.mu)/self.sd,\n                \'down_output_norm_raw\':self.denominator}\n        result.update({f\'residual_{block}\':value for block,value in self.residuals.items()})\n        return result\n\n\n@dataclass\nclass CacheStep:\n    cache: object\n    logits: torch.Tensor\n    arrays: dict\n    evidence: dict\n\n\ndef forward_capture(model, input_ids, coordinates, stats, *, cache=None, capture_positions=None,\n                    residual_layers=(13,14,15,17,18,19), pulse=None, residual_patch=None,\n                    weight_norms=None, capture=True):\n    """One native FP32 SDPA-MATH forward on a cloned cache, with passive capture.\n\n    ``logits`` is CPU FP32 [1,captured positions,vocab]. ``arrays`` contains CPU\n    FP32 tensors. ``pulse`` uses existing block/beta/mode/seed fields and is only\n    permitted for uncached prefill/rebuild. ``pulse[\'positions\']`` explicitly\n    fixes the original prompt position during full-prefix rebuild parity;\n    otherwise the default is the last input token. ``capture=False`` skips passive hooks.\n    ``residual_patch`` is a fixed {block, delta:[H]} added\n    at captured current-token positions; hooks end after this single forward.\n    """\n    if model.training or getattr(model.config,\'model_type\',None)!=\'qwen3\':\n        raise ValueError(\'Temporal capture requires model.eval() and original Qwen3 architecture\')\n    if getattr(model.config,\'_attn_implementation\',None)!=\'sdpa\':\n        raise ValueError(\'Temporal capture requires native SDPA with forced MATH backend\')\n    device = next(model.parameters()).device\n    if any(p.is_floating_point() and (p.dtype!=torch.float32 or p.device!=device) for p in model.parameters()):\n        raise ValueError(\'All floating parameters must be FP32 on one device\')\n    if device.type==\'cuda\' and torch.backends.cuda.matmul.allow_tf32:\n        raise ValueError(\'TF32 must be disabled\')\n    ids=torch.as_tensor(input_ids,device=device)\n    if ids.ndim==1:\n        ids=ids.unsqueeze(0)\n    if (ids.dtype!=torch.long or ids.ndim!=2 or ids.shape[0]!=1 or ids.shape[1]<1 or\n            bool((ids<0).any()) or bool((ids>=model.config.vocab_size).any())):\n        raise ValueError(\'input_ids must be nonempty batch1 integer vocabulary tokens\')\n    start=cache_length(cache); tokens=int(ids.shape[1]); end=start+tokens\n    positions=_integers(capture_positions if capture_positions is not None else [end-1],\'capture_positions\')\n    if min(positions)<start or max(positions)>=end:\n        raise ValueError(\'Capture positions must fall within current chunk\')\n    if pulse is not None and cache is not None:\n        raise ValueError(\'Attention pulse is prefill-only; cached continuations must be unpulsed\')\n    pulse_positions=[]\n    if pulse is not None:\n        pulse_positions=list(_integers(pulse.get(\'positions\',[tokens-1]),\'pulse positions\'))\n        if len(pulse_positions)!=1 or pulse_positions[0]>=tokens:\n            raise ValueError(\'Temporal pulse requires one original prompt position within the uncached input\')\n    if cache is not None and len(cache.layers)!=len(model.model.layers):\n        raise ValueError(\'Cache layer count differs from model\')\n    branch=clone_cache(cache)\n    patch_count=0;patch_evidence=None\n    if residual_patch is not None:\n        block=residual_patch[\'block\']\n        if isinstance(block,bool) or not isinstance(block,Integral) or not 0<=block<len(model.model.layers):\n            raise ValueError(\'Residual patch block outside model\')\n        delta=torch.as_tensor(residual_patch[\'delta\'],device=device,dtype=torch.float32).detach()\n        if tuple(delta.shape)!=(model.config.hidden_size,) or not bool(torch.isfinite(delta).all()):\n            raise ValueError(\'Residual delta must be one finite fixed hidden-width vector\')\n        local=[p-start for p in positions]\n\n        def patch(module,args,output):\n            nonlocal patch_count,patch_evidence\n            if patch_count:\n                raise RuntimeError(\'Residual patch fired twice\')\n            value=output[0] if isinstance(output,(tuple,list)) else output\n            result=value.clone(); old=value[:,local,:]\n            result[:,local,:]=old+delta\n            actual=result[:,local,:].double()-old.double()\n            patch_count+=1\n            patch_evidence={\'block\':int(block),\'absolute_positions\':list(positions),\'requested_delta_norm\':float(delta.double().norm()),\n                            \'actual_delta_norms\':actual.norm(dim=-1).cpu().tolist(),\n                            \'rounding_error_norm\':float((actual-delta.double()).norm()),\'fired_count\':patch_count}\n            if isinstance(output,tuple):\n                return (result,*output[1:])\n            if isinstance(output,list):\n                return [result,*output[1:]]\n            return result\n    pulse_hook=None\n    with torch.no_grad(),torch.autocast(device_type=device.type,enabled=False),torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.MATH):\n        with contextlib.ExitStack() as stack:\n            if residual_patch is not None:\n                handle=model.model.layers[block].register_forward_hook(patch)\n                stack.callback(handle.remove)  # Installed first: residual capture observes the patched output.\n            capture_hook=(stack.enter_context(AbsoluteCapture(model,coordinates,positions,stats,\n                residual_layers=residual_layers,weight_norms=weight_norms)) if capture else None)\n            if pulse is not None:\n                pulse_hook=stack.enter_context(AttentionPulse(model,pulse[\'block\'],pulse_positions,\n                    beta=pulse.get(\'beta\',1/1.08),mode=pulse.get(\'mode\',\'scale\'),seed=pulse.get(\'seed\',0)))\n            output=model(input_ids=ids,attention_mask=torch.ones((1,end),dtype=torch.long,device=device),\n                         position_ids=torch.arange(start,end,device=device).unsqueeze(0),past_key_values=branch,use_cache=True)\n        if cache_length(output.past_key_values)!=end:\n            raise RuntimeError(\'Returned native cache has wrong consumed-token length\')\n        if residual_patch is not None and patch_count!=1:\n            raise RuntimeError(\'Residual patch did not fire exactly once\')\n        logits=output.logits[:,[p-start for p in positions],:].detach().float().cpu().clone()\n        if not bool(torch.isfinite(logits).all()):\n            raise ValueError(\'Nonfinite temporal logits\')\n    evidence={\'cache_input_length\':start,\'consumed_input_tokens\':tokens,\'cache_output_length\':end,\n              \'absolute_capture_positions\':list(positions),\'local_capture_positions\':[p-start for p in positions],\n              \'logit_predicts_absolute_positions\':[p+1 for p in positions],\n              \'incoming_cache_cloned\':cache is not None,\'capture_fired\':0 if capture_hook is None else capture_hook.fired_count,\n              \'pulse_fired\':0 if pulse_hook is None else pulse_hook.fired_count,\n              \'pulse_absolute_positions\':pulse_positions,\n              \'pulse_requested_delta_norms\':None if pulse_hook is None else pulse_hook.requested_delta_norms.tolist(),\n              \'pulse_actual_delta_norms\':None if pulse_hook is None else pulse_hook.actual_delta_norms.tolist(),\n              \'residual_patch\':patch_evidence,\'backend\':\'SDPA MATH\',\'dtype\':\'float32\',\'use_cache\':True}\n    return CacheStep(output.past_key_values,logits,{} if capture_hook is None else capture_hook.arrays(),evidence)\n', 'attention_probe/temporal_validation.py': '"""Offline checkpoint validation for the fixed temporal experiment.\n\nNo model is loaded. Stored full-vocabulary logits support independent FP64\nchecks; later teacher-forced token probabilities can only be checked for\ninternal consistency because their full logit vectors are not stored.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nfrom itertools import pairwise\n\nimport numpy as np\n\nSCHEMA = "qwen3_temporal_functional_v1"\nOFFSETS = (0, 1, 8, 32)\nLAYERS = (13, 14, 15, 17, 18, 19)\nMAIN = ("clean", "pulse", "clean_add_shared", "pulse_remove_shared")\nCACHE = (\n    "pulse_restore_prompt_cache",\n    "pulse_restore_continuation_cache",\n    "pulse_restore_both_cache",\n)\nARMS = (\n    *MAIN,\n    *[n for k in range(8) for n in (f"clean_add_noise_{k}", f"pulse_remove_noise_{k}")],\n    *CACHE,\n)\nPROOF_KEYS = (\n    "native_capture_exact",\n    "beta_one_sham_exact",\n    "cached_full_parity",\n    "null_causal_checks",\n    "source_tokens_exact",\n    "cache_donors_unchanged",\n    "frozen_parameters",\n    "global_rng",\n    "hook_cleanup",\n    "finite_arrays",\n)\nFIELDS = (\n    "raw_z",\n    "cett",\n    "zcett",\n    "down_output_norm_raw",\n    *[f"residual_{i}" for i in LAYERS],\n    "logits",\n)\n\n\ndef _require(ok, message):\n    if not ok:\n        raise ValueError(message)\n\n\ndef _canonical(value):\n    return json.dumps(\n        value,\n        sort_keys=True,\n        separators=(",", ":"),\n        ensure_ascii=False,\n        allow_nan=False,\n    ).encode()\n\n\ndef _json_sha(value):\n    return hashlib.sha256(_canonical(value)).hexdigest()\n\n\ndef _array_sha(value):\n    return hashlib.sha256(np.asarray(value, dtype="<f4").tobytes(order="C")).hexdigest()\n\n\ndef _sha(value):\n    _require(\n        isinstance(value, str)\n        and len(value) == 64\n        and all(c in "0123456789abcdef" for c in value),\n        "Expected lowercase SHA256",\n    )\n\n\ndef _finite(value, *, nonnegative=False):\n    _require(\n        type(value) in (int, float) and math.isfinite(value),\n        "Expected finite numeric scalar",\n    )\n    _require(not nonnegative or value >= 0, "Expected nonnegative scalar")\n    return float(value)\n\n\ndef _close(actual, expected, label, *, atol=1e-10, rtol=1e-8):\n    _finite(actual)\n    _require(\n        math.isclose(actual, float(expected), abs_tol=atol, rel_tol=rtol),\n        f"{label} differs from recomputation",\n    )\n\n\ndef _ids(values, vocab, *, count=None, nonempty=True):\n    _require(\n        isinstance(values, list) and (bool(values) or not nonempty),\n        "Expected token ID list",\n    )\n    _require(\n        all(type(x) is int and 0 <= x < vocab for x in values),\n        "Invalid vocabulary token ID",\n    )\n    _require(count is None or len(values) == count, "Token count differs")\n\n\ndef _cache(value, length, n_layers=None):\n    _require(\n        value["cache_type"] == "DynamicCache" and value["sequence_length"] == length,\n        "Cache type/consumed length differs",\n    )\n    _require(value["hashes_included"] is True, "Cache hashes missing")\n    rows = value["layers"]\n    _require(isinstance(rows, list) and len(rows) >= 20, "Cache lacks required layers")\n    _require(n_layers is None or len(rows) == n_layers, "Cache layer count differs")\n    layouts = set()\n    for index, row in enumerate(rows):\n        shape = row["shape"]\n        _require(\n            isinstance(shape, list)\n            and len(shape) == 4\n            and all(type(x) is int and x > 0 for x in shape),\n            "Invalid native cache shape",\n        )\n        _require(\n            row["layer"] == index\n            and row["type"] == "DynamicLayer"\n            and row["sequence_length"] == length\n            and shape[0] == 1\n            and shape[2] == length\n            and row["dtype"] == "torch.float32"\n            and row["is_initialized"] is True,\n            "Invalid cache layer metadata",\n        )\n        _require(\n            isinstance(row["device"], str) and bool(row["device"]),\n            "Missing cache device",\n        )\n        layouts.add((shape[1], shape[3], row["device"]))\n        _sha(row["keys_sha256"])\n        _sha(row["values_sha256"])\n    _require(len(layouts) == 1, "Cache layer layouts differ")\n    return len(rows)\n\n\ndef _snapshot(\n    snapshot, item, neutral, offset, start, pulse, n_layers, *, patched=False\n):\n    length = len(item["prompt_ids"]) + offset\n    forward = snapshot["forward"]\n    expected = {\n        "cache_input_length": start,\n        "consumed_input_tokens": length - start,\n        "cache_output_length": length,\n        "absolute_capture_positions": [length - 1],\n        "local_capture_positions": [length - start - 1],\n        "logit_predicts_absolute_positions": [length],\n        "capture_fired": 1,\n        "pulse_fired": int(pulse),\n        "pulse_absolute_positions": [len(item["prompt_ids"]) - 1] if pulse else [],\n        "backend": "SDPA MATH",\n        "dtype": "float32",\n        "use_cache": True,\n        "incoming_cache_cloned": start != 0,\n    }\n    for key, value in expected.items():\n        _require(\n            forward[key] == value\n            and (type(forward[key]) is bool if type(value) is bool else True),\n            f"Forward {key} differs",\n        )\n    prefix = item["prompt_ids"] + neutral[:offset]\n    _require(\n        snapshot["token_prefix_sha256"] == _json_sha(prefix),\n        "Token prefix hash differs",\n    )\n    _require(\n        snapshot["position_ids_sha256"] == _json_sha(list(range(length))),\n        "Position hash differs",\n    )\n    _sha(snapshot["logits_sha256"])\n    _cache(snapshot["cache"], length, n_layers)\n    patch = forward["residual_patch"]\n    if patched:\n        _require(\n            patch["block"] == 18\n            and patch["absolute_positions"] == [length - 1]\n            and patch["fired_count"] == 1,\n            "Residual patch location/count differs",\n        )\n        _finite(patch["requested_delta_norm"], nonnegative=True)\n        _finite(patch["rounding_error_norm"], nonnegative=True)\n        norms = np.asarray(patch["actual_delta_norms"], dtype=np.float64)\n        _require(\n            norms.shape == (1, 1) and np.isfinite(norms).all() and (norms >= 0).all(),\n            "Invalid actual patch norms",\n        )\n    else:\n        _require(patch is None, "Unexpected residual patch")\n    for key in ("pulse_requested_delta_norms", "pulse_actual_delta_norms"):\n        if pulse:\n            norms = np.asarray(forward[key], dtype=np.float64)\n            _require(\n                norms.ndim == 3\n                and norms.shape[:2] == (1, 1)\n                and norms.shape[2] > 0\n                and np.isfinite(norms).all()\n                and (norms >= 0).all(),\n                "Invalid pulse norms",\n            )\n        else:\n            _require(forward[key] is None, "Unpulsed forward has pulse norms")\n\n\ndef _scores(scores, item, snapshot, vocab):\n    end = snapshot["cache"]["sequence_length"]\n    for kind in ("gold", "foil"):\n        row = scores[kind]\n        ids = row["token_ids"]\n        _ids(ids, vocab)\n        _require(\n            ids == item[f"{kind}_token_ids"], "Scored answer tokens differ from source"\n        )\n        values = row["token_logprobs"]\n        _require(\n            isinstance(values, list) and len(values) == len(ids),\n            "Token logprob count differs",\n        )\n        for value in values:\n            _finite(value)\n            _require(value <= 1e-10, "Positive token logprob")\n        _close(row["first_logprob"], values[0], "First token score")\n        _close(row["sum_logprob"], math.fsum(values), "Sum score")\n        _close(row["mean_logprob"], math.fsum(values) / len(values), "Mean score")\n        _require(\n            row["token_count"] == len(ids)\n            and row["metric_dtype"] == "float64"\n            and row["eos_added"] is False\n            and row["endpoint_cache_unchanged"] is True,\n            "Invalid candidate scoring metadata",\n        )\n        _require(\n            row["cache_input_length"] == end\n            and row["cache_output_length"] == end + len(ids) - 1,\n            "Candidate teacher-forcing cache positions differ",\n        )\n        _require(\n            row["endpoint_cache_sha256"] == _json_sha(snapshot["cache"]),\n            "Scoring donor hash differs",\n        )\n    for key, field in (\n        ("first_token_margin", "first_logprob"),\n        ("full_sum_margin", "sum_logprob"),\n        ("mean_per_token_margin", "mean_logprob"),\n    ):\n        _close(scores[key], scores["gold"][field] - scores["foil"][field], key)\n    _require(\n        scores["primary"] == "full_sum_margin" and scores["metric_dtype"] == "float64",\n        "Score definition differs",\n    )\n\n\ndef _generation(row, endpoint, item, vocab):\n    from .answer_validation import classify_answer\n\n    selected, emitted, steps = (\n        row["selected_token_ids"],\n        row["generated_token_ids"],\n        row["steps"],\n    )\n    _ids(selected, vocab)\n    _ids(emitted, vocab, nonempty=False)\n    _ids(row["eos_ids"], vocab, nonempty=False)\n    _require(\n        row["max_new_tokens"] == 64 and row["top_k"] == 10 and 1 <= len(selected) <= 64,\n        "Greedy decoding budget differs",\n    )\n    _require(\n        len(steps) == len(selected) == row["num_steps"]\n        and len(emitted) == row["num_generated_tokens"],\n        "Greedy token counts differ",\n    )\n    for index, step in enumerate(steps):\n        is_eos = selected[index] in row["eos_ids"]\n        _require(\n            step["step"] == index\n            and step["selected_token_id"] == selected[index]\n            and step["is_eos"] is is_eos,\n            "Greedy step IDs/EOS differ",\n        )\n        _require(\n            not is_eos or index == len(steps) - 1, "Generation continued after EOS"\n        )\n        _require(\n            step["phase"] == ("endpoint" if index == 0 else "unhooked_cached_decode"),\n            "Unexpected greedy decode phase",\n        )\n        _finite(step["selected_token_logprob"])\n        _top_tokens(step["top_tokens"], vocab)\n        matching = [x for x in step["top_tokens"] if x["token_id"] == selected[index]]\n        # torch.topk may omit argmax\'s lowest-index token when >k logits tie.\n        if matching:\n            _close(\n                step["selected_token_logprob"],\n                matching[0]["logprob"],\n                "Selected top-k score",\n            )\n        _close(\n            step["selected_token_logprob"],\n            step["top_tokens"][0]["logprob"],\n            "Greedy maximal score",\n        )\n    stopped = selected[-1] in row["eos_ids"]\n    _require(\n        row["stop_reason"] == ("eos" if stopped else "max_new_tokens")\n        and row["truncated"] is (not stopped),\n        "Greedy stopping/truncation differs",\n    )\n    _require(stopped or len(selected) == 64, "Non-EOS generation stopped before cap")\n    _require(\n        emitted == (selected[:-1] if stopped else selected),\n        "Emitted tokens include EOS or omit tokens",\n    )\n    _require(\n        row["endpoint_cache_unchanged"] is True\n        and row["endpoint_cache_sha256"] == _json_sha(endpoint["snapshot"]["cache"]),\n        "Greedy donor differs",\n    )\n    _require(\n        row["final_cache_length"]\n        == endpoint["snapshot"]["cache"]["sequence_length"] + len(selected) - 1,\n        "Greedy terminal-token cache convention differs",\n    )\n    _require(isinstance(row["generated_text"], str), "Generated text missing")\n    _require(\n        row["classification"]\n        == classify_answer(\n            row["generated_text"], item["gold_text"], truncated=not stopped\n        ),\n        "Strict surface classification differs",\n    )\n    _finite(row["elapsed_seconds"], nonnegative=True)\n\n\ndef _top_tokens(rows, vocab):\n    _require(\n        isinstance(rows, list) and len(rows) == min(10, vocab), "Top-k count differs"\n    )\n    ids = [row["token_id"] for row in rows]\n    _ids(ids, vocab)\n    _require(len(set(ids)) == len(ids), "Duplicate top-k token IDs")\n    values = [_finite(row["logprob"]) for row in rows]\n    _require(\n        all(x <= 1e-10 for x in values) and all(a >= b for a, b in pairwise(values)),\n        "Invalid top-k logprobs/order",\n    )\n\n\ndef _candidate_binding(record, candidate):\n    binding = record["candidate_binding"]\n    _sha(record["candidate_sha256"])\n    _require(\n        type(record["candidate_valid"]) is bool\n        and binding["valid"] is record["candidate_valid"]\n        and binding["candidate_sha256"] == record["candidate_sha256"],\n        "Candidate identity/gate differs",\n    )\n    _sha(binding["vector_sha256"])\n    _require(\n        len(binding["noise_vector_sha256"]) == 8, "Eight frozen noise vectors required"\n    )\n    for value in binding["noise_vector_sha256"]:\n        _sha(value)\n    _finite(binding["native_norm"], nonnegative=True)\n    if candidate is not None:\n        from .temporal_candidate import verify_candidate\n\n        verify_candidate(candidate)\n        expected = {\n            "candidate_sha256": candidate["candidate_sha256"],\n            "valid": candidate["valid"],\n            "vector_sha256": _array_sha(candidate["vector"]),\n            "noise_vector_sha256": [_array_sha(v) for v in candidate["noise_vectors"]],\n        }\n        for key, value in expected.items():\n            _require(binding[key] == value, f"Frozen candidate {key} differs")\n        vector = np.asarray(candidate["vector"], dtype=np.float32)\n        _require(\n            vector.shape == (record["array_contract"]["hidden_size"],),\n            "Candidate hidden width differs",\n        )\n        _close(\n            binding["native_norm"],\n            np.linalg.norm(vector.astype(np.float64)),\n            "Candidate native norm",\n        )\n\n\ndef _parity_metric(row, shape, left=None, right=None):\n    _require(\n        row["cached_shape"] == row["full_shape"] == list(shape)\n        and row["cached_dtype"] == row["full_dtype"] == "torch.float32",\n        "Parity tensor shape/dtype differs",\n    )\n    for key in ("same_shape", "native_fp32", "finite", "passed"):\n        _require(row[key] is True, f"Parity tensor proof {key} failed")\n    _require(\n        row["rtol"] == 1e-3\n        and row["atol"] == 1e-4\n        and type(row["bitwise_equal"]) is bool,\n        "Parity tolerance/equality metadata differs",\n    )\n    for key in ("max_abs_error", "L2_error", "full_L2_norm"):\n        _finite(row[key], nonnegative=True)\n    _require(\n        not row["bitwise_equal"] or row["max_abs_error"] == row["L2_error"] == 0,\n        "Bitwise parity contradicts nonzero error",\n    )\n    if left is not None:\n        a, b = left.astype(np.float64), right.astype(np.float64)\n        _require(\n            np.allclose(a, b, rtol=1e-3, atol=1e-4),\n            "Saved capture array fails rebuild parity",\n        )\n        _require(\n            row["bitwise_equal"] is bool(np.array_equal(left, right)),\n            "Saved parity equality differs",\n        )\n        _close(row["max_abs_error"], np.max(np.abs(a - b)), "Saved parity max error")\n        _close(row["L2_error"], np.linalg.norm(a - b), "Saved parity L2 error")\n        _close(row["full_L2_norm"], np.linalg.norm(b), "Saved parity reference norm")\n\n\ndef _null_checks(baseline, L, n_layers):\n    rows = baseline["null_causal_checks"]\n    _require(\n        [r["offset"] for r in rows] == list(OFFSETS), "Causal null offsets incomplete"\n    )\n    for row in rows:\n        offset = row["offset"]\n        _require(\n            row["cache_length"] == L + offset and row["passed"] is True,\n            "Causal null length/result differs",\n        )\n        regions = {\n            "earlier_prompt_all_layers": (list(range(n_layers)), 0, L - 1),\n            "all_positions_layers0_through14": (list(range(15)), 0, L + offset),\n            "bridge_positions_layer15": ([15], L, L + offset),\n        }\n        _require(set(row["checks"]) == set(regions), "Causal null regions incomplete")\n        for name, (layers, start, stop) in regions.items():\n            region = row["checks"][name]\n            _require(\n                region["layer_indices"] == layers\n                and region["absolute_start"] == start\n                and region["absolute_stop_exclusive"] == stop\n                and region["applicable"] is (stop > start)\n                and region["passed"] is True,\n                "Causal null selection/result differs",\n            )\n            details = region["details"]\n            _require(\n                [(d["layer"], d["kind"]) for d in details]\n                == [(i, k) for i in layers for k in ("keys", "values")],\n                "Causal null tensor diagnostics incomplete",\n            )\n            _require(\n                all(\n                    d["bitwise_equal"] is True\n                    and d["L2_difference"] == 0\n                    and type(d["changed_scalars"]) is int\n                    and d["changed_scalars"] == 0\n                    for d in details\n                ),\n                "Causal null is not exact",\n            )\n\n\ndef _swap(row, record, n_layers):\n    evidence = row["cache_swap"]\n    L = len(record["item"]["prompt_ids"])\n    name = row["name"]\n    positions = (\n        [L - 1]\n        if name == CACHE[0]\n        else list(range(L, L + 8))\n        if name == CACHE[1]\n        else list(range(L - 1, L + 8))\n    )\n    _require(\n        evidence["layer_indices"] == list(range(15, n_layers))\n        and evidence["absolute_positions"] == positions,\n        "Cache-swap selection differs",\n    )\n    for key in (\n        "donor_unchanged",\n        "receiver_unchanged",\n        "untouched_slots_exact",\n        "existing_offset8_logits_unchanged",\n    ):\n        _require(evidence[key] is True, f"Cache-swap proof {key} missing")\n    snapshots = record["baseline"]["snapshots"]\n    _require(\n        evidence["receiver_before"] == snapshots["pulse_o8"]["cache"]\n        and evidence["donor"] == snapshots["clean_o8"]["cache"],\n        "Cache-swap donors differ",\n    )\n    _require(\n        evidence["token_prefix_sha256"] == snapshots["clean_o8"]["token_prefix_sha256"]\n        and evidence["position_ids_sha256"]\n        == snapshots["clean_o8"]["position_ids_sha256"],\n        "Cache-swap prefix binding differs",\n    )\n    _cache(evidence["result"], L + 8, n_layers)\n    for index in range(15):\n        _require(\n            evidence["result"]["layers"][index]\n            == evidence["receiver_before"]["layers"][index],\n            "Cache swap changed an upstream layer",\n        )\n    if name == CACHE[2]:\n        _require(\n            evidence["both_cache_exact_clean_o8"] is True\n            and evidence["both_endpoint_exact_clean_o32"] is True\n            and evidence["result"] == snapshots["clean_o8"]["cache"]\n            and row["snapshot"]["cache"] == snapshots["clean_o32"]["cache"],\n            "Both-slot technical cut does not reproduce clean cache/endpoint",\n        )\n    changes = evidence["changes"]\n    _require(\n        [(x["layer"], x["kind"]) for x in changes]\n        == [(i, k) for i in range(15, n_layers) for k in ("keys", "values")],\n        "Cache-swap diagnostics incomplete",\n    )\n    for change in changes:\n        _require(\n            type(change["changed_scalars"]) is int and change["changed_scalars"] >= 0,\n            "Invalid change count",\n        )\n        _finite(change["delta_norm"], nonnegative=True)\n        _sha(change["replacement_sha256"])\n\n\ndef validate_record(record, plan_hash, item=None, candidate=None):\n    """Reject incomplete/foreign checkpoints; return True, never mutate input."""\n    try:\n        _sha(plan_hash)\n        _require(\n            record["schema"] == SCHEMA\n            and record["state"] == "complete"\n            and record["plan_sha256"] == plan_hash,\n            "Checkpoint schema/state/plan differs",\n        )\n        split = record["split"]\n        _require(split in ("calibration", "test"), "Unknown data split")\n        technical = record["technical_checks"]\n        _require(\n            technical["schema"] == SCHEMA\n            and technical["state"] == "passed"\n            and technical["plan_sha256"] == plan_hash,\n            "Technical proof binding differs",\n        )\n        for checks in (record["checks"], technical["checks"]):\n            _require(\n                set(checks) == set(PROOF_KEYS)\n                and all(checks[k] is True for k in PROOF_KEYS),\n                "Technical proofs incomplete or failed",\n            )\n        source = record["item"]\n        _require(\n            source["split"] == split\n            and isinstance(source["id"], str)\n            and bool(source["id"]),\n            "Source item/split differs",\n        )\n        _require(\n            isinstance(source["request_id"], str) and bool(source["request_id"]),\n            "Source request ID missing",\n        )\n        if item is not None:\n            _require(\n                _canonical(source) == _canonical(item), "Frozen source item differs"\n            )\n        contract = record["array_contract"]\n        _require(\n            set(contract) == {"coordinates", "hidden_size", "vocab_size"}\n            and all(type(x) is int and x > 0 for x in contract.values()),\n            "Invalid array contract",\n        )\n        vocab = contract["vocab_size"]\n        for key in ("prompt_ids", "gold_token_ids", "foil_token_ids"):\n            _ids(source[key], vocab)\n        neutral = record["neutral_token_ids"]\n        _ids(neutral, vocab, count=32)\n        baseline = record["baseline"]\n        _require(\n            baseline["native_capture_exact"] is True\n            and baseline["beta_one_sham_exact"] is True,\n            "Native/sham proofs failed",\n        )\n        snapshots = baseline["snapshots"]\n        _require(\n            set(snapshots)\n            == {f"{a}_o{o}" for a in ("clean", "pulse") for o in OFFSETS},\n            "Baseline snapshots incomplete",\n        )\n        L = len(source["prompt_ids"])\n        n_layers = _cache(snapshots["clean_o0"]["cache"], L)\n        _null_checks(baseline, L, n_layers)\n        for arm in ("clean", "pulse"):\n            for offset, start in ((0, 0), (1, L), (8, L + 1), (32, L + 31)):\n                _snapshot(\n                    snapshots[f"{arm}_o{offset}"],\n                    source,\n                    neutral,\n                    offset,\n                    start,\n                    arm == "pulse" and offset == 0,\n                    n_layers,\n                )\n        parity = baseline["cached_full_parity"]\n        _require(\n            [(x["arm"], x["offset"]) for x in parity]\n            == [(a, o) for a in ("clean", "pulse") for o in (8, 32)],\n            "Rebuild parity rows incomplete",\n        )\n        for row in parity:\n            _require(\n                row["passed"] is True and row["logits_allclose"] is True,\n                "Rebuild parity failed",\n            )\n            _require(\n                row["cached"] == snapshots[f"{row[\'arm\']}_o{row[\'offset\']}"],\n                "Rebuild cached reference differs",\n            )\n            _snapshot(\n                row["full"],\n                source,\n                neutral,\n                row["offset"],\n                0,\n                row["arm"] == "pulse",\n                n_layers,\n            )\n            _require(\n                row["rtol"] == 1e-3 and row["atol"] == 1e-4, "Rebuild tolerances differ"\n            )\n            _finite(row["logits_max_abs_error"], nonnegative=True)\n            _finite(row["B18_L2_error"], nonnegative=True)\n            for key in (\n                "array_keys_equal",\n                "arrays_allclose",\n                "cache_layer_count_equal",\n                "kv_allclose",\n            ):\n                _require(row[key] is True, f"Rebuild {key} failed")\n            _parity_metric(row["logits"], (vocab,))\n            _require(\n                set(row["arrays_parity"]) == set(FIELDS) - {"logits"},\n                "Rebuild capture proof inventory differs",\n            )\n            for key, metric in row["arrays_parity"].items():\n                width = (\n                    contract["hidden_size"]\n                    if key.startswith("residual_")\n                    else contract["coordinates"]\n                )\n                _parity_metric(metric, (1, 1, width))\n            _require(\n                [r["layer"] for r in row["kv_parity"]] == list(range(n_layers)),\n                "Rebuild KV proofs incomplete",\n            )\n            for index, metric in enumerate(row["kv_parity"]):\n                for kind in ("keys", "values"):\n                    _parity_metric(\n                        metric[kind], row["cached"]["cache"]["layers"][index]["shape"]\n                    )\n        if split == "calibration":\n            _require(\n                not any(\n                    k in record\n                    for k in (\n                        "endpoints",\n                        "generations",\n                        "candidate_sha256",\n                        "candidate_binding",\n                    )\n                ),\n                "Calibration record contains test-derived fields",\n            )\n            for offset in (8, 32):\n                _close(\n                    record["fit_record"][f"cache_error{offset}"],\n                    max(x["B18_L2_error"] for x in parity if x["offset"] == offset),\n                    "Calibration numerical floor",\n                )\n        else:\n            _candidate_binding(record, candidate)\n            endpoints = record["endpoints"]\n            _require(\n                [r["name"] for r in endpoints] == list(ARMS),\n                "Endpoint arm slots/order differ",\n            )\n            for row in endpoints:\n                name = row["name"]\n                patched = name not in ("clean", "pulse", *CACHE)\n                _require(\n                    row["candidate_sha256"] == record["candidate_sha256"],\n                    "Endpoint candidate differs",\n                )\n                if patched and not record["candidate_valid"]:\n                    _require(\n                        row["state"] == "not_run"\n                        and row["reason"]\n                        == "frozen_candidate_failed_prespecified_numerical_gate",\n                        "Invalid candidate arm must be explicitly not_run",\n                    )\n                    _require(\n                        not any(k in row for k in ("snapshot", "scores", "readout")),\n                        "Not-run arm contains measured data",\n                    )\n                    continue\n                _require(row["state"] == "complete", "Endpoint incomplete")\n                _snapshot(\n                    row["snapshot"],\n                    source,\n                    neutral,\n                    32,\n                    L + 31,\n                    False,\n                    n_layers,\n                    patched=patched,\n                )\n                _scores(row["scores"], source, row["snapshot"], vocab)\n                _top_tokens(row["readout"]["top10"], vocab)\n                if name in CACHE:\n                    _swap(row, record, n_layers)\n                else:\n                    _require(row["cache_swap"] is None, "Unexpected cache swap")\n                if patched:\n                    receiver = "clean" if name.startswith("clean_add_") else "pulse"\n                    for index in range(19):\n                        _require(\n                            row["snapshot"]["cache"]["layers"][index]\n                            == snapshots[f"{receiver}_o32"]["cache"]["layers"][index],\n                            "Post18 patch changed an upstream/current KV layer",\n                        )\n            generations = record["generations"]\n            _require(\n                [r["name"] for r in generations] == list(MAIN),\n                "Greedy arm slots/order differ",\n            )\n            by_name = {r["name"]: r for r in endpoints}\n            for row in generations:\n                if row["name"] in MAIN[2:] and not record["candidate_valid"]:\n                    _require(\n                        row["state"] == "not_run"\n                        and row["reason"] == "invalid_frozen_candidate",\n                        "Invalid candidate generated output",\n                    )\n                else:\n                    _require(row["state"] == "complete", "Greedy record incomplete")\n                    _generation(row, by_name[row["name"]], source, vocab)\n        return True\n    except (KeyError, TypeError, IndexError, OverflowError) as exc:\n        raise ValueError(f"Malformed temporal checkpoint: {exc}") from exc\n\n\ndef _lp(logits):\n    values = logits.astype(np.float64)\n    maximum = values.max()\n    return values - (maximum + np.log(np.exp(values - maximum).sum(dtype=np.float64)))\n\n\ndef _logit_checks(array, snapshot):\n    _require(_array_sha(array) == snapshot["logits_sha256"], "Saved logits SHA differs")\n\n\ndef validate_arrays(record, arrays):\n    """Check exact array inventory, native FP32 geometry and FP64 metric math."""\n    try:\n        validate_record(record, record["plan_sha256"])\n        contract = record["array_contract"]\n        H, C, V = (contract[k] for k in ("hidden_size", "coordinates", "vocab_size"))\n        prefixes = [f"{a}_o{o}" for a in ("clean", "pulse") for o in OFFSETS]\n        prefixes += [f"{a}_full_o{o}" for a in ("clean", "pulse") for o in (8, 32)]\n        extra = []\n        if record["split"] == "calibration":\n            extra = ["calibration_delta8", "calibration_delta32"]\n        else:\n            for row in record["endpoints"]:\n                if row["state"] == "complete":\n                    prefixes.append("endpoint_" + row["name"])\n                    if row["name"] not in ("clean", "pulse", *CACHE):\n                        extra.append("endpoint_" + row["name"] + "_requested_delta")\n        expected = {p + "_" + f for p in prefixes for f in FIELDS} | set(extra)\n        _require(\n            set(arrays) == expected,\n            "Saved array inventory differs from completed arm plan",\n        )\n        for key in expected:\n            a = arrays[key]\n            shape = (\n                (H,)\n                if key in extra\n                else (V,)\n                if key.endswith("_logits")\n                else (1, 1, H)\n                if "_residual_" in key\n                else (1, 1, C)\n            )\n            _require(\n                isinstance(a, np.ndarray)\n                and a.dtype == np.dtype("float32")\n                and a.shape == shape\n                and np.isfinite(a).all(),\n                f"Invalid dtype/shape/finite array: {key}",\n            )\n            if key.endswith(("_cett", "_down_output_norm_raw")):\n                _require((a >= 0).all(), f"Negative unsigned measurement: {key}")\n        snapshots = record["baseline"]["snapshots"]\n        for prefix, snapshot in snapshots.items():\n            _logit_checks(arrays[prefix + "_logits"], snapshot)\n        for row in record["baseline"]["cached_full_parity"]:\n            p, q = (\n                f"{row[\'arm\']}_o{row[\'offset\']}",\n                f"{row[\'arm\']}_full_o{row[\'offset\']}",\n            )\n            _logit_checks(arrays[q + "_logits"], row["full"])\n            a, b = (\n                arrays[p + "_logits"].astype(np.float64),\n                arrays[q + "_logits"].astype(np.float64),\n            )\n            _require(\n                np.allclose(a, b, rtol=row["rtol"], atol=row["atol"]),\n                "Stored rebuild logits fail parity",\n            )\n            _close(\n                row["logits_max_abs_error"],\n                np.max(np.abs(a - b)),\n                "Rebuild max logit error",\n            )\n            difference = arrays[p + "_residual_18"].astype(np.float64) - arrays[\n                q + "_residual_18"\n            ].astype(np.float64)\n            _close(row["B18_L2_error"], np.linalg.norm(difference), "Rebuild B18 error")\n            _parity_metric(\n                row["logits"], (V,), arrays[p + "_logits"], arrays[q + "_logits"]\n            )\n            for field, metric in row["arrays_parity"].items():\n                _parity_metric(\n                    metric,\n                    arrays[p + "_" + field].shape,\n                    arrays[p + "_" + field],\n                    arrays[q + "_" + field],\n                )\n        if record["split"] == "calibration":\n            for offset in (8, 32):\n                expected_delta = (\n                    arrays[f"pulse_o{offset}_residual_18"][0, 0]\n                    - arrays[f"clean_o{offset}_residual_18"][0, 0]\n                )\n                _require(\n                    np.array_equal(\n                        arrays[f"calibration_delta{offset}"], expected_delta\n                    ),\n                    "Calibration delta is not native pulse-minus-clean",\n                )\n            return True\n        refs = {a: _lp(arrays[f"{a}_o32_logits"]) for a in ("clean", "pulse")}\n        binding = record["candidate_binding"]\n        for row in record["endpoints"]:\n            if row["state"] != "complete":\n                continue\n            name, readout = row["name"], row["readout"]\n            prefix = "endpoint_" + name\n            logits = arrays[prefix + "_logits"]\n            lp = _lp(logits)\n            _logit_checks(logits, row["snapshot"])\n            _require(\n                readout["logits_sha256"] == _array_sha(logits)\n                and readout["metric_dtype"] == "float64",\n                "Readout logit binding/dtype differs",\n            )\n            _require(\n                readout["argmax_id"] == int(np.argmax(logits)), "Argmax token differs"\n            )\n            for entry in readout["top10"]:\n                _close(\n                    entry["logprob"],\n                    lp[entry["token_id"]],\n                    "Full-vocabulary top-k score",\n                )\n            threshold = np.partition(lp, -min(10, V))[-min(10, V)]\n            _require(\n                all(lp[x["token_id"]] >= threshold for x in readout["top10"]),\n                "Top-k IDs are not maximal",\n            )\n            for kind in ("gold", "foil"):\n                token = record["item"][f"{kind}_token_ids"][0]\n                _require(\n                    readout[f"{kind}_first_token_id"] == token,\n                    "First target token differs",\n                )\n                _close(\n                    readout[f"{kind}_first_token_logprob"],\n                    lp[token],\n                    "First-token logprob",\n                )\n                _close(\n                    row["scores"][kind]["first_logprob"],\n                    lp[token],\n                    "Scoring first-token logprob",\n                )\n            _require(\n                set(readout["references"]) == {"clean", "pulse"}, "KL references differ"\n            )\n            for ref, ref_lp in refs.items():\n                _close(\n                    readout["references"][ref]["reference_to_arm_KL"],\n                    np.sum(np.exp(ref_lp) * (ref_lp - lp), dtype=np.float64),\n                    "KL(reference||arm)",\n                )\n                _close(\n                    readout["references"][ref]["arm_to_reference_KL"],\n                    np.sum(np.exp(lp) * (lp - ref_lp), dtype=np.float64),\n                    "KL(arm||reference)",\n                )\n            if name in ("clean", "pulse"):\n                for field in FIELDS:\n                    _require(\n                        np.array_equal(\n                            arrays[prefix + "_" + field], arrays[name + "_o32_" + field]\n                        ),\n                        "Baseline endpoint copy differs",\n                    )\n            elif name == CACHE[2]:\n                for field in FIELDS:\n                    _require(\n                        np.array_equal(\n                            arrays[prefix + "_" + field], arrays["clean_o32_" + field]\n                        ),\n                        "Both-slot technical endpoint does not reproduce clean arrays",\n                    )\n            elif name not in CACHE:\n                delta = arrays[prefix + "_requested_delta"]\n                positive = delta if name.startswith("clean_add_") else -delta\n                expected_hash = (\n                    binding["vector_sha256"]\n                    if name.endswith("_shared")\n                    else binding["noise_vector_sha256"][int(name.rsplit("_", 1)[1])]\n                )\n                _require(\n                    _array_sha(positive) == expected_hash,\n                    "Applied candidate/noise vector or sign differs",\n                )\n                _close(\n                    np.linalg.norm(delta.astype(np.float64)).item(),\n                    binding["native_norm"],\n                    "Norm-matched patch",\n                    atol=1e-12,\n                    rtol=1e-6,\n                )\n                patch = row["snapshot"]["forward"]["residual_patch"]\n                _close(\n                    patch["requested_delta_norm"],\n                    np.linalg.norm(delta.astype(np.float64)),\n                    "Requested patch norm",\n                )\n                receiver = "clean" if name.startswith("clean_add_") else "pulse"\n                before = arrays[receiver + "_o32_residual_18"]\n                after = arrays[prefix + "_residual_18"]\n                _require(\n                    np.array_equal(after, before + delta),\n                    "Captured post18 state does not equal native applied patch",\n                )\n                actual = after.astype(np.float64) - before.astype(np.float64)\n                _close(\n                    patch["actual_delta_norms"][0][0],\n                    np.linalg.norm(actual),\n                    "Actual patch norm",\n                )\n                _close(\n                    patch["rounding_error_norm"],\n                    np.linalg.norm(actual - delta.astype(np.float64)),\n                    "Patch rounding error",\n                )\n                for layer in (13, 14, 15, 17):\n                    _require(\n                        np.array_equal(\n                            arrays[f"{prefix}_residual_{layer}"],\n                            arrays[f"{receiver}_o32_residual_{layer}"],\n                        ),\n                        "Post18 patch changed an upstream residual",\n                    )\n        for row in record["generations"]:\n            if row["state"] != "complete":\n                continue\n            lp = _lp(arrays[f"endpoint_{row[\'name\']}_logits"])\n            _require(\n                row["selected_token_ids"][0] == int(np.argmax(lp)),\n                "Greedy first token differs from endpoint",\n            )\n            for entry in row["steps"][0]["top_tokens"]:\n                _close(\n                    entry["logprob"],\n                    lp[entry["token_id"]],\n                    "Greedy initial top-k score",\n                )\n        return True\n    except (KeyError, TypeError, IndexError, OverflowError) as exc:\n        raise ValueError(f"Malformed temporal arrays: {exc}") from exc\n', 'qwen3_temporal_functional_runtime.py': '"""Frozen-calibration temporal candidate/cache experiment on original Qwen3.\n\nEight calibration cases precede candidate publication and every test forward.\nOnly the final neutral token32 is patched; earlier cache swaps occur after8 and\ntherefore affect later forwards. No test-dependent fitting or arm selection.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport datetime as dt\nimport hashlib\nimport json\nimport os\nimport platform\nimport shutil\nimport tempfile\nimport time\nimport traceback\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport qwen3_h286_runtime as base\n\nSCHEMA = \'qwen3_temporal_functional_v1\'\nITEMS_SHA256 = \'997f59b8ab3fae7190a609958def98227bcad2546666e1cc66bcdf89778e225e\'\nPLAN_SHA256 = \'af42159b61e639da19cdb99c140e048cd5e88ef2c82bd467b5217f88dfb40b62\'\nSEED = 20261118\nOFFSETS = (0, 1, 8, 32)\nRESIDUAL_LAYERS = (13, 14, 15, 17, 18, 19)\nGREEDY_CAP = 64\nBUDGET_SECONDS = 1200\nMAIN_ARMS = (\'clean\', \'pulse\', \'clean_add_shared\', \'pulse_remove_shared\')\nCACHE_ARMS = (\'pulse_restore_prompt_cache\', \'pulse_restore_continuation_cache\', \'pulse_restore_both_cache\')\nENDPOINT_ARMS = (*MAIN_ARMS, *[name for k in range(8) for name in (f\'clean_add_noise_{k}\', f\'pulse_remove_noise_{k}\')], *CACHE_ARMS)\nSOURCE_NAMES = (\'qwen3_h286_runtime.py\', \'attention_probe/reference.py\',\n                \'attention_probe/fixed_refw.py\', \'attention_probe/interventions.py\',\n                \'attention_probe/answer_validation.py\', \'attention_probe/temporal_cache.py\',\n                \'attention_probe/temporal_candidate.py\', \'attention_probe/temporal_validation.py\')\nPROOF_KEYS = (\'native_capture_exact\', \'beta_one_sham_exact\', \'cached_full_parity\',\n              \'source_tokens_exact\', \'cache_donors_unchanged\', \'frozen_parameters\',\n              \'global_rng\', \'hook_cleanup\', \'finite_arrays\', \'null_causal_checks\')\n\n\ndef cache_api():\n    from attention_probe import temporal_cache\n    return temporal_cache\n\n\ndef hook_count(model):\n    return sum(len(module._forward_hooks)+len(module._forward_pre_hooks) for module in model.modules())\n\n\ndef tensor_sha(value):\n    return hashlib.sha256(value.detach().float().cpu().contiguous().numpy().tobytes()).hexdigest()\n\n\ndef token_sha(values):\n    return hashlib.sha256(base.canonical(list(values))).hexdigest()\n\n\ndef manifest_sha(cache):\n    return hashlib.sha256(base.canonical(cache_api().cache_manifest(cache))).hexdigest()\n\n\ndef _last_logits(step):\n    if step.logits.ndim != 3 or tuple(step.logits.shape[:2]) != (1, 1):\n        raise ValueError(\'Endpoint must carry exactly one last-position logit vector\')\n    return step.logits[0, 0]\n\n\ndef score_sequence(engine, step, token_ids, reference, stats):\n    """Teacher-force one fixed candidate without EOS or donor-cache mutation."""\n    tc = cache_api()\n    model = engine[\'model\']\n    ids = list(token_ids)\n    if not ids or any(type(token) is not int or not 0 <= token < model.config.vocab_size for token in ids):\n        raise ValueError(\'Scoring requires fixed nonempty vocabulary token IDs\')\n    from attention_probe.answer_validation import _eos_ids\n    eos, _ = _eos_ids(model, model.config.vocab_size)\n    if any(token in eos for token in ids):\n        raise ValueError(\'Fixed gold/foil scoring candidates must exclude EOS\')\n    start = tc.cache_length(step.cache)\n    first = _last_logits(step).double().log_softmax(-1)\n    logprobs = [float(first[ids[0]])]\n    incoming_hash = manifest_sha(step.cache)\n    if len(ids) > 1:\n        positions = list(range(start, start+len(ids)-1))\n        scored = tc.forward_capture(model, ids[:-1], reference.coordinates, stats,\n            cache=step.cache, capture_positions=positions, capture=False,\n            weight_norms=engine.get(\'weight_norms\'))\n        matrix = scored.logits[0].double().log_softmax(-1)\n        if tuple(matrix.shape) != (len(ids)-1, model.config.vocab_size):\n            raise ValueError(\'Teacher-forced logits do not match candidate positions\')\n        logprobs.extend(float(matrix[index, token]) for index, token in enumerate(ids[1:]))\n        output_length = tc.cache_length(scored.cache)\n    else:\n        output_length = start\n    if incoming_hash != manifest_sha(step.cache):\n        raise ValueError(\'Teacher forcing mutated the endpoint donor cache\')\n    return {\'token_ids\': ids, \'token_logprobs\': logprobs, \'first_logprob\': logprobs[0],\n            \'sum_logprob\': sum(logprobs), \'mean_logprob\': sum(logprobs)/len(logprobs),\n            \'token_count\': len(ids), \'metric_dtype\': \'float64\', \'eos_added\': False,\n            \'cache_input_length\': start, \'cache_output_length\': output_length,\n            \'endpoint_cache_sha256\': incoming_hash, \'endpoint_cache_unchanged\': True,\n            \'scoring\': \'first token from endpoint logits; remaining tokens from a fresh clone fed candidate[:-1]\'}\n\n\ndef score_candidates(engine, step, item, reference, stats):\n    gold = score_sequence(engine, step, item[\'gold_token_ids\'], reference, stats)\n    foil = score_sequence(engine, step, item[\'foil_token_ids\'], reference, stats)\n    return {\'gold\': gold, \'foil\': foil,\n            \'first_token_margin\': gold[\'first_logprob\']-foil[\'first_logprob\'],\n            \'full_sum_margin\': gold[\'sum_logprob\']-foil[\'sum_logprob\'],\n            \'mean_per_token_margin\': gold[\'mean_logprob\']-foil[\'mean_logprob\'],\n            \'primary\': \'full_sum_margin\', \'metric_dtype\': \'float64\',\n            \'interpretation\': \'Fixed gold-versus-foil likelihood contrast; not generated semantic accuracy\'}\n\n\ndef greedy_answer(engine, step, item, reference, stats, *, max_new_tokens=GREEDY_CAP, top_k=10):\n    """Greedy continuation from a cloned endpoint, with no decode interventions."""\n    import torch\n\n    from attention_probe.answer_validation import _eos_ids, classify_answer\n    if type(max_new_tokens) is not int or not 1 <= max_new_tokens <= GREEDY_CAP:\n        raise ValueError(\'Greedy cap must be in1..64 selected tokens including EOS\')\n    tc = cache_api()\n    model, tokenizer = engine[\'model\'], engine[\'tokenizer\']\n    eos, eos_source = _eos_ids(model, model.config.vocab_size)\n    cache = tc.clone_cache(step.cache)\n    logits = _last_logits(step)\n    original_hash = manifest_sha(step.cache)\n    original_hooks = hook_count(model)\n    emitted, selected, records = [], [], []\n    start = time.monotonic()\n    for index in range(max_new_tokens):\n        lp = logits.double().log_softmax(-1)\n        token = int(torch.argmax(logits))\n        selected.append(token)\n        values, indices = lp.topk(min(top_k, lp.numel()))\n        is_eos = token in eos\n        records.append({\'step\': index, \'selected_token_id\': token,\n            \'selected_token_text\': tokenizer.decode([token], skip_special_tokens=False, clean_up_tokenization_spaces=False),\n            \'selected_token_logprob\': float(lp[token]), \'is_eos\': is_eos,\n            \'phase\': \'endpoint\' if index == 0 else \'unhooked_cached_decode\',\n            \'top_tokens\': [{\'token_id\': int(tok), \'logprob\': float(value),\n                            \'text\': tokenizer.decode([int(tok)], skip_special_tokens=False, clean_up_tokenization_spaces=False)}\n                           for tok, value in zip(indices, values, strict=True)]})\n        if is_eos:\n            break\n        emitted.append(token)\n        if index+1 < max_new_tokens:\n            next_step = tc.forward_capture(model, [token], reference.coordinates, stats,\n                cache=cache, capture=False, weight_norms=engine.get(\'weight_norms\'))\n            if next_step.evidence[\'pulse_fired\'] or next_step.evidence[\'residual_patch\'] is not None or next_step.evidence[\'capture_fired\']:\n                raise ValueError(\'Greedy cached decode installed an intervention/capture\')\n            cache, logits = next_step.cache, _last_logits(next_step)\n    stopped = bool(records[-1][\'is_eos\'])\n    text = tokenizer.decode(emitted, skip_special_tokens=False, clean_up_tokenization_spaces=False)\n    if manifest_sha(step.cache) != original_hash or hook_count(model) != original_hooks:\n        raise ValueError(\'Greedy continuation mutated donor cache or leaked hooks\')\n    return {\'state\': \'complete\', \'max_new_tokens\': max_new_tokens, \'top_k\': top_k,\n            \'generated_text\': text, \'generated_token_ids\': emitted, \'selected_token_ids\': selected,\n            \'steps\': records, \'stop_reason\': \'eos\' if stopped else \'max_new_tokens\', \'truncated\': not stopped,\n            \'classification\': classify_answer(text, item[\'gold_text\'], truncated=not stopped),\n            \'eos_ids\': list(eos), \'eos_source\': eos_source,\n            \'num_generated_tokens\': len(emitted), \'num_steps\': len(selected),\n            \'endpoint_cache_sha256\': original_hash, \'endpoint_cache_unchanged\': True,\n            \'final_cache_length\': tc.cache_length(cache),\n            \'cache_convention\': \'selected terminal token is not fed; EOS excluded from emitted IDs/text\',\n            \'elapsed_seconds\': time.monotonic()-start,\n            \'scope\': \'four prespecified main arms only; no generated-answer selectivity test versus noise\'}\n\n\ndef save_step(arrays, prefix, step):\n    arrays[prefix+\'_logits\'] = _last_logits(step).numpy()\n    arrays.update({prefix+\'_\'+key: value.numpy() for key, value in step.arrays.items()})\n\n\ndef step_record(step, token_prefix):\n    tc = cache_api()\n    return {\'forward\': step.evidence, \'token_prefix_sha256\': token_sha(token_prefix),\n            \'position_ids_sha256\': token_sha(range(len(token_prefix))),\n            \'cache\': tc.cache_manifest(step.cache), \'logits_sha256\': tensor_sha(_last_logits(step))}\n\n\ndef _parity_tensor(cached, rebuilt, *, rtol=1e-3, atol=1e-4):\n    """Measure native FP32 drift using FP64 reductions, never infer from argmax."""\n    import torch\n    same_shape = tuple(cached.shape) == tuple(rebuilt.shape)\n    same_dtype = cached.dtype == rebuilt.dtype == torch.float32\n    row = {\'cached_shape\': list(cached.shape), \'full_shape\': list(rebuilt.shape),\n           \'cached_dtype\': str(cached.dtype), \'full_dtype\': str(rebuilt.dtype),\n           \'same_shape\': same_shape, \'native_fp32\': same_dtype,\n           \'rtol\': rtol, \'atol\': atol, \'passed\': False}\n    if not same_shape:\n        return row\n    left = cached.detach().double()\n    right = rebuilt.detach().to(device=left.device, dtype=torch.float64)\n    finite = bool(torch.isfinite(left).all() and torch.isfinite(right).all())\n    row[\'finite\'] = finite\n    if finite:\n        difference = left-right\n        row.update(max_abs_error=float(difference.abs().max()) if difference.numel() else 0.,\n                   L2_error=float(difference.norm()), full_L2_norm=float(right.norm()),\n                   bitwise_equal=bool(torch.equal(cached, rebuilt.to(cached.device))),\n                   passed=bool(same_dtype and torch.allclose(left, right, rtol=rtol, atol=atol)))\n    return row\n\n\ndef _parity_snapshot(cached, rebuilt):\n    """Full readout and every native K/V layer must pass the declared tolerance."""\n    logits = _parity_tensor(_last_logits(cached), _last_logits(rebuilt))\n    same_keys = set(cached.arrays) == set(rebuilt.arrays)\n    arrays = {key: _parity_tensor(cached.arrays[key], rebuilt.arrays[key])\n              for key in sorted(set(cached.arrays) & set(rebuilt.arrays))}\n    same_layers = len(cached.cache.layers) == len(rebuilt.cache.layers)\n    cache = []\n    for layer, (left, right) in enumerate(zip(cached.cache.layers, rebuilt.cache.layers)):\n        cache.append({\'layer\': layer,\n                      \'keys\': _parity_tensor(left.keys, right.keys),\n                      \'values\': _parity_tensor(left.values, right.values)})\n    arrays_passed = bool(same_keys and arrays and all(row[\'passed\'] for row in arrays.values()))\n    kv_passed = bool(same_layers and cache and\n                     all(row[kind][\'passed\'] for row in cache for kind in (\'keys\', \'values\')))\n    return {\'logits\': logits, \'array_keys_equal\': same_keys, \'arrays_parity\': arrays,\n            \'arrays_allclose\': arrays_passed, \'cache_layer_count_equal\': same_layers,\n            \'kv_parity\': cache, \'kv_allclose\': kv_passed,\n            \'passed\': bool(logits[\'passed\'] and arrays_passed and kv_passed)}\n\n\ndef _null_causal_snapshots(clean, pulse, *, prompt_length, offset):\n    """Actual-run causal nulls; these are exact comparisons, not tolerance gates."""\n    import torch\n    if (len(clean.cache.layers) != len(pulse.cache.layers) or\n            cache_api().cache_length(clean.cache) != prompt_length+offset or\n            cache_api().cache_length(pulse.cache) != prompt_length+offset):\n        raise ValueError(\'Causal null snapshots have incompatible cache shapes or lengths\')\n    regions = {\n        \'earlier_prompt_all_layers\': (range(len(clean.cache.layers)), 0, prompt_length-1),\n        \'all_positions_layers0_through14\': (range(min(15, len(clean.cache.layers))), 0, prompt_length+offset),\n        \'bridge_positions_layer15\': ([15] if len(clean.cache.layers)>15 else [], prompt_length, prompt_length+offset),\n    }\n    checks = {}\n    for label, (layers, start, end) in regions.items():\n        layers = list(layers)\n        details = []\n        for layer in layers:\n            for kind in (\'keys\', \'values\'):\n                left = getattr(clean.cache.layers[layer], kind)[..., start:end, :]\n                right = getattr(pulse.cache.layers[layer], kind)[..., start:end, :]\n                same_shape = left.shape == right.shape\n                exact = bool(same_shape and torch.equal(left, right))\n                difference = left.double()-right.double() if same_shape else None\n                details.append({\'layer\': layer, \'kind\': kind, \'bitwise_equal\': exact,\n                                \'L2_difference\': float(difference.norm()) if difference is not None else None,\n                                \'changed_scalars\': int(torch.count_nonzero(difference)) if difference is not None else None})\n        checks[label] = {\'layer_indices\': layers, \'absolute_start\': start, \'absolute_stop_exclusive\': end,\n                         \'applicable\': bool(layers and end>start), \'details\': details,\n                         \'passed\': all(row[\'bitwise_equal\'] for row in details)}\n    return {\'offset\': offset, \'cache_length\': prompt_length+offset,\n            \'checks\': checks, \'passed\': all(row[\'passed\'] for row in checks.values()),\n            \'scope\': \'Empty position ranges are explicitly inapplicable; no signal is inferred from them.\'}\n\n\ndef run_baselines(engine, reference, stats, item, neutral_ids):\n    """Paired trajectories; keep offset31 internally solely to replay token32."""\n    import torch\n    tc = cache_api()\n    model = engine[\'model\']\n    prompt = list(item[\'prompt_ids\'])\n    neutral = list(neutral_ids)\n    if len(neutral) != 32:\n        raise ValueError(\'Exactly the same thirty-two frozen neutral tokens are required\')\n    started = time.monotonic()\n    arrays, snapshots, histories, parity = {}, {}, {}, []\n    for arm in (\'clean\', \'pulse\'):\n        pulse = None if arm == \'clean\' else {\'block\': 14, \'beta\': 1/1.08, \'mode\': \'scale\',\n                                            \'positions\': [len(prompt)-1], \'seed\': SEED}\n        first = tc.forward_capture(model, prompt, reference.coordinates, stats, pulse=pulse,\n                                    weight_norms=engine.get(\'weight_norms\'))\n        if arm == \'clean\' and engine.get(\'on_model_run\'):\n            engine[\'on_model_run\']()\n        history = {0: first}\n        for previous, offset in ((0, 1), (1, 8), (8, 31), (31, 32)):\n            history[offset] = tc.forward_capture(model, neutral[previous:offset], reference.coordinates, stats,\n                cache=history[previous].cache, capture=offset != 31, weight_norms=engine.get(\'weight_norms\'))\n        histories[arm] = history\n        for offset in OFFSETS:\n            prefix = f\'{arm}_o{offset}\'\n            save_step(arrays, prefix, history[offset])\n            snapshots[prefix] = step_record(history[offset], prompt+neutral[:offset])\n        for offset in (8, 32):\n            rebuilt = tc.forward_capture(model, prompt+neutral[:offset], reference.coordinates, stats,\n                capture_positions=[len(prompt)+offset-1], pulse=pulse,\n                weight_norms=engine.get(\'weight_norms\'))\n            save_step(arrays, f\'{arm}_full_o{offset}\', rebuilt)\n            cached = history[offset]\n            proof = _parity_snapshot(cached, rebuilt)\n            residual_error = float((cached.arrays[\'residual_18\'].double()-rebuilt.arrays[\'residual_18\'].double()).norm())\n            passed = proof[\'passed\']\n            parity.append({\'arm\': arm, \'offset\': offset, \'logits_allclose\': proof[\'logits\'][\'passed\'],\n                           \'rtol\': 1e-3, \'atol\': 1e-4, \'logits_max_abs_error\': proof[\'logits\'].get(\'max_abs_error\'),\n                           \'B18_L2_error\': residual_error,\n                           **proof,\n                           \'cached\': snapshots[f\'{arm}_o{offset}\'],\n                           \'full\': step_record(rebuilt, prompt+neutral[:offset])})\n            if not passed:\n                raise ValueError(\'Cached/full-prefix parity failed: \'+json.dumps(parity[-1], allow_nan=False))\n    null_checks = [_null_causal_snapshots(histories[\'clean\'][offset], histories[\'pulse\'][offset],\n                   prompt_length=len(prompt), offset=offset) for offset in OFFSETS]\n    if not all(row[\'passed\'] for row in null_checks):\n        raise ValueError(\'Measured clean/pulse causal cache null failed: \'+json.dumps(null_checks, allow_nan=False))\n    # Direct native/passive-capture and beta1 sham checks bind actual KV state,\n    # not just the first argmax of two instrumented forwards.\n    plain = tc.forward_capture(model, prompt, reference.coordinates, stats, capture=False)\n    sham = tc.forward_capture(model, prompt, reference.coordinates, stats, capture=False,\n        pulse={\'block\': 14, \'beta\': 1., \'mode\': \'scale\', \'positions\': [len(prompt)-1], \'seed\': SEED})\n    clean = histories[\'clean\'][0]\n    clean_cache = manifest_sha(clean.cache)\n    native_exact = bool(torch.equal(plain.logits, clean.logits) and manifest_sha(plain.cache) == clean_cache)\n    sham_exact = bool(torch.equal(sham.logits, clean.logits) and manifest_sha(sham.cache) == clean_cache)\n    if not native_exact or not sham_exact:\n        raise ValueError(\'Native/passive-capture/beta-one sham prefill cache/logit identity failed\')\n    return {\'snapshots\': snapshots, \'cached_full_parity\': parity, \'null_causal_checks\': null_checks,\n            \'native_capture_exact\': native_exact, \'beta_one_sham_exact\': sham_exact,\n            \'elapsed_seconds\': time.monotonic()-started}, arrays, histories\n\n\ndef _proof_start(model):\n    import torch\n    device = next(model.parameters()).device\n    return {\'parameters\': base.parameter_evidence(model), \'hooks\': hook_count(model),\n            \'cpu_rng\': torch.random.get_rng_state().clone(),\n            \'cuda_rng\': torch.cuda.get_rng_state(device).clone() if device.type == \'cuda\' else None}\n\n\ndef _proof_finish(model, before, baseline, arrays, donors_unchanged=True):\n    import numpy as np\n    import torch\n    device = next(model.parameters()).device\n    checks = {\'native_capture_exact\': baseline[\'native_capture_exact\'],\n              \'beta_one_sham_exact\': baseline[\'beta_one_sham_exact\'],\n              \'cached_full_parity\': all(row[\'passed\'] for row in baseline[\'cached_full_parity\']),\n              \'null_causal_checks\': all(row[\'passed\'] for row in baseline.get(\'null_causal_checks\', [])),\n              \'source_tokens_exact\': True, \'cache_donors_unchanged\': donors_unchanged,\n              \'frozen_parameters\': base.parameter_evidence(model) == before[\'parameters\'],\n              \'global_rng\': bool(torch.equal(torch.random.get_rng_state(), before[\'cpu_rng\']) and\n                                  (before[\'cuda_rng\'] is None or torch.equal(torch.cuda.get_rng_state(device), before[\'cuda_rng\']))),\n              \'hook_cleanup\': hook_count(model) == before[\'hooks\'],\n              \'finite_arrays\': all(value.dtype == np.float32 and np.isfinite(value).all() for value in arrays.values())}\n    if not all(checks.values()):\n        raise ValueError(\'Temporal technical checks failed: \'+str(checks))\n    return checks\n\n\ndef _contract(engine, reference):\n    return {\'coordinates\': len(reference.coordinates), \'hidden_size\': engine[\'model\'].config.hidden_size,\n            \'vocab_size\': engine[\'model\'].config.vocab_size}\n\n\ndef compute_calibration(engine, reference, stats, item, neutral_ids):\n    import numpy as np\n    if item[\'split\'] != \'calibration\':\n        raise ValueError(\'Calibration computation accepts designated calibration cases only\')\n    started = time.monotonic()\n    before = _proof_start(engine[\'model\'])\n    baseline, arrays, _histories = run_baselines(engine, reference, stats, item, neutral_ids)\n    fit = {}\n    for offset in (8, 32):\n        delta = arrays[f\'pulse_o{offset}_residual_18\'][0, 0]-arrays[f\'clean_o{offset}_residual_18\'][0, 0]\n        arrays[f\'calibration_delta{offset}\'] = delta.astype(np.float32)\n        fit[f\'cache_error{offset}\'] = max(row[\'B18_L2_error\'] for row in baseline[\'cached_full_parity\'] if row[\'offset\'] == offset)\n    checks = _proof_finish(engine[\'model\'], before, baseline, arrays)\n    return {\'state\': \'complete\', \'split\': \'calibration\', \'item\': item,\n            \'neutral_token_ids\': list(neutral_ids),\n            \'baseline\': baseline, \'fit_record\': fit, \'checks\': checks,\n            \'array_contract\': _contract(engine, reference),\n            \'timing\': {\'baseline_seconds\': baseline[\'elapsed_seconds\'], \'total_seconds\': time.monotonic()-started}}, arrays\n\n\ndef endpoint_readout(step, clean, pulse, item):\n    import torch\n    current = _last_logits(step).double().log_softmax(-1)\n    values, ids = current.topk(min(10, current.numel()))\n    refs = {}\n    for name, donor in ((\'clean\', clean), (\'pulse\', pulse)):\n        ref = _last_logits(donor).double().log_softmax(-1)\n        refs[name] = {\'reference_to_arm_KL\': float((ref.exp()*(ref-current)).sum()),\n                      \'arm_to_reference_KL\': float((current.exp()*(current-ref)).sum())}\n    return {\'logits_sha256\': tensor_sha(_last_logits(step)), \'metric_dtype\': \'float64\',\n            \'top10\': [{\'token_id\': int(token), \'logprob\': float(value)} for token, value in zip(ids, values, strict=True)],\n            \'references\': refs, \'gold_first_token_id\': item[\'gold_token_ids\'][0],\n            \'foil_first_token_id\': item[\'foil_token_ids\'][0],\n            \'gold_first_token_logprob\': float(current[item[\'gold_token_ids\'][0]]),\n            \'foil_first_token_logprob\': float(current[item[\'foil_token_ids\'][0]]),\n            \'argmax_id\': int(torch.argmax(current))}\n\n\ndef compute_test(engine, reference, stats, item, neutral_ids, candidate, *, stop_requested=None):\n    import torch\n\n    from attention_probe.temporal_candidate import verify_candidate\n    if item[\'split\'] != \'test\':\n        raise ValueError(\'Held-out computation accepts test cases only\')\n    verify_candidate(candidate)\n    stop = stop_requested or (lambda: False)\n    started = time.monotonic()\n    before = _proof_start(engine[\'model\'])\n    baseline, arrays, histories = run_baselines(engine, reference, stats, item, neutral_ids)\n    tc = cache_api()\n    endpoints = {\'clean\': histories[\'clean\'][32], \'pulse\': histories[\'pulse\'][32]}\n    endpoint_records, generations = [], []\n    prompt_length = len(item[\'prompt_ids\'])\n    model = engine[\'model\']\n    device = next(model.parameters()).device\n    vectors = {\'shared\': torch.tensor(candidate[\'vector\'], dtype=torch.float32, device=device)}\n    vectors.update({f\'noise_{k}\': torch.tensor(row, dtype=torch.float32, device=device)\n                    for k, row in enumerate(candidate[\'noise_vectors\'])})\n    candidate_binding = {\'candidate_sha256\': candidate[\'candidate_sha256\'], \'valid\': candidate[\'valid\'],\n        \'vector_sha256\': tensor_sha(vectors[\'shared\']),\n        \'noise_vector_sha256\': [tensor_sha(vectors[f\'noise_{k}\']) for k in range(8)],\n        \'native_norm\': float(vectors[\'shared\'].double().norm())}\n    cache_evidence = {}\n    source_prefix = list(item[\'prompt_ids\'])+list(neutral_ids[:8])\n    assert baseline[\'snapshots\'][\'clean_o8\'][\'token_prefix_sha256\'] == token_sha(source_prefix)\n    assert baseline[\'snapshots\'][\'pulse_o8\'][\'token_prefix_sha256\'] == token_sha(source_prefix)\n    for name in ENDPOINT_ARMS:\n        if stop():\n            return {\'state\': \'partial\', \'split\': \'test\', \'item\': item,\n                    \'neutral_token_ids\': list(neutral_ids), \'candidate_binding\': candidate_binding,\n                    \'candidate_sha256\': candidate[\'candidate_sha256\'], \'baseline\': baseline,\n                    \'endpoints\': endpoint_records, \'generations\': generations,\n                    \'stop_reason\': \'budget_exhausted_between_endpoint_arms\',\n                    \'array_contract\': _contract(engine, reference)}, arrays\n        if name in (\'clean\', \'pulse\'):\n            step = endpoints[name]\n        elif name in CACHE_ARMS:\n            positions = ([prompt_length-1] if name == CACHE_ARMS[0] else\n                         list(range(prompt_length, prompt_length+8)) if name == CACHE_ARMS[1] else\n                         list(range(prompt_length-1, prompt_length+8)))\n            changed, evidence = tc.swap_cache_positions(histories[\'pulse\'][8].cache, histories[\'clean\'][8].cache,\n                layer_indices=list(range(15, len(model.model.layers))), absolute_positions=positions)\n            evidence.update(token_prefix_sha256=token_sha(source_prefix),\n                            position_ids_sha256=token_sha(range(len(source_prefix))),\n                            existing_offset8_logits_unchanged=True)\n            if name == CACHE_ARMS[2]:\n                evidence[\'both_cache_exact_clean_o8\'] = manifest_sha(changed) == manifest_sha(histories[\'clean\'][8].cache)\n                if not evidence[\'both_cache_exact_clean_o8\']:\n                    raise ValueError(\'Both-slot technical cache cut did not restore the complete clean cache\')\n            # A swap changes the stored history, never the already computed o8\n            # logits. Its first possible effect is the forward for neutral token9.\n            through31 = tc.forward_capture(model, neutral_ids[8:31], reference.coordinates, stats,\n                cache=changed, capture=False, weight_norms=engine.get(\'weight_norms\'))\n            step = tc.forward_capture(model, neutral_ids[31:32], reference.coordinates, stats,\n                cache=through31.cache, weight_norms=engine.get(\'weight_norms\'))\n            if name == CACHE_ARMS[2]:\n                clean32 = histories[\'clean\'][32]\n                exact = bool(torch.equal(step.logits, clean32.logits) and\n                             manifest_sha(step.cache) == manifest_sha(clean32.cache) and\n                             set(step.arrays) == set(clean32.arrays) and\n                             all(torch.equal(step.arrays[key], clean32.arrays[key]) for key in step.arrays))\n                evidence[\'both_endpoint_exact_clean_o32\'] = exact\n                if not exact:\n                    raise ValueError(\'Both-slot technical cache cut failed to reproduce the clean offset32 endpoint\')\n            cache_evidence[name] = evidence\n            endpoints[name] = step\n        else:\n            if not candidate[\'valid\']:\n                endpoint_records.append({\'name\': name, \'state\': \'not_run\',\n                    \'reason\': \'frozen_candidate_failed_prespecified_numerical_gate\',\n                    \'candidate_sha256\': candidate[\'candidate_sha256\']})\n                continue\n            receiver = \'clean\' if name.startswith(\'clean_add_\') else \'pulse\'\n            key = name.split(\'_add_\', 1)[1] if receiver == \'clean\' else name.split(\'_remove_\', 1)[1]\n            delta = vectors[key] if receiver == \'clean\' else -vectors[key]\n            step = tc.forward_capture(model, neutral_ids[31:32], reference.coordinates, stats,\n                cache=histories[receiver][31].cache,\n                residual_patch={\'block\': 18, \'delta\': delta}, weight_norms=engine.get(\'weight_norms\'))\n            endpoints[name] = step\n            arrays[\'endpoint_\'+name+\'_requested_delta\'] = delta.detach().cpu().numpy()\n        save_step(arrays, \'endpoint_\'+name, step)\n        scored = score_candidates(engine, step, item, reference, stats)\n        endpoint_records.append({\'name\': name, \'state\': \'complete\',\n            \'candidate_sha256\': candidate[\'candidate_sha256\'],\n            \'snapshot\': step_record(step, list(item[\'prompt_ids\'])+list(neutral_ids)),\n            \'readout\': endpoint_readout(step, endpoints[\'clean\'], endpoints[\'pulse\'], item),\n            \'scores\': scored, \'cache_swap\': cache_evidence.get(name)})\n    for name in MAIN_ARMS:\n        if name not in endpoints:\n            generations.append({\'name\': name, \'state\': \'not_run\', \'reason\': \'invalid_frozen_candidate\'})\n            continue\n        if stop():\n            return {\'state\': \'partial\', \'split\': \'test\', \'item\': item,\n                    \'neutral_token_ids\': list(neutral_ids), \'candidate_binding\': candidate_binding,\n                    \'candidate_sha256\': candidate[\'candidate_sha256\'], \'baseline\': baseline,\n                    \'endpoints\': endpoint_records, \'generations\': generations,\n                    \'stop_reason\': \'budget_exhausted_between_greedy_arms\',\n                    \'array_contract\': _contract(engine, reference)}, arrays\n        generations.append({\'name\': name, **greedy_answer(engine, endpoints[name], item, reference, stats)})\n    if len(endpoint_records) != 23:\n        raise ValueError(\'All twenty-three fixed endpoint slots must be explicit\')\n    donors_ok = all(row[\'cache_swap\'][\'donor_unchanged\'] and row[\'cache_swap\'][\'receiver_unchanged\']\n                    for row in endpoint_records if row.get(\'cache_swap\'))\n    checks = _proof_finish(model, before, baseline, arrays, donors_ok)\n    return {\'state\': \'complete\', \'split\': \'test\', \'item\': item,\n            \'neutral_token_ids\': list(neutral_ids), \'candidate_binding\': candidate_binding,\n            \'candidate_sha256\': candidate[\'candidate_sha256\'], \'candidate_valid\': candidate[\'valid\'],\n            \'baseline\': baseline, \'endpoints\': endpoint_records, \'generations\': generations,\n            \'array_contract\': _contract(engine, reference), \'checks\': checks,\n            \'timing\': {\'baseline_seconds\': baseline[\'elapsed_seconds\'], \'total_seconds\': time.monotonic()-started}}, arrays\n\n\ndef _pinned_json(path, expected):\n    raw = Path(path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != expected:\n        raise ValueError(\'Exact pinned temporal input required: \'+Path(path).name)\n    return json.loads(raw)\n\n\ndef load_inputs(hn, policy, refw, items_path, plan_path):\n    from attention_probe.fixed_refw import load_fixed_refw\n    from attention_probe.reference import load_reference\n    reference = load_reference(hn, policy)\n    stats = load_fixed_refw(refw, reference)\n    document = _pinned_json(items_path, ITEMS_SHA256)\n    plan = _pinned_json(plan_path, PLAN_SHA256)\n    rows = document.get(\'items\', [])\n    neutral = document.get(\'neutral_token_ids\', [])\n    if (len(rows) != 16 or len({row[\'id\'] for row in rows}) != 16 or len(neutral) != 32\n            or any(type(token) is not int or not 0 <= token < 151936 or token in (151643, 151645) for token in neutral)):\n        raise ValueError(\'Expected sixteen source-bound cases and thirty-two fixed scaffold tokens\')\n    for row in rows:\n        for key in (\'prompt_ids\', \'gold_token_ids\', \'foil_token_ids\'):\n            ids = row.get(key, [])\n            limit = 64 if key == \'prompt_ids\' else 32\n            if (not 1 <= len(ids) <= limit or any(type(token) is not int or not 0 <= token < 151936 for token in ids)\n                    or (key != \'prompt_ids\' and any(token in (151643, 151645) for token in ids))):\n                raise ValueError(\'Source or fixed-candidate token budget/EOS rule changed\')\n        if row.get(\'prompt_ids_sha256\') != token_sha(row[\'prompt_ids\']):\n            raise ValueError(\'Original source prompt token hash mismatch\')\n        if row[\'gold_token_ids\'] == row[\'foil_token_ids\']:\n            raise ValueError(\'Gold and foil must be distinct frozen continuations\')\n    expected = {\'schema\': \'temporal_functional_plan_v1\', \'offsets\': list(OFFSETS), \'patch_offset\': 32,\n                \'pulse_block\': 14, \'target_block\': 18, \'pulse_beta\': 1/1.08, \'candidate_seed\': SEED,\n                \'noise_count\': 8, \'greedy_max_new_tokens\': GREEDY_CAP,\n                \'seconds_after_model_load\': BUDGET_SECONDS, \'cache_layers\': list(range(15, 28)),\n                \'parity_rtol\': 1e-3, \'parity_atol\': 1e-4}\n    if any(plan.get(key) != value for key, value in expected.items()):\n        raise ValueError(\'Temporal plan changes fixed experiment rules\')\n    for split, field in ((\'calibration\', \'calibration_ids\'), (\'test\', \'test_ids\')):\n        chosen = [row for row in rows if row[\'split\'] == split]\n        if (len(chosen) != 8 or [row[\'id\'] for row in chosen] != plan.get(field)\n                or {family: sum(row[\'family\'] == family for row in chosen) for family in (\'s3\', \'adet\')} != {\'s3\': 4, \'adet\': 4}):\n            raise ValueError(\'The fixed disjoint eight-case calibration/test split changed\')\n    return reference, stats, document, plan\n\n\ndef source_path(name):\n    folder = Path(__file__).parent\n    candidate = folder/name\n    return candidate if candidate.exists() else folder.parent/name\n\n\ndef plan_document(reference, stats, document, scientific_plan):\n    return {\'schema\': SCHEMA, \'helper_sha256\': base.sha_file(__file__),\n        \'source_sha256\': {name: base.sha_file(source_path(name)) for name in SOURCE_NAMES},\n        \'items_sha256\': ITEMS_SHA256, \'scientific_plan_sha256\': PLAN_SHA256,\n        \'scientific_plan\': scientific_plan, \'input_document\': document,\n        \'model\': reference.model.to_dict(), \'coordinates\': [list(value) for value in reference.coordinates],\n        \'hn_sha256\': reference.hn_sha256, \'policy_sha256\': reference.policy_sha256,\n        \'refw_sha256\': base.REFW_SHA256, \'refw_source_sha256\': stats[\'source_hn_meta_sha256\'],\n        \'calibration_ids\': scientific_plan[\'calibration_ids\'], \'test_ids\': scientific_plan[\'test_ids\'],\n        \'dtype\': \'float32\', \'tf32\': False, \'attention\': \'SDPA MATH\',\n        \'transformers\': base.TRANSFORMERS, \'accelerate\': base.ACCELERATE,\n        \'candidate_seed\': SEED, \'endpoints\': list(ENDPOINT_ARMS), \'generation_arms\': list(MAIN_ARMS),\n        \'generation_cap\': GREEDY_CAP,\n        \'scope\': \'fixed forced scaffold, calibration-only shared vector, eight exploratory held-out-from-fit contexts\'}\n\n\ndef validate_record(record, plan_hash, item=None, candidate=None):\n    from attention_probe.temporal_validation import validate_record as validate\n    return validate(record, plan_hash, item, candidate)\n\n\ndef validate_arrays(record, arrays):\n    from attention_probe.temporal_validation import validate_arrays as validate\n    return validate(record, arrays)\n\n\ndef checkpoint_path(output, item_id):\n    if not isinstance(item_id, str) or not item_id or any(c not in \'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-\' for c in item_id):\n        raise ValueError(\'Unsafe checkpoint item identifier\')\n    return Path(output)/\'requests\'/item_id\n\n\ndef write_checkpoint(output, record, arrays, plan_hash, hardware, candidate=None):\n    import numpy as np\n    destination = checkpoint_path(output, record[\'item\'][\'id\'])\n    if destination.exists():\n        raise ValueError(\'A complete case already exists; no checkpoint overwrite\')\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    staging = Path(tempfile.mkdtemp(prefix=\'.\'+destination.name+\'-\', dir=destination.parent))\n    try:\n        np.savez_compressed(staging/\'arrays.npz\', **arrays)\n        final = dict(record, schema=SCHEMA, state=\'complete\', plan_sha256=plan_hash,\n                     hardware=hardware, completed_utc=dt.datetime.now(dt.UTC).isoformat(),\n                     npz_sha256=base.sha_file(staging/\'arrays.npz\'))\n        final[\'technical_checks\'] = {\'state\': \'passed\', \'schema\': SCHEMA,\n                                    \'plan_sha256\': plan_hash, \'checks\': final[\'checks\']}\n        validate_record(final, plan_hash, record[\'item\'], candidate)\n        validate_arrays(final, arrays)\n        base.atomic_json(staging/\'result.json\', final)\n        os.rename(staging, destination)\n    finally:\n        if staging.exists():\n            shutil.rmtree(staging)\n    return final\n\n\ndef verified(output, item, plan_hash, candidate=None):\n    import numpy as np\n    folder = checkpoint_path(output, item[\'id\'])\n    if not folder.exists():\n        return None\n    record = json.loads((folder/\'result.json\').read_text())\n    validate_record(record, plan_hash, item, candidate)\n    if base.sha_file(folder/\'arrays.npz\') != record.get(\'npz_sha256\'):\n        raise ValueError(\'Case NPZ hash changed\')\n    with np.load(folder/\'arrays.npz\', allow_pickle=False) as saved:\n        validate_arrays(record, {name: saved[name] for name in saved.files})\n    return record\n\n\ndef freeze_candidate(output, calibration_items, plan_hash):\n    """Reconstruct from validated calibration bytes even when resuming.\n\n    A self-consistent candidate hash alone is insufficient. The saved artifact\n    must exactly equal this deterministic fit of the eight actual checkpoints.\n    """\n    import numpy as np\n\n    from attention_probe.temporal_candidate import fit_candidate, verify_candidate\n    if len(calibration_items) != 8 or any(item[\'split\'] != \'calibration\' for item in calibration_items):\n        raise ValueError(\'Candidate freeze requires exactly eight calibration items\')\n    calibration_records, hashes = [], {}\n    for item in calibration_items:\n        record = verified(output, item, plan_hash)\n        if record is None:\n            raise ValueError(\'Cannot fit a candidate from incomplete calibration\')\n        folder = checkpoint_path(output, item[\'id\'])\n        digest = base.sha_file(folder/\'result.json\')\n        hashes[item[\'id\']] = digest\n        with np.load(folder/\'arrays.npz\', allow_pickle=False) as saved:\n            calibration_records.append({\'id\': item[\'id\'], \'split\': \'calibration\',\n                \'delta8\': saved[\'calibration_delta8\'].tolist(), \'delta32\': saved[\'calibration_delta32\'].tolist(),\n                \'cache_error8\': record[\'fit_record\'][\'cache_error8\'],\n                \'cache_error32\': record[\'fit_record\'][\'cache_error32\'], \'checkpoint_sha256\': digest})\n    ids = [item[\'id\'] for item in calibration_items]\n    binding = {\'runtime_plan_sha256\': plan_hash, \'items_sha256\': ITEMS_SHA256,\n               \'scientific_plan_sha256\': PLAN_SHA256, \'calibration_checkpoints\': hashes}\n    fitted = fit_candidate(calibration_records, calibration_ids=ids, source_binding=binding, seed=SEED)\n    verify_candidate(fitted, expected_calibration_ids=ids, expected_source_binding=binding)\n    destination = Path(output)/\'candidate\'\n    candidate_bytes = base.canonical(fitted)+b\'\\n\'\n    freeze = {\'state\': \'frozen\', \'schema\': SCHEMA, \'plan_sha256\': plan_hash,\n              \'candidate_sha256\': fitted[\'candidate_sha256\'],\n              \'candidate_file_sha256\': hashlib.sha256(candidate_bytes).hexdigest(),\n              \'calibration_checkpoints\': hashes}\n    if destination.exists():\n        if ((destination/\'candidate.json\').read_bytes() != candidate_bytes\n                or json.loads((destination/\'freeze.json\').read_text()) != freeze):\n            raise ValueError(\'Frozen candidate differs from the actual validated calibration fit\')\n    else:\n        staging = Path(tempfile.mkdtemp(prefix=\'.candidate-\', dir=Path(output)))\n        try:\n            base.atomic_json(staging/\'candidate.json\', fitted)\n            base.atomic_json(staging/\'freeze.json\', freeze)\n            os.rename(staging, destination)\n        finally:\n            if staging.exists():\n                shutil.rmtree(staging)\n    # Readback precedes every possible test forward.\n    saved = json.loads((destination/\'candidate.json\').read_text())\n    verify_candidate(saved, expected_calibration_ids=ids, expected_source_binding=binding)\n    if base.sha_file(destination/\'candidate.json\') != freeze[\'candidate_file_sha256\']:\n        raise ValueError(\'Candidate freeze readback hash mismatch\')\n    return saved\n\n\ndef write_partial(output, record, arrays, plan_hash):\n    import numpy as np\n    parent = Path(output)/\'partial\'/record[\'item\'][\'id\']\n    parent.mkdir(parents=True, exist_ok=True)\n    destination = parent/str(time.time_ns())\n    staging = Path(tempfile.mkdtemp(prefix=\'.partial-\', dir=parent))\n    try:\n        np.savez_compressed(staging/\'arrays.npz\', **arrays)\n        base.atomic_json(staging/\'result.json\', dict(record, schema=SCHEMA, state=\'partial\',\n            plan_sha256=plan_hash, npz_sha256=base.sha_file(staging/\'arrays.npz\'), resume_eligible=False))\n        os.rename(staging, destination)\n    finally:\n        if staging.exists():\n            shutil.rmtree(staging)\n    return destination\n\n\ndef _copy_atomic(source, destination):\n    source, destination = Path(source), Path(destination)\n    if destination.exists():\n        source_files = {path.relative_to(source): path.read_bytes() for path in source.rglob(\'*\') if path.is_file()}\n        destination_files = {path.relative_to(destination): path.read_bytes() for path in destination.rglob(\'*\') if path.is_file()}\n        if source_files != destination_files:\n            raise ValueError(\'Resume would overwrite a different existing complete artifact\')\n        return\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    staging = Path(tempfile.mkdtemp(prefix=\'.\'+destination.name+\'-copy-\', dir=destination.parent))\n    try:\n        shutil.copytree(source, staging, dirs_exist_ok=True)\n        os.rename(staging, destination)\n    finally:\n        if staging.exists():\n            shutil.rmtree(staging)\n\n\ndef restore_resume(resume_zip, output, plan_hash, document, scientific_plan):\n    if not resume_zip:\n        return\n    items = {item[\'id\']: item for item in document[\'items\']}\n    calibration = [items[item_id] for item_id in scientific_plan[\'calibration_ids\']]\n    with zipfile.ZipFile(resume_zip) as bundle, tempfile.TemporaryDirectory(prefix=\'temporal-resume-\') as temporary:\n        names = bundle.namelist()\n        if (len(names) != len(set(names)) or \'COMPACT_PACKAGE.json\' in names or bundle.testzip() is not None):\n            raise ValueError(\'Only intact full temporal archives can resume, never compact analysis summaries\')\n        if hashlib.sha256(base.canonical(json.loads(bundle.read(\'run_plan.json\')))).hexdigest() != plan_hash:\n            raise ValueError(\'Resume scientific/source plan changed\')\n        allowed = {f\'requests/{item_id}/{name}\' for item_id in items for name in (\'result.json\', \'arrays.npz\')}\n        selected = [name for name in names if name.startswith(\'requests/\') and not name.endswith(\'/\')]\n        if set(selected)-allowed:\n            raise ValueError(\'Unexpected resume checkpoint path\')\n        staging = Path(temporary)\n        for item in document[\'items\']:\n            pair = [f\'requests/{item["id"]}/{name}\' for name in (\'result.json\', \'arrays.npz\')]\n            present = [name in names for name in pair]\n            if any(present) and not all(present):\n                raise ValueError(\'Incomplete pair in complete resume namespace\')\n            if all(present):\n                target = checkpoint_path(staging, item[\'id\'])\n                target.mkdir(parents=True)\n                for name in pair:\n                    (target/Path(name).name).write_bytes(bundle.read(name))\n        candidate_names = (\'candidate/candidate.json\', \'candidate/freeze.json\')\n        if any(name in names for name in candidate_names):\n            if not all(name in names for name in candidate_names):\n                raise ValueError(\'Incomplete candidate freeze pair\')\n            (staging/\'candidate\').mkdir()\n            for name in candidate_names:\n                (staging/name).write_bytes(bundle.read(name))\n        complete_cal = []\n        for item in calibration:\n            record = verified(staging, item, plan_hash)\n            if record is not None:\n                complete_cal.append(item)\n        test_present = any(checkpoint_path(staging, item_id).exists() for item_id in scientific_plan[\'test_ids\'])\n        if len(complete_cal) == 8:\n            candidate = freeze_candidate(staging, calibration, plan_hash)\n        else:\n            candidate = None\n            if (staging/\'candidate\').exists() or test_present:\n                raise ValueError(\'Candidate/test checkpoints cannot precede complete calibration\')\n        for item in document[\'items\']:\n            if checkpoint_path(staging, item[\'id\']).exists():\n                verified(staging, item, plan_hash, candidate if item[\'split\'] == \'test\' else None)\n                _copy_atomic(checkpoint_path(staging, item[\'id\']), checkpoint_path(output, item[\'id\']))\n        if candidate is not None:\n            _copy_atomic(staging/\'candidate\', Path(output)/\'candidate\')\n\n\ndef archive(output):\n    output = Path(output)\n    destination = output/\'temporal_functional_results.zip\'\n    temporary = output/\'.temporal_functional_results.zip.tmp\'\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_DEFLATED) as bundle:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file() and path not in (destination, temporary) and not any(part.startswith(\'.\') for part in path.relative_to(output).parts):\n                bundle.write(path, path.relative_to(output))\n    with zipfile.ZipFile(temporary) as bundle:\n        if bundle.testzip() is not None:\n            raise ValueError(\'Final temporal archive CRC failure\')\n    os.replace(temporary, destination)\n    return destination\n\n\ndef initialize_engine(reference, cache, on_loaded):\n    from importlib.metadata import version\n\n    import torch\n    import transformers\n    if transformers.__version__ != base.TRANSFORMERS or version(\'accelerate\') != base.ACCELERATE:\n        raise ValueError(\'Exact original Transformers5.22.2 and accelerate1.16.0 required\')\n    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False\n    torch.set_float32_matmul_precision(\'highest\')\n    torch.manual_seed(base.SEED)\n    transformers.utils.logging.disable_progress_bar()\n    hardware = base.hardware_preflight(torch, cache)\n    print(\'Stage model: Original Qwen3-1.7B FP32 laden; native SDPA MATH.\', flush=True)\n    tokenizer, model = base.load_model(reference, cache)\n    hardware.update(torch=torch.__version__, cuda=torch.version.cuda, python=platform.python_version(),\n                    transformers=transformers.__version__, dtype=\'float32\', tf32=False,\n                    attention=\'SDPA MATH\', compute_capability=list(torch.cuda.get_device_capability()))\n    on_loaded(hardware)\n    groups = defaultdict(list)\n    for block, channel in reference.coordinates:\n        groups[block].append(channel)\n    with torch.no_grad():\n        norms = {block: model.model.layers[block].mlp.down_proj.weight.norm(dim=0)[columns]\n                 for block, columns in groups.items()}\n    return {\'model\': model, \'tokenizer\': tokenizer, \'hardware\': hardware, \'weight_norms\': norms}\n\n\ndef run(hn, policy, refw, items_path, plan_path, out_dir, cache_dir=None, resume_zip=None, offline_plan_only=False):\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    nonce = os.environ.get(\'H286_WRAPPER_ATTEMPT\')\n    status = {\'state\': \'running\', \'stage\': \'reference\', \'schema\': SCHEMA,\n              \'completed_calibration\': 0, \'completed_test\': 0, \'completed_inputs\': 0,\n              \'wrapper_attempt_id\': nonce, \'confirmatory_eligible\': False}\n    metrics = {\'weights_loaded\': False, \'pretrained_model_run\': False, \'no_training\': True,\n               \'generation_count\': 0, \'wrapper_attempt_id\': nonce}\n    started = time.monotonic()\n    def save():\n        base.atomic_json(output/\'status.json\', status)\n        base.atomic_json(output/\'metrics.json\', metrics)\n    save()\n    try:\n        reference, stats, document, scientific_plan = load_inputs(hn, policy, refw, items_path, plan_path)\n        plan = plan_document(reference, stats, document, scientific_plan)\n        plan_hash = hashlib.sha256(base.canonical(plan)).hexdigest()\n        existing = output/\'run_plan.json\'\n        if existing.exists() and existing.read_bytes() != base.canonical(plan)+b\'\\n\':\n            raise ValueError(\'Output directory belongs to another exact temporal plan\')\n        if not existing.exists():\n            base.atomic_json(existing, plan)\n        if offline_plan_only:\n            status.update(state=\'complete\', stage=\'offline_plan_only\', plan_sha256=plan_hash, no_model_executed=True)\n            return status\n        restore_resume(resume_zip, output, plan_hash, document, scientific_plan)\n        by_id = {item[\'id\']: item for item in document[\'items\']}\n        calibration = [by_id[item_id] for item_id in scientific_plan[\'calibration_ids\']]\n        test = [by_id[item_id] for item_id in scientific_plan[\'test_ids\']]\n        cal_done = {item[\'id\']: verified(output, item, plan_hash) for item in calibration}\n        if (output/\'candidate\').exists() and not all(cal_done.values()):\n            raise ValueError(\'Candidate exists without eight complete valid calibration checkpoints\')\n        candidate = freeze_candidate(output, calibration, plan_hash) if all(cal_done.values()) else None\n        test_done = {item[\'id\']: (verified(output, item, plan_hash, candidate) if candidate is not None else None) for item in test}\n        if candidate is None and any(checkpoint_path(output, item[\'id\']).exists() for item in test):\n            raise ValueError(\'Test artifacts cannot exist before a validated frozen candidate\')\n        def progress():\n            status[\'completed_calibration\'] = sum(record is not None for record in cal_done.values())\n            status[\'completed_test\'] = sum(record is not None for record in test_done.values())\n            status[\'completed_inputs\'] = status[\'completed_calibration\']+status[\'completed_test\']\n            records = [record for record in test_done.values() if record is not None]\n            metrics[\'generation_count\'] = sum(row[\'state\'] == \'complete\' for record in records for row in record[\'generations\'])\n            metrics[\'endpoint_complete_count\'] = sum(row[\'state\'] == \'complete\' for record in records for row in record[\'endpoints\'])\n            metrics[\'endpoint_not_run_count\'] = sum(row[\'state\'] == \'not_run\' for record in records for row in record[\'endpoints\'])\n            metrics[\'completed_inputs\'] = status[\'completed_inputs\']\n        progress()\n        if all(cal_done.values()) and all(test_done.values()):\n            status.update(state=\'complete\', stage=\'complete\', resumed_without_gpu=True,\n                          candidate_sha256=candidate[\'candidate_sha256\'], candidate_valid=candidate[\'valid\'])\n            metrics.update(retained_pretrained_model_run=True,\n                           retained_technical_checks=[record[\'technical_checks\'] for record in [*cal_done.values(), *test_done.values()]])\n            return status\n        status[\'stage\'] = \'pinned_model_load\'\n        save()\n        def loaded(hardware):\n            metrics.update(weights_loaded=True, hardware=hardware)\n            save()\n        engine = initialize_engine(reference, Path(cache_dir or \'/content/h286_model_cache\'), loaded)\n        def model_ran():\n            if not metrics[\'pretrained_model_run\']:\n                metrics[\'pretrained_model_run\'] = True\n                save()\n        engine[\'on_model_run\'] = model_ran\n        compute_started = time.monotonic()\n        deadline = compute_started+BUDGET_SECONDS\n        stop = lambda: time.monotonic() >= deadline\n        durations = []\n        test_durations = []\n        for item in calibration:\n            if cal_done[item[\'id\']] is not None:\n                continue\n            if stop():\n                status.update(state=\'paused\', stage=\'safe_calibration_checkpoint\', reason=\'20-minute soft compute budget reached\')\n                break\n            status.update(stage=\'calibration\', current_input=item[\'id\'])\n            save()\n            print(f\'Stage calibration {status["completed_calibration"]+1}/8: {item["id"]}\', flush=True)\n            metrics[\'model_forward_attempted\'] = True\n            record, arrays = compute_calibration(engine, reference, stats, item, document[\'neutral_token_ids\'])\n            metrics[\'pretrained_model_run\'] = True\n            cal_done[item[\'id\']] = write_checkpoint(output, record, arrays, plan_hash, engine[\'hardware\'])\n            durations.append(record[\'timing\'][\'total_seconds\'])\n            progress()\n            save()\n            print(f\'Gespeichert calibration {status["completed_calibration"]}/8; letzte Rechnung {durations[-1]:.2f}s.\', flush=True)\n        if not all(cal_done.values()):\n            metrics[\'compute_elapsed_seconds\'] = time.monotonic()-compute_started\n            return status\n        # Refit uses only validated calibration bytes, and publication/readback\n        # occurs before the first possible held-out forward in this attempt.\n        candidate = freeze_candidate(output, calibration, plan_hash)\n        status.update(candidate_sha256=candidate[\'candidate_sha256\'], candidate_valid=candidate[\'valid\'])\n        save()\n        print(\'Stage candidate: eingefroren; gültig=\'+str(candidate[\'valid\'])+\'; SHA256 \'+candidate[\'candidate_sha256\'][:16], flush=True)\n        for item in test:\n            if test_done[item[\'id\']] is not None:\n                continue\n            if stop():\n                status.update(state=\'paused\', stage=\'safe_test_checkpoint\', reason=\'20-minute soft compute budget reached\')\n                break\n            status.update(stage=\'test\', current_input=item[\'id\'])\n            save()\n            print(f\'Stage test {status["completed_test"]+1}/8: {item["id"]}; eingefrorene23Endpunkte.\', flush=True)\n            case_started = time.monotonic()\n            # An exact reconstruction/readback precedes each case, including a\n            # resumed case, without inspecting any current test target states.\n            checked_candidate = freeze_candidate(output, calibration, plan_hash)\n            metrics[\'model_forward_attempted\'] = True\n            record, arrays = compute_test(engine, reference, stats, item, document[\'neutral_token_ids\'],\n                                         checked_candidate, stop_requested=stop)\n            metrics[\'pretrained_model_run\'] = True\n            if record[\'state\'] != \'complete\':\n                partial = write_partial(output, record, arrays, plan_hash)\n                status.update(state=\'paused\', stage=\'partial_test_checkpoint\',\n                              reason=record[\'stop_reason\'], partial_path=str(partial.relative_to(output)))\n                break\n            test_done[item[\'id\']] = write_checkpoint(output, record, arrays, plan_hash, engine[\'hardware\'], checked_candidate)\n            test_durations.append(time.monotonic()-case_started)\n            progress()\n            status.update(compute_elapsed_seconds=time.monotonic()-compute_started,\n                          estimated_remaining_seconds=sum(test_durations)/len(test_durations)*(8-status[\'completed_test\']),\n                          eta_scope=\'remaining test-case wall including candidate verification/checkpoint; excludes final archive/install/download\')\n            save()\n            print(f\'Gespeichert test {status["completed_test"]}/8; {metrics["generation_count"]} Antworten; \'\n                  f\'letzter Fall {test_durations[-1]:.2f}s; Rechenrestschätzung {status["estimated_remaining_seconds"]/60:.1f}min.\', flush=True)\n        else:\n            status.update(state=\'complete\', stage=\'complete\')\n        metrics[\'compute_elapsed_seconds\'] = time.monotonic()-compute_started\n    except BaseException as exc:  # noqa: BLE001 — preserve model/interrupt diagnostics.\n        status.update(state=\'failed\', error_type=type(exc).__name__, error=str(exc))\n        (output/\'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \'+str(exc), flush=True)\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic()-started\n        save()\n        archive(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in (\'hn-set\', \'policy\', \'refw\', \'items\', \'plan\', \'out-dir\'):\n        parser.add_argument(\'--\'+name, required=True)\n    parser.add_argument(\'--cache-dir\')\n    parser.add_argument(\'--resume-zip\')\n    parser.add_argument(\'--offline-plan-only\', action=\'store_true\')\n    options = parser.parse_args()\n    terminal = run(options.hn_set, options.policy, options.refw, options.items, options.plan,\n                   options.out_dir, options.cache_dir, options.resume_zip, options.offline_plan_only)\n    raise SystemExit(0 if terminal[\'state\'] in (\'complete\', \'paused\') else 1)\n', 'attention_probe/temporal_candidate.py': '"""A calibration-only, shared temporal direction; no model or test-data fitting.\n\nThe sign of the late projection is preserved. The eight random directions are\nfixed by one local PCG64 generator and are never selected by observed outcomes.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\n\nimport numpy as np\n\n\ndef _canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(",", ":"),\n                      ensure_ascii=True, allow_nan=False).encode()\n\n\ndef candidate_hash(candidate):\n    return hashlib.sha256(_canonical({k: v for k, v in candidate.items()\n                                     if k != "candidate_sha256"})).hexdigest()\n\n\ndef _array_hash(value):\n    return hashlib.sha256(np.asarray(value, dtype="<f4").tobytes(order="C")).hexdigest()\n\n\ndef _finite_vector(value, label):\n    result = np.asarray(value, dtype=np.float64)\n    if result.ndim != 1 or not result.size or not np.isfinite(result).all():\n        raise ValueError(f"{label} must be a nonempty finite vector")\n    return result\n\n\ndef _norm(value):\n    # Avoid platform/thread-dependent BLAS reductions in a frozen resume hash.\n    return math.sqrt(math.fsum(float(x) * float(x) for x in value))\n\n\ndef fit_candidate(calibration_records, *, calibration_ids, source_binding, seed=20261118):\n    """Fit once from exactly eight explicitly designated calibration records.\n\n    Each record binds its prior checkpoint and supplies B18 pulse-minus-clean\n    vectors at offsets8/32 plus maximum clean/pulse cached/full L2 discrepancies.\n    Numerical gates are informativeness checks, not significance tests.\n    """\n    ids = list(calibration_ids)\n    rows = list(calibration_records)\n    if len(ids) != 8 or len(set(ids)) != 8 or len(rows) != 8:\n        raise ValueError("Exactly eight distinct calibration IDs/records are required")\n    if any(row.get("split") != "calibration" for row in rows):\n        raise ValueError("Only designated calibration records may enter fitting")\n    by_id = {row["id"]: row for row in rows}\n    if len(by_id) != 8 or set(by_id) != set(ids):\n        raise ValueError("Calibration records do not match the frozen calibration IDs")\n    if not isinstance(source_binding, dict) or not source_binding:\n        raise ValueError("A nonempty source binding is required")\n    rows = [by_id[name] for name in ids]\n    early = [_finite_vector(row["delta8"], "delta8") for row in rows]\n    late = [_finite_vector(row["delta32"], "delta32") for row in rows]\n    if len({a.shape for a in early + late}) != 1:\n        raise ValueError("Calibration residual dimensions differ")\n    floors = {}\n    for offset in (8, 32):\n        errors = [float(row[f"cache_error{offset}"]) for row in rows]\n        if any(not np.isfinite(x) or x < 0 for x in errors):\n            raise ValueError("Cache parity errors must be finite and nonnegative")\n        floors[offset] = max(1e-6, 8.0 * max(errors))\n    for row in rows:\n        value = row.get("checkpoint_sha256", "")\n        if (not isinstance(value, str) or len(value) != 64 or\n                any(ch not in "0123456789abcdef" for ch in value)):\n            raise ValueError("Each calibration row requires an exact checkpoint SHA256")\n    mean8 = np.asarray([math.fsum(float(row[j]) for row in early) / 8\n                        for j in range(early[0].size)], dtype=np.float64)\n    norm8 = _norm(mean8)\n    direction = (mean8 / norm8 if norm8 > 0 else np.zeros_like(mean8)).astype(np.float32)\n    amplitude = math.fsum(math.fsum(float(a) * float(b) for a, b in zip(row, direction))\n                          for row in late) / 8\n    vector = (amplitude * direction.astype(np.float64)).astype(np.float32)\n    if not np.isfinite(vector).all():\n        raise ValueError("Candidate cannot be represented in native FP32")\n    norm = _norm(vector)\n    rng = np.random.Generator(np.random.PCG64(seed))\n    noise = rng.standard_normal((8, direction.size))\n    noise /= np.asarray([_norm(row) for row in noise])[:, None]\n    noise = (noise * norm).astype(np.float32)\n    reasons = []\n    if not norm8 > floors[8]:\n        reasons.append("mean_offset8_signal_below_prespecified_numeric_floor")\n    if not abs(amplitude) > floors[32]:\n        reasons.append("signed_offset32_projection_below_prespecified_numeric_floor")\n    candidate = {\n        "schema": "temporal_shared_candidate_v1",\n        "calibration_ids": ids,\n        "source_binding": json.loads(_canonical(source_binding)),\n        "calibration_checkpoints": {row["id"]: row["checkpoint_sha256"] for row in rows},\n        "direction": direction.tolist(), "amplitude": amplitude,\n        "vector": vector.tolist(), "noise_vectors": noise.tolist(),\n        "valid": not reasons, "invalid_reasons": reasons,\n        "diagnostics": {\n            "mean_delta8_norm": norm8, "floor8": floors[8], "floor32": floors[32],\n            "candidate_native_norm": norm,\n            "direction_sha256": _array_hash(direction),\n            "vector_sha256": _array_hash(vector),\n            "noise_vectors_sha256": _array_hash(noise),\n            "noise_seed": int(seed), "noise_generator": "NumPy PCG64",\n            "fit_arithmetic": "FP64 means/projections; direction and applied vectors stored FP32",\n            "reduction_policy": "fixed calibration order; math.fsum; no BLAS reductions in fit",\n            "sign_policy": "signed mean late projection; never absolute-valued or flipped",\n            "gate_scope": "prespecified numerical informativeness, not statistical significance",\n            "invalid_policy": "do not apply shared/noise patches; continue baseline/cache controls",\n        },\n    }\n    candidate["candidate_sha256"] = candidate_hash(candidate)\n    return candidate\n\n\ndef verify_candidate(candidate, expected_calibration_ids=None, expected_source_binding=None):\n    """Validate frozen bytes and internal geometry without fitting on new data."""\n    if candidate.get("schema") != "temporal_shared_candidate_v1":\n        raise ValueError("Unknown candidate schema")\n    if candidate.get("candidate_sha256") != candidate_hash(candidate):\n        raise ValueError("Frozen candidate hash mismatch")\n    if expected_calibration_ids is not None and candidate["calibration_ids"] != list(expected_calibration_ids):\n        raise ValueError("Candidate calibration IDs differ")\n    if expected_source_binding is not None and candidate["source_binding"] != expected_source_binding:\n        raise ValueError("Candidate source binding differs")\n    direction = _finite_vector(candidate["direction"], "direction")\n    vector = _finite_vector(candidate["vector"], "vector")\n    amplitude = float(candidate["amplitude"])\n    noise = np.asarray(candidate["noise_vectors"], dtype=np.float64)\n    if (vector.shape != direction.shape or noise.shape != (8, vector.size)\n            or not np.isfinite(amplitude) or not np.isfinite(noise).all()):\n        raise ValueError("Invalid candidate geometry")\n    expected_vector = (amplitude * direction).astype(np.float32)\n    if not np.array_equal(expected_vector, vector.astype(np.float32)):\n        raise ValueError("Candidate vector does not match signed amplitude/direction")\n    if candidate["valid"] and not np.isclose(np.linalg.norm(direction), 1, rtol=1e-6, atol=1e-7):\n        raise ValueError("Candidate direction is not unit length")\n    norm = np.linalg.norm(vector)\n    if not np.allclose(np.linalg.norm(noise, axis=1), norm, rtol=1e-6, atol=1e-12):\n        raise ValueError("Random controls are not norm matched")\n    for name, array in (("direction", direction), ("vector", vector), ("noise_vectors", noise)):\n        if _array_hash(array) != candidate["diagnostics"][f"{name}_sha256"]:\n            raise ValueError("Native candidate array hash mismatch")\n    if bool(candidate["invalid_reasons"]) == bool(candidate["valid"]):\n        raise ValueError("Candidate gate status is inconsistent")\n    return True\n\n\ndef projection_readout(delta, candidate):\n    """Evaluation only: never use held-out deltas to change the frozen patch."""\n    value = _finite_vector(delta, "delta")\n    direction = _finite_vector(candidate["direction"], "direction")\n    if value.shape != direction.shape:\n        raise ValueError("Readout dimension differs from frozen candidate")\n    return {"signed_projection": float(value @ direction),\n            "delta_norm": float(np.linalg.norm(value)),\n            "candidate_sha256": candidate["candidate_sha256"]}\n'}
MODULE_SHA256 = {'attention_probe/__init__.py': 'de626db34130b6d517d0dc776df8eff3b794bec5d7c9ba70157abe016f422705', 'attention_probe/reference.py': 'e89bd465d0e962e840ba589280ae89b84dd74f6fda1a459f4c76826ef4bd51e3', 'attention_probe/fixed_refw.py': 'b5cb362889f473f8b902454b4e22f4a615a54ed0a2bac0d7b0cc4c80e3c778e3', 'attention_probe/interventions.py': '1c7afb046108e2d7a214e1849198574128acf2c3e300888cdb6cc47e0e086561', 'qwen3_h286_runtime.py': 'e30b9da80bd9370eb5de04245ad43b8c0ac0e292611ba32d1a8721b28e1e4312', 'attention_probe/result_bundle.py': '1860c056df18124b26a990f0787df0b56be799a6d05fde319c1bded78d142774', 'attention_probe/answer_validation.py': '9bac92b5e286388d2f1e8cbf71b9fc181840de8b60f0988360eb9e5692a1ba3c', 'attention_probe/temporal_cache.py': '24eb148b24951d4b94eba11b2cb9fd9c339ce62d0a0881afad911bf04d85b82f', 'attention_probe/temporal_validation.py': 'c802cc2dd50860fee9e4f613ddf0d46ae79cc07f4ecfe100bdc00eb3f99e2931', 'qwen3_temporal_functional_runtime.py': '011db3034ce98acae92356630322c041ca7809e5a6801782a1bf65dbb95cc38c', 'attention_probe/temporal_candidate.py': 'dbfc3a213a3a98b3a7c9b1bb18d4ac0fe28a56a865a8c64bd78c6451f9de1be4'}


In [ ]:
import base64
from collections import deque
import datetime
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import traceback
import zipfile
import uuid

wrapper_attempt_id = uuid.uuid4().hex
base = Path('/content') if Path('/content').is_dir() else Path.cwd()
binding_fingerprint = hashlib.sha256(json.dumps({'modules': MODULE_SHA256, 'references': REFERENCE_SHA256},
                                                sort_keys=True).encode()).hexdigest()[:12]
work = (Path(RUN_DIRECTORY).expanduser() if RUN_DIRECTORY
        else base / ('temporal_functional_' + binding_fingerprint))
output = work / 'output'
output.mkdir(parents=True, exist_ok=True)
failure = None
helper_started = False
helper_status = None
return_code = None
status = {'state': 'running', 'stage': 'gpu_preflight', 'purpose': 'original_qwen3_temporal_functional',
          'pretrained_weights_loaded': False, 'inference': 'exploratory (post hoc)'}

def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + chr(10))

def run_command(command, logname, env=None):
    process = None
    tail = deque(maxlen=10)
    shown = 0
    try:
        with (output / logname).open('a', encoding='utf-8') as log:
            process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                log.write(line); log.flush()
                short = line.strip().replace(chr(13), ' ')[:500]
                tail.append(short)
                if logname == 'runtime.log' and shown < 120 and short.startswith(
                        ('Request ', 'Gespeichert ', 'Stage ', 'Status:', 'Status ', 'CHECK ', 'HARD FAILURE:')):
                    print(short, flush=True); shown += 1
            code = process.wait()
        if code:
            print('Kindprozess fehlgeschlagen; vollständiges Log im ZIP. Letzte Zeilen:', flush=True)
            for short in tail:
                print(short, flush=True)
            raise subprocess.CalledProcessError(code, command)
        return code
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
        raise

try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Keine CUDA-GPU. A100, L4 oder B200 auswählen; kein Warten.')
    capability = tuple(torch.cuda.get_device_capability(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    if capability[0] < 8 or free_bytes < MIN_FREE_VRAM_GIB * 1024**3:
        raise RuntimeError('Ampere-oder-neuere CUDA-GPU mit mindestens 12 GiB freiem Speicher erforderlich.')
    torch_before = importlib.metadata.version('torch')
    environment = {'python': sys.version, 'torch': torch_before, 'torch_cuda': torch.version.cuda,
                   'gpu': torch.cuda.get_device_name(0), 'capability': list(capability),
                   'free_vram_gib': free_bytes / 1024**3, 'total_vram_gib': total_bytes / 1024**3,
                   'transformers_required': TRANSFORMERS_VERSION, 'accelerate_required': ACCELERATE_VERSION,
                   'execution_dtype': 'float32', 'tf32': False}
    write_json(output / 'environment_before.json', environment)
    print('GPU:', environment['gpu'], '| frei:', round(environment['free_vram_gib'], 2), 'GiB')
    print('Torch/CUDA bleibt erhalten:', torch_before, torch.version.cuda)
    status['stage'] = 'reference_validation'
    inputs = work / 'inputs'; inputs.mkdir(exist_ok=True)
    for name, expected in REFERENCE_SHA256.items():
        if name in REFERENCE_FILES_B64:
            raw = base64.b64decode(REFERENCE_FILES_B64[name], validate=True)
        elif REFERENCE_DIRECTORY:
            raw = (Path(REFERENCE_DIRECTORY).expanduser() / name).read_bytes()
        else:
            raise RuntimeError('Öffentliches Template ohne private Referenz. Die bereitgestellte private Nutzdatei verwenden.')
        if hashlib.sha256(raw).hexdigest() != expected:
            raise ValueError('Originalreferenz-Prüfsumme weicht ab: ' + name)
        target = inputs / name
        if target.exists() and target.read_bytes() != raw:
            raise ValueError('Vorhandene Referenz gehört zu einer anderen Auswahl; Ergebnisse nicht mischen.')
        target.write_bytes(raw)
    binding = {'sha256': MODULE_SHA256, 'reference_sha256': REFERENCE_SHA256,
               'model': 'Qwen/Qwen3-1.7B', 'revision': '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'}
    binding_path = output / 'embedded_source.json'
    if binding_path.exists() and json.loads(binding_path.read_text()) != binding:
        raise ValueError('Vorhandener Lauf gehört zu anderem Quellcode oder Referenzen; Ergebnisse nicht mischen.')
    code = work / 'code'; code.mkdir(exist_ok=True)
    for name, source in MODULE_SOURCES.items():
        if hashlib.sha256(source.encode()).hexdigest() != MODULE_SHA256[name]:
            raise ValueError('Eingebetteter Quellcode weicht ab: ' + name)
        path = code / name; path.parent.mkdir(parents=True, exist_ok=True); path.write_text(source)
    write_json(binding_path, binding)
    status['stage'] = 'dependencies'
    constraint = work / 'preserve_torch.txt'
    constraint.write_text('torch==' + torch_before + chr(10))
    print('Exakte Transformers-/Accelerate-Version installieren; GPU-Torch bleibt festgelegt.', flush=True)
    child_env = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS='1', TQDM_DISABLE='1',
                     TOKENIZERS_PARALLELISM='false')
    run_command([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--upgrade-strategy', 'only-if-needed',
                 '--constraint', str(constraint), 'transformers==' + TRANSFORMERS_VERSION,
                 'accelerate==' + ACCELERATE_VERSION], 'install.log', child_env)
    installed = {name: importlib.metadata.version(name) for name in ('torch', 'transformers', 'accelerate')}
    if installed != {'torch': torch_before, 'transformers': TRANSFORMERS_VERSION, 'accelerate': ACCELERATE_VERSION}:
        raise RuntimeError('Paketversionen weichen ab; keine Modellgewichte angefordert.')
    write_json(output / 'environment_installed.json', installed)
    status['stage'] = 'runtime'
    helper_started = True
    status['pretrained_weights_loaded'] = False
    env = dict(child_env, PYTHONPATH=str(code), H286_WRAPPER_ATTEMPT=wrapper_attempt_id)
    command = [sys.executable, str(code / 'qwen3_temporal_functional_runtime.py'), '--hn-set', str(inputs / 'hn_set.json'),
               '--policy', str(inputs / 'htq_policy.json'), '--refw', str(inputs / 'refw_h286.json'),
               '--items', str(inputs / 'temporal_items.json'),
               '--plan', str(inputs / 'temporal_plan.json'),
               '--out-dir', str(output), '--cache-dir', str(base / 'h286_model_cache')]
    if RESUME_ZIP:
        command += ['--resume-zip', str(Path(RESUME_ZIP).expanduser())]
    print('Frischer Python-Prozess: Original-FP32-Modell, feste H286/REF-W, Zeitliche und funktionale Stufe: feste Kalibrierung, Testeingaben und Cache-Kontrollen.', flush=True)
    return_code = run_command(command, 'runtime.log', env)
    helper_status = json.loads((output / 'status.json').read_text())
    if helper_status.get('state') not in ('complete', 'paused'):
        raise RuntimeError('Runtime hat keinen vollständigen oder bewusst pausierten Abschluss gemeldet.')
    status.update(helper_status, helper_return_code=return_code, wrapper_verified=True)
except BaseException as error:
    failure = error
    status.update(state='failed', error_type=type(error).__name__, error=str(error), wrapper_verified=False)
    (output / 'wrapper_diagnostic.txt').write_text(traceback.format_exc())
finally:
    # Keep the child's original status for diagnosis, then publish one authoritative final status.
    child_path = output / 'status.json'
    if helper_started and child_path.exists():
        try:
            helper_status = json.loads(child_path.read_text())
            write_json(output / 'helper_status.json', helper_status)
            if failure is not None:
                status['helper_state'] = helper_status.get('state')
                status['pretrained_weights_loaded'] = helper_status.get('pretrained_weights_loaded')
        except Exception:
            status['helper_state'] = 'unreadable'
    # Only this attempt's metrics can establish whether its model weights loaded.
    # Stable work folders may contain metrics from an earlier successful attempt.
    if helper_started:
        status['pretrained_weights_loaded'] = None
        status['weights_loaded_evidence'] = 'missing_unreadable_or_stale_child_metrics'
        try:
            child_metrics = json.loads((output / 'metrics.json').read_text())
            loaded = child_metrics.get('weights_loaded')
            if (child_metrics.get('wrapper_attempt_id') == wrapper_attempt_id
                    and type(loaded) is bool):
                status['pretrained_weights_loaded'] = loaded
                status['weights_loaded_evidence'] = 'fresh_child_metrics'
        except (OSError, ValueError, TypeError, AttributeError):
            pass
    status['wrapper_attempt_id'] = wrapper_attempt_id
    status['written_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    write_json(output / 'status.json', status)
    members = {str(path.relative_to(output)): hashlib.sha256(path.read_bytes()).hexdigest()
               for path in sorted(output.rglob('*')) if path.is_file()
               and path.name not in ('WRAPPER_MANIFEST.json',) and path.suffix not in ('.zip', '.tmp')
               and not any(part.startswith('.') for part in path.relative_to(output).parts)}
    write_json(output / 'WRAPPER_MANIFEST.json', members)
    target = base / ('temporal_functional_diagnostic.zip' if failure is not None else 'temporal_functional_results.zip')
    temporary = target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob('*')):
            if (path.is_file() and path.suffix not in ('.zip', '.tmp')
                    and not any(part.startswith('.') for part in path.relative_to(output).parts)):
                archive.write(path, str(path.relative_to(output)))
    with zipfile.ZipFile(temporary) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('Ergebnis-ZIP ist beschädigt.')
    os.replace(temporary, target)
    raw_target = None
    if helper_started:
        # Compact analysis retains every state/derivative array. Full logit
        # vectors and exact resume checkpoints stay in a separate RAW archive.
        try:
            import importlib.util
            bundle_path = code / 'attention_probe/result_bundle.py'
            expected_bundle = MODULE_SHA256['attention_probe/result_bundle.py']
            if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != expected_bundle:
                raise ValueError('Result-export source SHA mismatch')
            bundle_spec = importlib.util.spec_from_file_location('b14_b18_result_export', bundle_path)
            bundle_module = importlib.util.module_from_spec(bundle_spec)
            bundle_spec.loader.exec_module(bundle_module)
            raw_target = target.with_name(target.stem + '_RAW.zip')
            os.replace(target, raw_target)
            compact_info = bundle_module.build_compact_result(raw_target, target)
            print('Kompakte Auswertung:', target, '| Rohdaten ebenfalls sichern:', raw_target, flush=True)
        except Exception as compact_error:
            print('Kompaktexport nicht verfügbar; vollständiges Original-ZIP verwenden:', str(compact_error)[:300], flush=True)
            if raw_target is not None and raw_target.is_file():
                os.replace(raw_target, target)
            raw_target = None
    print('Status:', status['state'], '| ZIP:', target, flush=True)
    try:
        from google.colab import files
        files.download(str(target))
        if raw_target is not None:
            files.download(str(raw_target))
    except Exception:
        try:
            from IPython.display import FileLink, display
            display(FileLink(os.path.relpath(target, Path.cwd())))
            if raw_target is not None:
                display(FileLink(os.path.relpath(raw_target, Path.cwd())))
        except Exception:
            print('ZIP aus diesem Verzeichnis herunterladen:', target)
if failure is not None:
    raise RuntimeError('Versuch angehalten; das Diagnose-ZIP enthält den tatsächlichen Fehler.') from None
if status['state'] == 'paused':
    print('Bewusst pausierter Checkpoint; kein abgeschlossener Versuch und kein PASS.')
else:
    print('Zeitliche und funktionale Stufe abgeschlossen. Messungen und Verhalten getrennt auswerten.')


Kompakte Auswertung und vollständige Rohdaten sichern. Kalibrierung, Testfälle, Cache-Kontrollen und funktionale Endpunkte getrennt beurteilen.
